# LD publication workspace

Working title: **Lateral-drag parameterisations for Antarctic landfast sea ice in CICE version 6.4.1**

This notebook is designed as the publication workspace for the Sep--Dec 1994 lateral-drag experiments, with the formal analysis window restricted to **1994-10-01 to 1994-12-15**. The shorter analysis window avoids the initial-condition adjustment period and reduces binary-days edge effects near the end of the available run.

The notebook is organised around the paper logic rather than around individual diagnostics:

1. free-slip versus no-slip conceptual model;
2. static form-factor generation from high-resolution coastline and grounded icebergs;
3. analytical and numerical form-function behaviour;
4. corrected `blend_strain` intercomparison over the 1994 peak and beginning of retreat season (as per observations);
5. comparison against static, quadratic, and linear form functions;
6. optional multi-year static versus best-blend comparison against AF2020;
7. collateral pack-ice checks.

All fast-ice outcome diagnostics should use the `shuga` binary-days classification wherever applicable.

In [1]:
from __future__ import annotations
import os, sys, warnings
import numpy             as np
import pandas            as pd
import xarray            as xr
import matplotlib.pyplot as plt
from IPython.display     import Video, Image, display
from dataclasses         import dataclass, replace 
from pathlib             import Path
from typing              import Mapping, Sequence
from math                import ceil
xr.set_options(keep_attrs=True)
warnings.filterwarnings("default")
repo_root = Path.home() / "AFIM" / "src" / "mawsons-chest"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Theoretical discussion on free-slip strain-rate expecation on C-grid

This note documents the expected behaviour of `divergU`, `tensionU`, `shearU`, and `DeltaU` for a uniform-grid, uniformly translating free-slip test case. It is intended as a short reference for validating `strain_rates_U_free_slip` before using `DeltaU` as the effective strain-rate input to the `blend_strain` lateral-drag form function.

--- 

## 1. Continuous free-slip expectation

For a two-dimensional velocity field

$$
\mathbf{u} = (u,v),
$$

the strain-rate components are

$$
\mathrm{e}_{11} = \frac{\partial u}{\partial x}, \qquad
\mathrm{e}_{22} = \frac{\partial v}{\partial y}, \qquad
\mathrm{e}_{12} = \frac{\partial u}{\partial y} + \frac{\partial v}{\partial x}.
$$

The CICE-style diagnostic combinations are

$$
\mathrm{div} = \frac{\partial u}{\partial x} + \frac{\partial v}{\partial y},
$$

$$
\mathrm{tension} = \frac{\partial u}{\partial x} - \frac{\partial v}{\partial y},
$$

$$
\mathrm{shear} = \frac{\partial u}{\partial y} + \frac{\partial v}{\partial x},
$$

and the EVP deformation invariant is

$$
\Delta\mathrm{U} = \sqrt{\mathrm{div}^{2} + \mathrm{e}_{\mathrm{fac}} \left(\mathrm{tension}^{2} + \mathrm{shear}^{2}\right)}.
$$

A free-slip wall imposes no normal flow and no normal derivative of tangential velocity:

$$
\mathbf{u}\cdot\mathbf{n} = 0,
$$

$$
\frac{\partial u_t}{\partial n} = 0.
$$

For a rigidly translating flow,

$$
u(x,y) = \mathrm{U}_0, \qquad v(x,y) = \mathrm{V}_0,
$$

where $\mathrm{U}_0$ and $\mathrm{V}_0$ are constants. Hence all spatial derivatives vanish:

$$
\frac{\partial u}{\partial x} = \frac{\partial u}{\partial y} = \frac{\partial v}{\partial x} = \frac{\partial v}{\partial y} = 0
$$

Therefore

$$
\mathrm{div} = 0,\qquad
\mathrm{tension} = 0,\qquad
\mathrm{shear} = 0,\qquad
\Delta\mathrm{U} = 0.
$$

This is the continuous expectation for a spatially uniform velocity field. Free-slip does not create deformation in rigid sea ice.

---

## 2. Why the discrete CICE free-slip routine should give zero on a uniform grid

On a uniform orthogonal C-grid,

$$
dx_\mathrm{E} = dx_\mathrm{U} = \Delta x, \qquad dy_\mathrm{N} = dy_\mathrm{U} = \Delta y,
$$

so the metric-gradient correction terms vanish:

$$
dx_{\mathrm{E}(i,j+1)} - dx_{\mathrm{E}(i,j)} = 0,
$$

$$
dy_{\mathrm{N}(i+1,j)} - dy_{\mathrm{N}(i,j)} = 0.
$$

For a spatially uniform sea ice velocity field,

$$
u_\mathrm{E} = u_\mathrm{N} = \mathrm{U}_0,
$$

$$
v_\mathrm{E} = v_\mathrm{N} = V_0.
$$

The free-slip routine uses even reflection at masked neighbouring faces. For example, one reflected value has the form

$$
u_{\mathrm{N}{i+1,j}} = u_{\mathrm{N}(i+1,j)}\,\mathrm{npm}(i+1,j) + \left[\mathrm{npm}(i,j)-\mathrm{npm}(i+1,j)\right] \; \mathrm{npm}(i,j) \; u_{\mathrm{N}(i,j)}.
$$

If both neighbouring faces are active,

$$
\mathrm{npm}(i,j) = \mathrm{npm}(i+1,j) = 1,
$$

then

$$
u_{\mathrm{N}{i+1,j}} = u_{\mathrm{N}(i+1,j)} = \mathrm{U}_0.
$$

If the neighbour is masked but the interior face is active,

$$
\mathrm{npm}(i+1,j) = 0, \qquad \mathrm{npm}(i,j) = 1,
$$

then

$$
u_{\mathrm{N}{i+1,j}} = 0+(1-0)(1)u_{\mathrm{N}(i,j)} = \mathrm{U}_0.
$$

Thus in either case

$$
u_{\mathrm{N}{i+1,j}} = \mathrm{U}_0.
$$

The same argument applies to the other reflected face values:

$$
u_{\mathrm{N}{ij}} = \mathrm{U}_0, \qquad u_{\mathrm{E}{i,j+1}} = \mathrm{U}_0, \qquad u_{\mathrm{E}{ij}} = \mathrm{U}_0,
$$

$$
v_{\mathrm{E}{i,j+1}} = \mathrm{V}_0, \qquad v_{\mathrm{E}{ij}} = \mathrm{V}_0, \qquad v_{\mathrm{N}{i+1,j}} = \mathrm{V}_0, \qquad v_{\mathrm{N}{ij}} = \mathrm{V}_0.
$$

Therefore every discrete difference in the strain-rate formulas vanishes.

## 3. Discrete deduction for the `strain_rates_U_free_slip` formulas

Using the structure of `strain_rates_U_free_slip`, the uniform-grid forms reduce to the following.

### divergence

$$
\mathrm{divergU}(i,j) = \Delta y \left(u_{\mathrm{N}{i+1,j}} - u_{\mathrm{N}{ij}}\right) + \Delta x \left(v_{\mathrm{E}{i,j+1}} - v_{\mathrm{E}{ij}}\right).
$$

Since

$$
u_{\mathrm{N}{i+1,j}} = u_{\mathrm{N}{ij}} = \mathrm{U}_0,
$$

and

$$
v_{\mathrm{E}{i,j+1}} = v_{\mathrm{E}{ij}} = \mathrm{V}_0,
$$

then

$$
\mathrm{divergU}(i,j) = 0.
$$

### tension

$$
\mathrm{tensionU}(i,j) = \Delta y \left(u_{\mathrm{N}{i+1,j}} - u_{\mathrm{N}{ij}}\right) - \Delta x \left(v_{\mathrm{E}{i,j+1}} - v_{\mathrm{E}{ij}}\right).
$$

The same equalities imply

$$
\mathrm{tensionU}(i,j) = 0.
$$

### shear

$$
\mathrm{shearU}(i,j) = \Delta x \left(u_{\mathrm{E}{i,j+1}} - u_{\mathrm{E}{ij}}\right) + \Delta y \left(v_{\mathrm{N}{i+1,j}} - v_{\mathrm{N}{ij}}\right).
$$

Since

$$
u_{\mathrm{E}{i,j+1}} = u_{\mathrm{E}{ij}} = \mathrm{U}_0,
$$

and

$$
v_{\mathrm{N}{i+1,j}} = v_{\mathrm{N}{ij}} = \mathrm{V}_0,
$$

then

$$
\mathrm{shearU}(i,j) = 0.
$$

### DeltaU

The EVP deformation invariant is

$$
\Delta \mathrm{U}(i,j) = \sqrt{\mathrm{divergU}(i,j)^2 + \mathrm{e}_{\mathrm{fac}} \left[\mathrm{tensionU}(i,j)^2 + \mathrm{shearU}(i,j)^2\right]}.
$$

Thus

$$
\Delta \mathrm{U}(i,j) = 0.
$$

---

## 4. Bottom line

For the uniform-grid, uniform-flow, free-slip benchmark, the expected result is

$$
\boxed{\mathrm{divergU} = \mathrm{tensionU} = \mathrm{shearU} = \Delta \mathrm{U} = 0}
$$

everywhere in the interior, and also at boundary-adjacent U-points if the free-slip reflection logic is implemented consistently.

If this idealised test produces nonzero values, the likely causes are:

1. the free-slip reflection logic is wrong;
2. the metric terms are not being handled consistently;
3. the test case is not truly uniform;
4. there is a halo, indexing, or mask inconsistency.

---

## 5. Important caveat

The result above does **not** mean that free-slip always forces zero deformation.

It only means that free-slip admits a zero-strain rigid-translation solution on a uniform grid, and that the toy test should recover that solution.

In realistic simulations, even with free-slip, nonzero deformation is expected because of spatially variable forcing, coastline geometry, form factors, grounded icebergs, Coriolis effects, internal stress gradients, ice thickness gradients, concentration gradients, ocean-current gradients, and pack-ice interactions. In those simulations, `DeltaU` should generally be nonzero in deforming regions.

For the `blend_strain` lateral-drag form function, the physically relevant quantity is the area-normalised strain-rate scale:

$$
\epsilon_{\mathrm{eff}} = \frac{\Delta \mathrm{U}_1+\Delta \mathrm{U}_2}{\mathrm{A}_1+\mathrm{A}_2},
$$

where the two neighbouring U-cell deformation invariants are averaged to the E or N velocity point. This yields units of $\mathrm{s}^{-1}$, suitable for comparison with `eps_blend`.

---

# Developer note: 4x4 uniform-grid discrete worked example

This section provides a compact worked example suitable for insertion as a developer note above `strain_rates_U_free_slip`.

Assume a 4x4 uniform C-grid with

$$
dx_{\mathrm{E}} = dx_{\mathrm{U}} =\Delta x, \qquad dy_{\mathrm{N}} = \Delta y,
$$

and spatially uniform velocities

$$
u_{\mathrm{E}} = u_{\mathrm{N}} = \mathrm{U}_0,
$$

$$
v_{\mathrm{E}} = v_{\mathrm{N}} = \mathrm{V}_0,
$$

The free-slip reflected face values in the routine are

$$
v_{\mathrm{E}{i,j+1}} = v_{\mathrm{E}(i,j+1)}\,\mathrm{epm}(i,j+1) + \left[\mathrm{epm}(i,j)-\mathrm{epm}(i,j+1)\right]\mathrm{epm}(i,j)v_{\mathrm{E}(i,j)},
$$

$$
v_{\mathrm{E}{ij}} = v_{\mathrm{E}(i,j)}\,\mathrm{epm}(i,j) + \left[\mathrm{epm}(i,j+1)-\mathrm{epm}(i,j)\right]\mathrm{epm}(i,j+1)v_{\mathrm{E}(i,j+1)},
$$

$$
u_{\mathrm{N}{i+1,j}} = u_{\mathrm{N}(i+1,j)}\,\mathrm{npm}(i+1,j) + \left[\mathrm{npm}(i,j)-\mathrm{npm}(i+1,j)\right]\mathrm{npm}(i,j)u_{\mathrm{N}(i,j)},
$$

$$
u_{\mathrm{N}{ij}} = u_{\mathrm{N}(i,j)}\,\mathrm{npm}(i,j) + \left[\mathrm{npm}(i+1,j)-\mathrm{npm}(i,j)\right]\mathrm{npm}(i+1,j)u_{\mathrm{N}(i+1,j)}.
$$

For active-active neighbouring faces, both masks are one and each reflected value reduces to the neighbouring uniform velocity. For active-masked neighbouring faces, the masked value is replaced by the active interior value. Therefore, for all active U-points adjacent to a free-slip boundary,

$$
v_{\mathrm{E}{i,j+1}} = v_{\mathrm{E}{ij}} = \mathrm{V}_0,
$$

$$
u_{\mathrm{N}{i+1,j}} = u_{\mathrm{N}{ij}} = \mathrm{U}_0.
$$

Similarly,

$$
u_{\mathrm{E}{i,j+1}} = u_{\mathrm{E}{ij}} = \mathrm{U}_0,
$$

$$
v_{\mathrm{N}{i+1,j}} = v_{\mathrm{N}{ij}} = \mathrm{V}_0.
$$

Substituting into the routine's discrete formulas gives

$$
\mathrm{divergU} = \Delta y(\mathrm{U}_0 - \mathrm{U}_0) + \Delta x(\mathrm{V}_0 - \mathrm{V}_0) + \mathrm{U}_0(\Delta y-\Delta y) + \mathrm{V}_0(\Delta x-\Delta x) = 0,
$$

$$
\mathrm{tensionU} = \Delta y(\mathrm{U}_0 - \mathrm{U}_0) - \Delta x(\mathrm{V}_0 - \mathrm{V}_0) - \mathrm{U}_0(\Delta y-\Delta y) + \mathrm{V}_0(\Delta x-\Delta x) = 0,
$$

$$
\mathrm{shearU} = \Delta x(\mathrm{U}_0 - \mathrm{U}_0) + \Delta y(\mathrm{V}_0 - \mathrm{V}_0) - \mathrm{U}_0(\Delta x-\Delta x) - \mathrm{V}_0(\Delta y-\Delta y) = 0.
$$

Therefore,

$$
\Delta \mathrm{U} = \sqrt{\mathrm{divergU}^2 + \mathrm{e}_{\mathrm{fac}} \left(\mathrm{tensionU}^2+ \mathrm{shearU}^2\right)} = 0.
$$

This is the expected pass condition for a 4x4 uniform-grid free-slip verification test.

# 4x4 free-slip strain-rate verification

This notebook checks the discrete expectation for `strain_rates_U_free_slip` on a uniform 4x4 grid with uniform velocity.

Expected result:

$$
\mathrm{divergU} =\mathrm{tensionU} = \mathrm{shearU} = \Delta \mathrm{U} = 0.
$$

This is a toy verification problem. It does not imply that `DeltaU` should be zero in realistic Antarctic simulations.

In [ ]:
import numpy as np
import pandas as pd
# setup/config:
nx = 4        # number of grid cells
ny = 4
dx = 12_000.0 # rougly corresponding to grid cell size of Antarctic coastline; 12km
dy = 12_000.0
U0 = 0.02     # easterly (rightward) velocity; 2 cm/s
V0 = 0.01     # northerly (upward) velocity; 1 cm/s 
e_factor = 1.4 # dimensionless
# create arrays
dxE   = np.full((nx, ny), dx)
dxU   = np.full((nx, ny), dx)
dyN   = np.full((nx, ny), dy)
dyU   = np.full((nx, ny), dy)
uvelE = np.full((nx, ny), U0)
uvelN = np.full((nx, ny), U0)
uvelU = np.full((nx, ny), U0)
vvelE = np.full((nx, ny), V0)
vvelN = np.full((nx, ny), V0)
vvelU = np.full((nx, ny), V0)
epm   = np.ones((nx, ny))
npm   = np.ones((nx, ny))
print(f"Uniform velocity: U0={U0} m/s, V0={V0} m/s")
print(f"Uniform grid: dx={dx} m, dy={dy} m")

## Case A: fully active interior

All E and N masks are active. The uniform velocity field should produce zero discrete deformation.


In [ ]:
def strain_rates_U_free_slip_discrete(i, j, uvelE, vvelE, uvelN, vvelN, uvelU, vvelU, dxE, dyN, dxU, dyU, epm, npm, e_factor=1.0):
    # a python translation of the algebra used in strain_rates_U_free_slip.
    # Indices are zero-based here, but the symbolic structure follows the Fortran routine.
    vEijp1   = vvelE[i, j+1] * epm[i, j+1] + (epm[i, j]   - epm[i, j+1]) * epm[i, j]   * vvelE[i, j]
    vEij     = vvelE[i, j  ] * epm[i, j  ] + (epm[i, j+1] - epm[i, j  ]) * epm[i, j+1] * vvelE[i, j+1]
    uNip1j   = uvelN[i+1, j] * npm[i+1, j] + (npm[i, j]   - npm[i+1, j]) * npm[i, j]   * uvelN[i, j]
    uNij     = uvelN[i,   j] * npm[i,   j] + (npm[i+1, j] - npm[i,   j]) * npm[i+1, j] * uvelN[i+1, j]
    divergU  = (dyU[i, j] * (uNip1j - uNij) 
               + uvelU[i, j] * (dyN[i+1, j] - dyN[i, j])
               + dxU[i, j] * (vEijp1 - vEij)
               + vvelU[i, j] * (dxE[i, j+1] - dxE[i, j]))
    tensionU = (dyU[i, j] * (uNip1j - uNij)
                - uvelU[i, j] * (dyN[i+1, j] - dyN[i, j])
                - dxU[i, j] * (vEijp1 - vEij)
                + vvelU[i, j] * (dxE[i, j+1] - dxE[i, j]))
    uEijp1   = uvelE[i, j+1] * epm[i, j+1] + (epm[i, j]   - epm[i, j+1]) * epm[i, j]   * uvelE[i, j]
    uEij     = uvelE[i, j  ] * epm[i, j  ] + (epm[i, j+1] - epm[i, j  ]) * epm[i, j+1] * uvelE[i, j+1]
    vNip1j   = vvelN[i+1, j] * npm[i+1, j] + (npm[i, j]   - npm[i+1, j]) * npm[i, j]   * vvelN[i, j]
    vNij     = vvelN[i,   j] * npm[i,   j] + (npm[i+1, j] - npm[i,   j]) * npm[i+1, j] * vvelN[i+1, j]
    shearU   = (dxU[i, j] * (uEijp1 - uEij)
                - uvelU[i, j] * (dxE[i, j+1] - dxE[i, j])
                + dyU[i, j] * (vNip1j - vNij)
                - vvelU[i, j] * (dyN[i+1, j] - dyN[i, j]))
    DeltaU   = np.sqrt(divergU**2 + e_factor * (tensionU**2 + shearU**2))
    return {"vEijp1"  : vEijp1,
            "vEij"    : vEij,
            "uNip1j"  : uNip1j,
            "uNij"    : uNij,
            "uEijp1"  : uEijp1,
            "uEij"    : uEij,
            "vNip1j"  : vNip1j,
            "vNij"    : vNij,
            "divergU" : divergU,
            "tensionU": tensionU,
            "shearU"  : shearU,
            "DeltaU"  : DeltaU}

In [ ]:
rows = []
for i in [1, 2]:
    for j in [1, 2]:
        out = strain_rates_U_free_slip_discrete(i, j, uvelE, vvelE, uvelN, vvelN, uvelU, vvelU, dxE, dyN, dxU, dyU, epm, npm, e_factor)
        rows.append({"i": i, "j": j, **{k: out[k] for k in ["divergU", "tensionU", "shearU", "DeltaU"]}})
pd.DataFrame(rows)

## Case B: free-slip reflection beside a masked face

Here we mask one neighbouring face for a U-point. The free-slip reflection should substitute the active interior value and still produce zero deformation for the uniform-flow case.


In [ ]:
epm_b = epm.copy()
npm_b = npm.copy()
# U point (i=1, j=1), zero-based indexing.
epm_b[1, 2] = 0.0   # E(i,j+1) masked
npm_b[2, 1] = 0.0   # N(i+1,j) masked
out = strain_rates_U_free_slip_discrete(1, 1, uvelE, vvelE, uvelN, vvelN, uvelU, vvelU, dxE, dyN, dxU, dyU, epm_b, npm_b, e_factor)
pd.DataFrame([out]).T.rename(columns={0: "value"})

## Case C: demonstrate that non-uniform velocity gives nonzero deformation

Free-slip does not force `DeltaU` to zero in general. It only preserves the zero-deformation state for rigid translation on a uniform grid.


In [ ]:
uvelE_c = uvelE.copy()
uvelN_c = uvelN.copy()
uvelU_c = uvelU.copy()
vvelE_c = vvelE.copy()
vvelN_c = vvelN.copy()
vvelU_c = vvelU.copy()
# Add a simple velocity gradient.
for i in range(nx):
    uvelE_c[i, :] += 0.002 * i
    uvelN_c[i, :] += 0.002 * i
    uvelU_c[i, :] += 0.002 * i
out = strain_rates_U_free_slip_discrete(1, 1, uvelE_c, vvelE_c, uvelN_c, vvelN_c, uvelU_c, vvelU_c, dxE, dyN, dxU, dyU, epm, npm, e_factor)
pd.DataFrame([out]).T.rename(columns={0: "value"})

## Note

A correct `strain_rates_U_free_slip` implementation should satisfy:

1. Uniform grid + uniform velocity + all active masks: `divergU` = `tensionU` = `shearU` = `DeltaU` = 0.
2. Uniform grid + uniform velocity + free-slip reflected masked neighbour: `divergU` = `tensionU` = `shearU` = `DeltaU` = 0.
3. Uniform grid + non-uniform velocity: at least one of the diagnostics should become nonzero.

This is the minimal verification logic before using `DeltaU/uarea` as the strain-rate gate for `blend_strain`.


# 0. Setup, imports, experiments, and analysis windows

The model output can cover 1994-09-01 to 1994-12-31, but the main analysis window is deliberately narrower:

- **run context window:** 1994-09-01 to 1994-12-31;
- **publication analysis window:** 1994-10-01 to 1994-12-15;
- **classification:** `binary-days`, using the existing `shuga` workflow.

The `blend_strain` experiments contain additional lateral-drag diagnostics (`ldphi`, `ldwgt`, `ldeps`, `ldspd`, branch terms) that are absent from the earlier static/quadratic/linear runs. The helpers below degrade gracefully when those fields are unavailable.

## Project / paths

In [ ]:
PROJECT           = "gv90"
USER              = "da1339"
HEMISPHERE        = "SH"
ICE_TYPE          = "FI"
GRID_TYPE         = "Tc"
AFIM_OUTPUT_ROOT  = Path(f"/g/data/{PROJECT}/{USER}/afim_output")
AFIM_ARCHIVE_ROOT = Path.home() / "AFIM_archive"
GRAPHICS_ROOT     = Path(f"/g/data/{PROJECT}/{USER}/GRAPHICAL")
NOTEBOOK_FIG_ROOT = GRAPHICS_ROOT / "LD-pub-workspace"
NOTEBOOK_FIG_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Using repo_root={repo_root}")
print(f"Figures -> {NOTEBOOK_FIG_ROOT}")
# Static form-factor input. Update if the new shuga output path differs.
FORM_FACTOR_FILE = Path("/g/data/gv90/da1339/coastal_drag/form_factors/ADD_high-res_cstln_v7p9_GI_CICE_free-slip.nc")

## Constants

In [ ]:
RHO_ICE = 917.0
EPS = 1.0e-30

## Classification controls

In [ ]:
METHOD       = "binary-days"
ISPD_THRESH  = 5.0e-4
BIN_WINDOW   = 11
BIN_MIN_DAYS = 9
ROLL_WINDOW  = 15

## date-time

In [ ]:
RUN_START      = "1999-09-01"
RUN_END        = "1994-12-31"
# Main publication analysis window.
ANALYSIS_START = "1994-09-15"
ANALYSIS_END   = "1994-12-15"
# Optional shorter diagnostic windows.
MAP_MONTHS     = ["1994-10", "1994-11", "1994-12"]
REGION_NAME    = "Aus"      # other options: DML/WIO/EIO/VOL/AS/BS/WS
SAMPLE_FRAC    = 0.02
RNG_SEED       = 42
SAVE_FIGS      = True
SHOW_FIGS      = True

## experiments

In [ ]:
COMPARATOR_SIMS = ["LD-NIL",           
                   "LD-static-Cs5e-4",
                   "LD-quad-Cq75",
                   "LD-linear-CL0p25"] 
BLEND_SIMS      = ["LD-blend-base",
                   "LD-blend-Cq100-exp10",
                   "LD-blend-Cq75-exp10",
                   "LD-blend-Cq75-exp20",
                   "LD-blend-CsCq-tiny",
                   "LD-blend-exp10"]
CORE_SIMS       = COMPARATOR_SIMS + BLEND_SIMS
EXPERIMENT_META = pd.DataFrame([dict(sim_name="LD-blend-base", group="blend", purpose="central hypothesis",
                                     e_f=1.5, ktens=0.05, Cs=5.0e-4, Cq=50.0, u0=5.0e-5, u_blend=5.0e-4, eps_blend=5.0e-8, blend_exp=3.0),
                                dict(sim_name="LD-blend-Cq100-exp10", group="blend", purpose="central hypothesis",
                                     e_f=1.5, ktens=0.05, Cs=5.0e-4, Cq=100.0, u0=5.0e-5, u_blend=5.0e-4, eps_blend=5.0e-8, blend_exp=10.0),
                                dict(sim_name="LD-blend-Cq75-exp10", group="blend", purpose="central hypothesis",
                                     e_f=1.5, ktens=0.05, Cs=5.0e-4, Cq=75.0, u0=5.0e-5, u_blend=5.0e-4, eps_blend=5.0e-8, blend_exp=10.0),
                                dict(sim_name="LD-blend-Cq75-exp10", group="blend", purpose="central hypothesis",
                                     e_f=1.5, ktens=0.05, Cs=5.0e-4, Cq=75.0, u0=5.0e-5, u_blend=5.0e-4, eps_blend=5.0e-8, blend_exp=20.0),
                                dict(sim_name="LD-blend-CsCq-tiny", group="blend", purpose="central hypothesis",
                                     e_f=1.5, ktens=0.05, Cs=1.0e-4, Cq=10.0, u0=5.0e-5, u_blend=5.0e-4, eps_blend=5.0e-8, blend_exp=3.0),
                                dict(sim_name="LD-blend-exp10", group="blend", purpose="central hypothesis",
                                     e_f=1.5, ktens=0.05, Cs=5.0e-4, Cq=50.0, u0=5.0e-5, u_blend=5.0e-4, eps_blend=5.0e-8, blend_exp=10.0),
                                dict(sim_name="NS", group="comparator", purpose="no-slip only",
                                     e_f=1.5, ktens=0.05, Cs=np.nan, Cq=np.nan, u0=np.nan, u_blend=np.nan, eps_blend=np.nan, blend_exp=np.nan),
                                dict(sim_name="LD-NIL", group="comparator", purpose="free-slip only",
                                     e_f=1.5, ktens=0.05, Cs=np.nan, Cq=np.nan, u0=np.nan, u_blend=np.nan, eps_blend=np.nan, blend_exp=np.nan),
                                dict(sim_name="LD-static-Cs5e-4", group="comparator", purpose="Liu-style static reference",
                                     e_f=1.5, ktens=0.05, Cs=5.0e-4, Cq=np.nan, u0=5.0e-5, u_blend=np.nan, eps_blend=np.nan, blend_exp=np.nan),
                                dict(sim_name="LD-quad-Cq75", group="comparator", purpose="quadratic mobile comparator",
                                     e_f=1.5, ktens=0.05, Cs=np.nan, Cq=75.0, u0=5.0e-5, u_blend=np.nan, eps_blend=np.nan, blend_exp=np.nan),
                                dict(sim_name="LD-linear-CL0p25", group="comparator", purpose="linear/Rayleigh comparator",
                                     e_f=1.5, ktens=0.05, Cs=np.nan, Cq=np.nan, u0=5.0e-5, u_blend=np.nan, eps_blend=np.nan, blend_exp=np.nan)])
display(EXPERIMENT_META)

## 0.1 analysis helpers

These helper functions wrap the `shuga` loaders and standardise how this notebook handles missing variables, spatial areas, fast-ice masks, diagnostics, and publication tables.

In [ ]:
def make_run(sim_name: str, start: str = RUN_START, end: str = RUN_END) -> RunSpec:
    return RunSpec(sim_name   = sim_name,
                   start_date = start,
                   end_date   = end,
                   hemisphere = HEMISPHERE,
                   project    = PROJECT,
                   user       = USER)


def make_classify(methods: Sequence[str] = (METHOD,)) -> ClassificationSpec:
    return ClassificationSpec(ice_type     = ICE_TYPE,
                              grid_type    = GRID_TYPE,
                              ispd_thresh  = ISPD_THRESH,
                              methods      = tuple(methods),
                              bin_window   = BIN_WINDOW,
                              bin_min_days = BIN_MIN_DAYS,
                              roll_window  = ROLL_WINDOW)

def list_month_groups(sim_name: str, root: Path = AFIM_ARCHIVE_ROOT) -> list[str]:
    zroot = root / sim_name / "zarr" / "iceh_daily.zarr"
    if not zroot.exists():
        # fall back to afim_output layout
        zroot = AFIM_OUTPUT_ROOT / sim_name / "zarr" / "iceh_daily.zarr"
    if not zroot.exists():
        return []
    return sorted(p.name for p in zroot.iterdir() if p.is_dir() and len(p.name) == 7 and p.name[4] == "-")

def list_group_variables(sim_name: str, group: str = "1993-10", root: Path = AFIM_ARCHIVE_ROOT) -> list[str]:
    zroot = root / sim_name / "zarr" / "iceh_daily.zarr" / group
    if not zroot.exists():
        zroot = AFIM_OUTPUT_ROOT / sim_name / "zarr" / "iceh_daily.zarr" / group
    if not zroot.exists():
        return []
    skip = {".", "..", ".zattrs", ".zgroup", ".zmetadata", "zarr.json"}
    return sorted(p.name for p in zroot.iterdir() if p.is_dir() and p.name not in skip)

def experiment_inventory(sims: Sequence[str] = CORE_SIMS, group: str = "1993-10") -> pd.DataFrame:
    rows = []
    for sim in sims:
        groups = list_month_groups(sim)
        vars_ = list_group_variables(sim, group=group)
        rows.append({"sim_name": sim,
                     "first_month": groups[0] if groups else np.nan,
                     "last_month": groups[-1] if groups else np.nan,
                     "n_months": len(groups),
                     "group_checked": group,
                     "n_vars_in_group": len(vars_),
                     "has_Ku_proxy": any(v in vars_ for v in ["KuxE", "KuxN", "KuyE", "KuyN"]),
                     "has_ocean_stress": all(v in vars_ for v in ["strocnx", "strocny"]),
                     "has_coriolis": all(v in vars_ for v in ["strcorx", "strcory"]),
                     "has_tilt": all(v in vars_ for v in ["strtltx", "strtlty"]),
                     "sample_variables": ", ".join(vars_[:18]) + (" ..." if len(vars_) > 18 else "")})
    return pd.DataFrame(rows)

def safe_load_cice(sim_name: str, start: str, end: str,
                   variables: Sequence[str] | None = None, *,
                   chunks: dict | None = None,
                   afim_output_root: Path | str | None = None,
                   cice_store: Path | str | None = None,
                   static_store: Path | str | None = None) -> xr.Dataset:
    run     = make_run(sim_name, start, end)
    classify = make_classify()
    return load_cice(run              = run,
                     classify         = classify,
                     variables        = list(variables) if variables is not None else None,
                     hemisphere       = HEMISPHERE,
                     chunks           = chunks or {"time": 31},
                     afim_output_root = afim_output_root,
                     cice_store       = cice_store,
                     static_store     = static_store)

def safe_load_classified(sim_name: str, start: str, end: str,
                         variables: Sequence[str] | None = ("FI_mask",), *,
                         method: str = METHOD,
                         chunks: dict | None = None) -> xr.Dataset:
    run      = make_run(sim_name, start, end)
    classify = make_classify((method,))
    return load_classified(run            = run,
                           classify       = classify,
                           classification = method,
                           variables      = list(variables) if variables is not None else None,
                           hemisphere     = HEMISPHERE,
                           grid_type      = GRID_TYPE,
                           chunks         = chunks or {"time": 31})

def safe_load_metrics(sim_name: str, start: str, end: str, *, method: str = METHOD, chunks: dict | None = None) -> xr.Dataset:
    run = make_run(sim_name, start, end)
    classify = make_classify((method,))
    return load_metrics(run            = run,
                        classify       = classify,
                        classification = method,
                        hemisphere     = HEMISPHERE,
                        grid_type      = GRID_TYPE,
                        chunks         = chunks or {"time": 31})

def mag(ds: xr.Dataset | Mapping[str, xr.DataArray], x: str, y: str, name: str | None = None) -> xr.DataArray | None:
    if x in ds and y in ds:
        out = xr.apply_ufunc(np.hypot, ds[x], ds[y], dask="allowed")
        if name:
            out = out.rename(name)
        return out
    return None

def first_present(ds: xr.Dataset, candidates: Sequence[str]) -> str | None:
    return next((v for v in candidates if v in ds), None)

def area_da(ds: xr.Dataset) -> xr.DataArray:
    for name in ("tarea", "uarea", "narea", "earea"):
        if name in ds:
            return ds[name]
    raise KeyError("No recognised area field found. Expected one of tarea/uarea/narea/earea.")

def ice_speed(ds: xr.Dataset) -> xr.DataArray:
    out = mag(ds, "uvel", "vvel", "ice_speed")
    if out is None:
        raise KeyError("uvel/vvel are required for ice_speed.")
    out.attrs.update(units="m s-1", long_name="Ice speed")
    return out

def rel_ice_ocean_speed(ds: xr.Dataset) -> xr.DataArray | None:
    if all(v in ds for v in ["uvel", "vvel", "uocn", "vocn"]):
        out = xr.apply_ufunc(np.hypot, ds["uvel"] - ds["uocn"], ds["vvel"] - ds["vocn"], dask="allowed")
        out = out.rename("rel_ice_ocean_speed")
        out.attrs.update(units="m s-1", long_name="Relative ice-ocean speed")
        return out
    return None

def strain_invariant(ds: xr.Dataset) -> xr.DataArray | None:
    """Prefer existing CICE invariants. Fallback to sqrt(divu^2 + shear^2)."""
    if "sigP" in ds:
        return ds["sigP"].rename("strain_invariant")
    if all(v in ds for v in ["divu", "shear"]):
        return xr.apply_ufunc(np.hypot, ds["divu"], ds["shear"], dask="allowed").rename("strain_invariant")
    if "shear" in ds:
        return abs(ds["shear"]).rename("strain_invariant")
    return None

def ku_proxy(ds: xr.Dataset) -> xr.Dataset:
    """
    Return a dataset containing lateral-drag coefficient/acceleration proxies.

    The current history files expose edge-oriented variables such as KuxE/KuyE and
    KuxN/KuyN. This helper averages available edge fields to a T-grid-like proxy.
    Verify exact units/sign in the branch before treating tau_ld_est as a formal stress.
    """
    out = xr.Dataset()
    x_fields = [v for v in ("KuxE", "KuxN") if v in ds]
    y_fields = [v for v in ("KuyE", "KuyN") if v in ds]
    if x_fields:
        out["ld_x_proxy"] = xr.concat([ds[v] for v in x_fields], dim="edge_x").mean("edge_x")
    if y_fields:
        out["ld_y_proxy"] = xr.concat([ds[v] for v in y_fields], dim="edge_y").mean("edge_y")
    if "ld_x_proxy" in out and "ld_y_proxy" in out:
        out["ld_mag_proxy"] = xr.apply_ufunc(np.hypot, out["ld_x_proxy"], out["ld_y_proxy"], dask="allowed")
    elif "ld_x_proxy" in out:
        out["ld_mag_proxy"] = abs(out["ld_x_proxy"])
    elif "ld_y_proxy" in out:
        out["ld_mag_proxy"] = abs(out["ld_y_proxy"])
    if "ld_mag_proxy" in out:
        out["ld_mag_proxy"].attrs.update(long_name="Lateral-drag K/acceleration proxy from Kux/Kuy history fields")
    return out

def compute_diagnostic_terms(ds: xr.Dataset) -> xr.Dataset:
    """Build a compact diagnostic dataset from one experiment's CICE history fields."""
    out = xr.Dataset(coords={k: v for k, v in ds.coords.items() if k in ("time", "ni", "nj")})
    if all(v in ds for v in ["uvel", "vvel"]):
        out["ice_speed"] = ice_speed(ds)
    rel = rel_ice_ocean_speed(ds)
    if rel is not None:
        out["rel_ice_ocean_speed"] = rel
    strain = strain_invariant(ds)
    if strain is not None:
        out["strain_invariant"] = strain
    for label, x, y in [("tau_air", "strairx", "strairy"),
                        ("tau_ocean", "strocnx", "strocny"),
                        ("tau_internal", "strintx", "strinty"),
                        ("tau_coriolis", "strcorx", "strcory"),
                        ("tau_tilt", "strtltx", "strtlty")]:
        z = mag(ds, x, y, label)
        if z is not None:
            out[label] = z
            out[label].attrs.update(units="nominal Pa", long_name=label)
    ku = ku_proxy(ds)
    for v in ku.data_vars:
        out[v] = ku[v]
    if "ld_mag_proxy" in out and "hi" in ds:
        # If Kux/Kuy are accelerations [m s-2], rho_i * hi * K gives Pa.
        h_eff = ds["hi"] * ds.get("aice", 1.0)
        out["tau_ld_est"] = (RHO_ICE * h_eff * out["ld_mag_proxy"]).rename("tau_ld_est")
        out["tau_ld_est"].attrs.update(units="Pa if Kux/Kuy are accelerations",
                                       long_name="Estimated lateral-drag stress magnitude")
    if all(v in out for v in ["tau_ld_est", "tau_air", "tau_ocean", "tau_internal"]):
        denom = out["tau_air"] + out["tau_ocean"] + out["tau_internal"] + EPS
        out["R_ld_budget"] = (out["tau_ld_est"] / denom).rename("R_ld_budget")
        out["R_ld_budget"].attrs.update(long_name="tau_ld_est/(tau_air+tau_ocean+tau_internal)")
    if all(v in out for v in ["ld_x_proxy", "ld_y_proxy"]) and all(v in ds for v in ["uvel", "vvel", "hi"]):
        h_eff = ds["hi"] * ds.get("aice", 1.0)
        tau_x = RHO_ICE * h_eff * out["ld_x_proxy"]
        tau_y = RHO_ICE * h_eff * out["ld_y_proxy"]
        out["P_ld_est"] = (tau_x * ds["uvel"] + tau_y * ds["vvel"]).rename("P_ld_est")
        out["P_ld_est"].attrs.update(units="W m-2 if Kux/Kuy signs are stress accelerations",
                                     long_name="Estimated lateral-drag power tau_ld · u")
    return out

def area_weighted_mean(da: xr.DataArray, area: xr.DataArray, mask: xr.DataArray | None = None, dims: Sequence[str] | None = None) -> xr.DataArray:
    if dims is None:
        dims = [d for d in area.dims if d in da.dims]
    work = da.where(mask) if mask is not None else da
    w = area.where(np.isfinite(work))
    return (work * w).sum(dim=dims, skipna=True) / w.sum(dim=dims, skipna=True)

def area_sum(mask_or_field: xr.DataArray, area: xr.DataArray, dims: Sequence[str] | None = None) -> xr.DataArray:
    if dims is None:
        dims = [d for d in area.dims if d in mask_or_field.dims]
    return (mask_or_field.astype("float64") * area).sum(dim=dims, skipna=True)

def compute_fip(mask: xr.DataArray) -> xr.DataArray:
    return mask.astype("float32").mean("time", skipna=True).rename("FIP")

def robust_align_time(*arrays_or_dsets):
    return xr.align(*arrays_or_dsets, join="inner")

def sample_points_for_phase_space(ds: xr.Dataset, variables: Sequence[str],
                                  mask: xr.DataArray | None = None,
                                  sample_frac: float = SAMPLE_FRAC,
                                  max_points: int = 200_000) -> pd.DataFrame:
    """Convert selected variables to a manageable sampled dataframe.

    This function avoids building a dataframe from the full CICE grid. It first
    applies a regular spatial stride based on ``sample_frac`` and then applies
    a random sample after dataframe conversion if the result is still large.
    """
    present = [v for v in variables if v in ds]
    if not present:
        return pd.DataFrame()
    sub = ds[present]
    if mask is not None:
        sub = sub.where(mask)
    spatial_dims = [d for d in sub.dims if d not in ("time", "edge", "edge_x", "edge_y")]
    if 0 < sample_frac < 1 and spatial_dims:
        stride = max(1, int(np.ceil(1.0 / np.sqrt(sample_frac))))
        indexer = {d: slice(None, None, stride) for d in spatial_dims}
        sub = sub.isel(indexer)
    df = sub.to_dataframe().reset_index()
    df = df.dropna(subset=present, how="all")
    if len(df) > max_points:
        df = df.sample(n=max_points, random_state=RNG_SEED)
    return df

def plot_timeseries_df(df: pd.DataFrame, x: str, y: str, hue: str, *, title: str, ylabel: str, out: Path | None = None):
    fig, ax = plt.subplots(figsize=(13, 5))
    for key, part in df.groupby(hue):
        ax.plot(part[x], part[y], label=str(key), lw=1.7)
    ax.set_title(title)
    ax.set_xlabel(x)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    if out is not None:
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=180)
        print(f"Saved: {out}")
    if SHOW_FIGS:
        plt.show()
    else:
        plt.close(fig)
    return fig

def display_df(df: pd.DataFrame, n: int = 20):
    with pd.option_context("display.max_columns", 60, "display.width", 240):
        display(df.head(n))

# ---------------------------------------------------------------------
# some helpful extensions
def present_vars(ds: xr.Dataset, candidates: Sequence[str]) -> list[str]:
    return [v for v in candidates if v in ds]

def edge_mean(ds: xr.Dataset, stem: str, *, name: str | None = None) -> xr.DataArray | None:
    """Mean E/N edge diagnostics to a plotting/summary proxy.

    Expected names are e.g. stem='ldphi' -> ldphiE/ldphiN.
    """
    fields = [f"{stem}E", f"{stem}N"]
    arrs = [ds[v] for v in fields if v in ds]
    if not arrs:
        return None
    out = xr.concat(arrs, dim="edge").mean("edge", skipna=True)
    return out.rename(name or stem)

def add_blend_diagnostics(ds: xr.Dataset) -> xr.Dataset:
    """Collect blend-specific diagnostics if present."""
    out = xr.Dataset(coords={k: v for k, v in ds.coords.items() if k in ("time", "ni", "nj")})
    for stem in ["ldphi", "ldwgt", "ldeps", "ldspd", "ldpstat", "ldpquad", "ldplin"]:
        da = edge_mean(ds, stem)
        if da is not None:
            out[stem] = da
    kx = edge_mean(ds, "Kux", name="Kux")
    ky = edge_mean(ds, "Kuy", name="Kuy")
    if kx is not None and ky is not None:
        out["tau_ld_mag"] = xr.apply_ufunc(np.hypot, kx, ky, dask="allowed").rename("tau_ld_mag")
        if "uvel" in ds and "vvel" in ds:
            out["P_ld"] = (kx * ds["uvel"] + ky * ds["vvel"]).rename("P_ld")
    return out

def load_pub_cice(sim_name: str, start: str = ANALYSIS_START, end: str = ANALYSIS_END, variables: Sequence[str] | None = None) -> xr.Dataset:
    return safe_load_cice(sim_name, start, end, variables=variables)

def load_pub_classified(sim_name: str, start: str = ANALYSIS_START, end: str = ANALYSIS_END, variables: Sequence[str] | None = ("FI_mask",)) -> xr.Dataset:
    return safe_load_classified(sim_name, start, end, variables=variables, method=METHOD)

def daily_fia_from_mask(sim_name: str, start: str = ANALYSIS_START, end: str = ANALYSIS_END) -> xr.DataArray:
    cice = load_pub_cice(sim_name, start, end, variables=["tarea", "TLON", "TLAT"])
    cls = load_pub_classified(sim_name, start, end, variables=["FI_mask"])
    cice, cls = robust_align_time(cice, cls)
    return (area_sum(cls["FI_mask"].astype(bool), area_da(cice)) / 1e9).rename("FIA_10^3_km2")

def safe_to_series(da: xr.DataArray, value_name: str) -> pd.DataFrame:
    s = da.compute().to_series().reset_index()
    if s.shape[1] >= 2:
        s = s.rename(columns={s.columns[-1]: value_name})
    return s

def load_af2020_mask(start: str = ANALYSIS_START, end: str = ANALYSIS_END, reference_sim: str | None = None) -> xr.DataArray:
    """Load AF2020 regridded fast-ice mask on the model grid."""
    ref = reference_sim or next((s for s in CORE_SIMS if list_month_groups(s)), CORE_SIMS[0])
    run = make_run(ref, start, end)
    obs = SeaIceObservations(run)
    ds = obs.load_af2020_regridded()
    var_name = getattr(obs.observations, "af2020_regridded_var", None)
    if var_name not in ds:
        var_name = next(iter(ds.data_vars))
    da = ds[var_name]
    if "time" in da.dims:
        da = da.sel(time=slice(start, end))
    return (da > 0).rename("AF2020_mask")

def binary_spatial_skill(model_mask: xr.DataArray, obs_mask: xr.DataArray, area: xr.DataArray) -> pd.DataFrame:
    model_mask, obs_mask = xr.align(model_mask.astype(bool), obs_mask.astype(bool), join="inner")
    hit = model_mask & obs_mask
    miss = (~model_mask) & obs_mask
    false = model_mask & (~obs_mask)
    correct_reject = (~model_mask) & (~obs_mask)
    hit_a = area_sum(hit, area) / 1e9
    miss_a = area_sum(miss, area) / 1e9
    false_a = area_sum(false, area) / 1e9
    cr_a = area_sum(correct_reject, area) / 1e9
    precision = hit_a / (hit_a + false_a + EPS)
    recall = hit_a / (hit_a + miss_a + EPS)
    f1 = 2 * precision * recall / (precision + recall + EPS)
    jaccard = hit_a / (hit_a + miss_a + false_a + EPS)
    far = false_a / (hit_a + false_a + EPS)
    ds = xr.Dataset({"hit_area": hit_a,
                     "miss_area": miss_a,
                     "false_alarm_area": false_a,
                     "correct_reject_area": cr_a,
                     "precision": precision,
                     "recall": recall,
                     "F1": f1,
                     "Jaccard": jaccard,
                     "false_alarm_ratio": far})
    return ds.to_dataframe().reset_index()

def experiment_inventory_pub(sims: Sequence[str] = CORE_SIMS, group: str = "1994-10") -> pd.DataFrame:
    return experiment_inventory(sims, group=group)


## 0.2 Inventory check

Run this before analysis. It confirms which experiments have been converted to daily Zarr, which months exist, and which diagnostic fields are available. The `blend_strain` experiments should show the extra `ld*` fields once the history-output plumbing is correct.

In [ ]:
INV_1994_10 = experiment_inventory_pub(CORE_SIMS, group="1994-10")
display_df(INV_1994_10, n=100)
INV_1994_10.to_csv(NOTEBOOK_FIG_ROOT / "inventory_1994-10.csv", index=False)
for sim in CORE_SIMS:
    print("\n" + "="*100)
    report_sim_status(sim_name=sim)

# 1. Free-slip versus no-slip: conceptual model

This section is the conceptual entry point for the paper. The purpose is not to reproduce the full CICE stress solver, but to illustrate why a C-grid free-slip boundary condition is a necessary starting point for a lateral-drag parameterisation.

A no-slip boundary condition suppresses tangential motion at land/ice-shelf boundaries. That can artificially immobilise coastal ice and confound the interpretation of landfast ice. A free-slip boundary condition removes the tangential constraint and allows coastal-parallel motion. Lateral drag can then be added explicitly through a form factor and form function, so that landfast ice emerges where the ice state and anchoring geometry support it.

## Compact 2 x 2 C-grid/T-grid shear toy model

Cell layout:

   | c2 | c3 |
   |----|----|
   | c1 | c4 |

flows order:
```
   ((u0, v0), (u1, v1), (u2, v2), (u3, v3))
    c1         c2         c3         c4
```

In [ ]:
from matplotlib.patches import Rectangle, Circle
LAND       = "__LAND__"
OCEAN      = "__OCEAN__"
CELL_NAMES = ("c1", "c2", "c3", "c4")
CELL_XY    = {"c1": (0, 0),
              "c2": (0, 1),
              "c3": (1, 1),
              "c4": (1, 0)}

### Declarative edge stencils

* U-point shear stencil: $d(u_E)/dy + d(v_N)/dx$
* T-centred shear stencil: $d(u_N)/dy + d(v_E)/dx$

This preserves convention: bottom/left/top exterior edges are *land-like*, right exterior edges *ocean-like*

In [ ]:
U_E_SPECS = {"E(i,j)"      : ("c1", "c4"),
             "E(i+1,j)"    : ("c4", OCEAN),
             "E(i,j+1)"    : ("c2", "c3"),
             "E(i+1,j+1)"  : ("c3", OCEAN),
             "E(i,j+2)"    : ("c2", "c3"),
             "E(i+1,j+2)"  : ("c3", OCEAN)}
U_N_SPECS = {"N(i,j)"      : ("c1", "c2"),
             "N(i+1,j)"    : ("c4", "c3"),
             "N(i,j+1)"    : ("c2", OCEAN),
             "N(i+1,j+1)"  : ("c3", OCEAN)}
T_E_SPECS = {"E(i-1,j)"    : (LAND, "c1"),
             "E(i,j)"      : ("c1", "c4"),
             "E(i+1,j)"    : ("c4", OCEAN),
             "E(i-1,j+1)"  : (LAND, "c2"),
             "E(i,j+1)"    : ("c2", "c3"),
             "E(i+1,j+1)"  : ("c3", OCEAN)}
T_N_SPECS = {"N(i,j-1)"    : (LAND, "c1"),
             "N(i+1,j-1)"  : (LAND, "c4"),
             "N(i,j)"      : ("c1", "c2"),
             "N(i+1,j)"    : ("c4", "c3"),
             "N(i,j+1)"    : ("c2", LAND),
             "N(i+1,j+1)"  : ("c3", LAND)}

### core helpers

In [ ]:
def _validate_method(method):
    if method not in ("free-slip", "no-slip"):
        raise ValueError("method must be 'free-slip' or 'no-slip'")


def _as_maps(land_tuple, flows):
    if len(land_tuple) != 4:
        raise ValueError("land_tuple must contain four booleans: c1, c2, c3, c4")
    if len(flows) != 4:
        raise ValueError("flows must contain four (u, v) pairs: c1, c2, c3, c4")

    land = {name: bool(v) for name, v in zip(CELL_NAMES, land_tuple)}
    flow = {name: tuple(map(float, uv)) for name, uv in zip(CELL_NAMES, flows)}
    return land, flow


def _edge_mask(spec, land):
    """
    Edge is valid only when all adjacent real/virtual cells are ocean.
    Virtual OCEAN validates the edge but contributes no velocity value.
    Virtual LAND masks the edge.
    """
    for node in spec:
        if node == LAND:
            return 0
        if node == OCEAN:
            continue
        if land[node]:
            return 0
    return 1


def _edge_component(spec, land, flow, comp, mask):
    """
    Edge velocity = mean of adjacent real ocean-cell values.
    Masked edges are zero.
    """
    if mask == 0:
        return 0.0

    vals = []
    for node in spec:
        if node in (LAND, OCEAN):
            continue
        if not land[node]:
            vals.append(flow[node][comp])

    return float(np.mean(vals)) if vals else 0.0


def _build_edge_family(specs, land, flow):
    mask = {k: _edge_mask(spec, land) for k, spec in specs.items()}
    u = {k: _edge_component(spec, land, flow, 0, mask[k]) for k, spec in specs.items()}
    v = {k: _edge_component(spec, land, flow, 1, mask[k]) for k, spec in specs.items()}
    return mask, u, v


def _build_edges(land_tuple, flows, stencil):
    land, flow = _as_maps(land_tuple, flows)

    if stencil == "U":
        e_specs, n_specs = U_E_SPECS, U_N_SPECS
    elif stencil == "T":
        e_specs, n_specs = T_E_SPECS, T_N_SPECS
    else:
        raise ValueError("stencil must be 'U' or 'T'")

    epm, uE, vE = _build_edge_family(e_specs, land, flow)
    npm, uN, vN = _build_edge_family(n_specs, land, flow)

    return epm, npm, uE, vE, uN, vN


def _forward_diff(high_key, low_key, values, masks, method):
    """
    One-sided/forward difference used at U points.

    If the high-side edge is missing:
      free-slip -> mirror low-side value
      no-slip   -> zero
    """
    _validate_method(method)

    low_eff = values[low_key] if masks.get(low_key, 0) == 1 else 0.0

    if high_key is None:
        high_eff = low_eff if method == "free-slip" else 0.0
    elif masks.get(high_key, 0) == 1:
        high_eff = values[high_key]
    else:
        high_eff = low_eff if method == "free-slip" else 0.0

    return float(high_eff - low_eff)


def _centred_diff(high_key, low_key, values, masks, method):
    """
    Centred difference used at T cells.

    For free-slip, a missing side mirrors the opposite existing side.
    For no-slip, a missing side is zero.
    """
    _validate_method(method)

    high_present = masks.get(high_key, 0) == 1
    low_present = masks.get(low_key, 0) == 1

    high_val = values.get(high_key, 0.0)
    low_val = values.get(low_key, 0.0)

    if method == "free-slip":
        high_eff = high_val if high_present else (low_val if low_present else 0.0)
        low_eff = low_val if low_present else (high_val if high_present else 0.0)
    else:
        high_eff = high_val if high_present else 0.0
        low_eff = low_val if low_present else 0.0

    return float(high_eff - low_eff)


def sign_label(x, atol=1e-12):
    x = float(x)
    return "0" if np.isclose(x, 0.0, atol=atol) else (">0" if x > 0 else "<0")

### compute functions

In [ ]:
def compute_shears_for_case(land_tuple, flows, method):
    """
    Return U-point shear = d(u_E)/dy + d(v_N)/dx.

    Compatible with:
        data = compute_shears_for_case(land_tuple, flows, method)
        data["shear"]
    """
    _validate_method(method)

    epm, npm, uE, vE, uN, vN = _build_edges(land_tuple, flows, stencil="U")

    shear_specs = {
        "U(i,j)"      : (("E(i,j+1)",   "E(i,j)"),     ("N(i+1,j)",   "N(i,j)")),
        "U(i+1,j)"    : (("E(i+1,j+1)", "E(i+1,j)"),   (None,         "N(i+1,j)")),
        "U(i,j+1)"    : (("E(i,j+2)",   "E(i,j+1)"),   ("N(i+1,j+1)", "N(i,j+1)")),
        "U(i+1,j+1)"  : (("E(i+1,j+2)", "E(i+1,j+1)"), (None,         "N(i+1,j+1)")),
    }

    shear = {}
    for key, ((e_hi, e_lo), (n_hi, n_lo)) in shear_specs.items():
        du_dy = _forward_diff(e_hi, e_lo, uE, epm, method)
        dv_dx = _forward_diff(n_hi, n_lo, vN, npm, method)
        shear[key] = du_dy + dv_dx

    return {
        "epm": epm,
        "npm": npm,
        "uE": uE,
        "vE": vE,
        "uN": uN,
        "vN": vN,
        "shear": shear,
    }


def compute_shearT_case(land_tuple, flows, method):
    """
    Return T-centred shear = d(u_N)/dy + d(v_E)/dx.

    Compatible with:
        data = compute_shearT_case(land_tuple, flows, method)
        data["shearT"]
    """
    _validate_method(method)

    epm, npm, uE, vE, uN, vN = _build_edges(land_tuple, flows, stencil="T")

    shearT_specs = {
        "T(i,j)"      : (("N(i,j)",     "N(i,j-1)"),   ("E(i,j)",     "E(i-1,j)")),
        "T(i+1,j)"    : (("N(i+1,j)",   "N(i+1,j-1)"), ("E(i+1,j)",   "E(i,j)")),
        "T(i,j+1)"    : (("N(i,j+1)",   "N(i,j)"),     ("E(i,j+1)",   "E(i-1,j+1)")),
        "T(i+1,j+1)"  : (("N(i+1,j+1)", "N(i+1,j)"),   ("E(i+1,j+1)", "E(i,j+1)")),
    }

    shearT = {}
    for key, ((n_hi, n_lo), (e_hi, e_lo)) in shearT_specs.items():
        du_dy = _centred_diff(n_hi, n_lo, uN, npm, method)
        dv_dx = _centred_diff(e_hi, e_lo, vE, epm, method)
        shearT[key] = du_dy + dv_dx

    return {
        "epm": epm,
        "npm": npm,
        "uE": uE,
        "vE": vE,
        "uN": uN,
        "vN": vN,
        "shearT": shearT,
    }

### plotting functions

In [ ]:
def _sgn_for_title(x):
    x = float(x)
    return "=0" if np.isclose(x, 0.0) else (">0" if x > 0 else "<0")


def _case_title(case_name, flows, method):
    (U0, V0), (U1, V1), (U2, V2), (U3, V3) = flows
    return (
        f"{case_name} | "
        f"c1(u,v)=({_sgn_for_title(U0)},{_sgn_for_title(V0)})  "
        f"c2=({_sgn_for_title(U1)},{_sgn_for_title(V1)})  "
        f"c3=({_sgn_for_title(U2)},{_sgn_for_title(V2)})  "
        f"c4=({_sgn_for_title(U3)},{_sgn_for_title(V3)})"
        f" | method: {method}"
    )


def _plot_cells(ax, land_tuple):
    for cname, is_land in zip(CELL_NAMES, land_tuple):
        x0, y0 = CELL_XY[cname]
        ax.add_patch(
            Rectangle(
                (x0, y0), 1, 1,
                fill=False,
                hatch="//" if is_land else None,
            )
        )
        ax.text(x0 + 0.5, y0 + 0.5, cname, ha="center", va="center", fontsize=9)


def _plot_edge_arrows(ax, centers, masks, u, v, label_offset):
    for key, (x, y) in centers.items():
        alpha = 1.0 if masks.get(key, 0) == 1 else 0.2

        ax.arrow(
            x, y,
            0.20 * u.get(key, 0.0), 0.0,
            length_includes_head=True,
            head_width=0.06,
            color="blue",
            alpha=alpha,
        )
        ax.arrow(
            x, y,
            0.0, 0.20 * v.get(key, 0.0),
            length_includes_head=True,
            head_width=0.06,
            color="red",
            alpha=alpha,
        )

        dx, dy, rot = label_offset(key)
        ax.text(
            x + dx, y + dy, key,
            ha="center",
            va="center",
            fontsize=8,
            rotation=rot,
        )

def draw_case(case_name, land_tuple, data, flows, method):
    """Plot U-point shear stencil."""
    epm, npm = data["epm"], data["npm"]
    uE, vE, uN, vN = data["uE"], data["vE"], data["uN"], data["vN"]

    fig, ax = plt.subplots(figsize=(6, 6))
    _plot_cells(ax, land_tuple)

    E_centers = {
        "E(i,j)"      : (1.0, 0.5),
        "E(i+1,j)"    : (2.0, 0.5),
        "E(i,j+1)"    : (1.0, 1.5),
        "E(i+1,j+1)"  : (2.0, 1.5),
        "E(i,j+2)"    : (1.0, 2.5),
        "E(i+1,j+2)"  : (2.0, 2.5),
    }

    N_centers = {
        "N(i,j)"      : (0.5, 1.0),
        "N(i+1,j)"    : (1.5, 1.0),
        "N(i,j+1)"    : (0.5, 2.0),
        "N(i+1,j+1)"  : (1.5, 2.0),
    }

    _plot_edge_arrows(
        ax, E_centers, epm, uE, vE,
        lambda k: (0.0, -0.18, 0),
    )
    _plot_edge_arrows(
        ax, N_centers, npm, uN, vN,
        lambda k: (-0.22, 0.0, 90),
    )

    U_points = {
        "U(i,j)"      : (1.0, 1.0),
        "U(i+1,j)"    : (2.0, 1.0),
        "U(i,j+1)"    : (1.0, 2.0),
        "U(i+1,j+1)"  : (2.0, 2.0),
    }

    for uname, (x, y) in U_points.items():
        is_primary = uname == "U(i,j)"
        ax.add_patch(
            Circle(
                (x, y),
                0.06 if is_primary else 0.045,
                color="royalblue" if is_primary else "grey",
            )
        )
        ax.text(
            x,
            y - (0.29 if is_primary else 0.24),
            f"{uname}: {sign_label(data['shear'][uname])}",
            ha="center",
            va="center",
            fontsize=8,
        )

    ax.set_title(_case_title(case_name, flows, method), fontsize=10)
    ax.set_xlim(-0.1, 2.6)
    ax.set_ylim(-0.1, 2.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()

    return fig, ax


def draw_case_T(case_name, land_tuple, data, flows, method):
    """Plot T-cell shear stencil."""
    epm, npm = data["epm"], data["npm"]
    uE, vE, uN, vN = data["uE"], data["vE"], data["uN"], data["vN"]

    fig, ax = plt.subplots(figsize=(6, 6))
    _plot_cells(ax, land_tuple)

    E_centers = {
        "E(i-1,j)"    : (0.0, 0.5),
        "E(i,j)"      : (1.0, 0.5),
        "E(i+1,j)"    : (2.0, 0.5),
        "E(i-1,j+1)"  : (0.0, 1.5),
        "E(i,j+1)"    : (1.0, 1.5),
        "E(i+1,j+1)"  : (2.0, 1.5),
    }

    N_centers = {
        "N(i,j-1)"    : (0.5, 0.0),
        "N(i,j)"      : (0.5, 1.0),
        "N(i,j+1)"    : (0.5, 2.0),
        "N(i+1,j-1)"  : (1.5, 0.0),
        "N(i+1,j)"    : (1.5, 1.0),
        "N(i+1,j+1)"  : (1.5, 2.0),
    }

    _plot_edge_arrows(
        ax, E_centers, epm, uE, vE,
        lambda k: (0.0, 0.18 if "j+1" in k else -0.18, 0),
    )
    _plot_edge_arrows(
        ax, N_centers, npm, uN, vN,
        lambda k: (0.22 if "i+1" in k else -0.22, 0.0, 90),
    )

    T_points = {
        "T(i,j)"      : (0.5, 0.5),
        "T(i+1,j)"    : (1.5, 0.5),
        "T(i,j+1)"    : (0.5, 1.5),
        "T(i+1,j+1)"  : (1.5, 1.5),
    }

    for tname, (x, y) in T_points.items():
        ax.add_patch(Circle((x, y), 0.05, color="royalblue"))
        ax.text(
            x,
            y - 0.22,
            f"{tname}: {sign_label(data['shearT'][tname])}",
            ha="center",
            va="center",
            fontsize=9,
        )

    ax.set_title(_case_title(case_name, flows, method), fontsize=10)
    ax.set_xlim(-0.4, 2.4)
    ax.set_ylim(-0.4, 2.4)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()

    return fig, ax

### conceptual figures

In [ ]:
# Choose boundary treatment
method = "free-slip"   # "free-slip" (Neumann mirror) or "no-slip" (Dirichlet zero)
# c1 (bottom-left), c2 (top-left), c3 (top-right), c4 (bottom-right)
u0, v0 = -1.0, 0.0    # c1
u1, v1 =  1.0, 1.0    # c2
u2, v2 =  1.0, -1.0   # c3
u3, v3 = -1.0, 0.0    # c4
cases  = [("Case 1: open ocean",                (False, False, False, False )),
          ("Case 2: c2 & c3 land (top row)",    (False, True , True , False )),
          ("Case 3: c3 & c4 land (right col.)", (False, False, True , True  )),
          ("Case 4: c1 land (bottom-left)",     (True , False, False, False )),
          ("Case 5: c4 land (bottom-right)",    (False, False, False, True  )),
          ("Case 6: c3 land (upper-right)",     (False, False, True , False )),
          ("Case 6: c3 land (upper-left)",      (False, True , False, False ))]
# -------------------- run and display --------------------
flows = ((u0,v0),(u1,v1),(u2,v2),(u3,v3))
rows = []
for case_name, land_tuple in cases:
    data = compute_shears_for_case(land_tuple, flows, method)
    rows.append({"case"  : case_name,
                 "method": method,
                 **{f"shear_{k}": v for k,v in data["shear"].items()}})
    draw_case(case_name, land_tuple, data, flows, method)
for case_name, land_tuple in cases:
    data = compute_shearT_case(land_tuple, flows, method)
    rows.append({"case"  : case_name,
                 "method": method,
                 **{f"shearT_{k}": v for k,v in data["shearT"].items()}})
    draw_case_T(case_name, land_tuple, data, flows, method)

## plot results from small CICE uniform grid test case

### config

In [ ]:
from matplotlib.animation import FFMpegWriter
import matplotlib.clrs_dict as mcolors
import glob
RUN_FREE     = Path("/g/data/gv90/da1339/cice-dirs/runs/BCtest1_free/history")
RUN_NOSLIP   = Path("/g/data/gv90/da1339/cice-dirs/runs/BCtest1_noslip/history")
PATTERN      = "iceh_inst.2005-01-0[1-2]-*.nc"   # 48 hourly frames over two days
files_free   = sorted(glob.glob(str(RUN_FREE / PATTERN)))
files_noslip = sorted(glob.glob(str(RUN_NOSLIP / PATTERN)))
assert files_free and len(files_free)==len(files_noslip), "Check paths/patterns."

### load and assign

In [ ]:
dsF        = xr.open_mfdataset(files_free,   combine="nested", concat_dim="time").load()
dsN        = xr.open_mfdataset(files_noslip, combine="nested", concat_dim="time").load()
ydim, xdim = [d for d in dsF["divu_1"].dims if d != "time"]
ny, nx     = dsF.sizes[ydim], dsF.sizes[xdim]
# Shorthands
tmaskF, emaskF, nmaskF = dsF["tmask"], dsF["emask"], dsF["nmask"]
tmaskN, emaskN, nmaskN = dsN["tmask"], dsN["emask"], dsN["nmask"]
divF,  divN            = dsF["divu_1"],  dsN["divu_1"]          # T-grid
sheF,  sheN            = dsF["shear_1"], dsN["shear_1"]         # T-grid
sigPF, sigPN           = dsF["sigP_1"],  dsN["sigP_1"]          # T-grid (alt background)
uE_F,  uE_N            = dsF["uvelE_1"], dsN["uvelE_1"]         # E-grid (horizontal)
vN_F,  vN_N            = dsF["vvelN_1"], dsN["vvelN_1"]         # N-grid (vertical)

### Native coordinates (index space)

In [ ]:
xE = (np.arange(nx)[None, :] + 1) * np.ones((ny, nx))
yE = (np.arange(ny)[:, None] + 0.5) * np.ones((ny, nx))
xN = (np.arange(nx)[None, :] + 0.5) * np.ones((ny, nx))
yN = (np.arange(ny)[:, None] + 1)   * np.ones((ny, nx))

### Scales

In [ ]:
DIV_VMIN   = float(xr.concat([divF, divN], "run").quantile(0.02))
DIV_VMAX   = float(xr.concat([divF, divN], "run").quantile(0.98))
SHE_VMIN   = float(xr.concat([sheF, sheN], "run").quantile(0.02))
SHE_VMAX   = float(xr.concat([sheF, sheN], "run").quantile(0.98))
SIG_VMIN   = float(xr.concat([sigPF, sigPN], "run").quantile(0.02))
SIG_VMAX   = float(xr.concat([sigPF, sigPN], "run").quantile(0.98))
UMAX       = float(np.nanmax([np.abs(uE_F.values), np.abs(uE_N.values), np.abs(vN_F.values), np.abs(vN_N.values)]))
quiv_scale = 8.0/UMAX if UMAX>0 else 1.0

### norms (diverging only if zero is inside the range)

In [ ]:
if DIV_VMIN < 0.0 < DIV_VMAX:
    div_norm = mcolors.TwoSlopeNorm(vmin=DIV_VMIN, vcenter=0.0, vmax=DIV_VMAX)
else:
    div_norm = mcolors.Normalize(vmin=DIV_VMIN, vmax=DIV_VMAX)
she_norm = mcolors.Normalize(vmin=SHE_VMIN, vmax=SHE_VMAX)
sig_norm = mcolors.Normalize(vmin=SIG_VMIN, vmax=SIG_VMAX)

### helper functions

In [ ]:
def Tmask(arr, mask, k):  # -> (ny,nx) float with NaNs
    return arr.isel(time=k).where(mask.isel(time=k)==1).transpose(ydim, xdim).values

def Emask(arr, mask, k):  # -> masked array U for E-grid quiver
    U = arr.isel(time=k).where(mask.isel(time=k)==1).transpose(ydim, xdim).values
    U = ma.masked_invalid(U)  # mask NaNs so quiver skips them
    return U

def Nmask(arr, mask, k):  # -> masked array V for N-grid quiver
    V = arr.isel(time=k).where(mask.isel(time=k)==1).transpose(ydim, xdim).values
    V = ma.masked_invalid(V)
    return V

def fmt_time(t_da, k):
    try:
        return t_da.isel(time=k).dt.strftime("%Y-%m-%d %H:%M:%S").item()
    except Exception:
        t = t_da.values[k]
        return t.strftime("%Y-%m-%d %H:%M:%S") if hasattr(t, "strftime") else str(t)

def horiz(U):  # horizontal-only (u, 0) with matching mask
    return U, ma.masked_array(np.zeros_like(U), mask=U.mask)

def vert(V):   # vertical-only (0, v)
    return ma.masked_array(np.zeros_like(V), mask=V.mask), V

### figure creation

In [ ]:
fig = plt.figure(figsize=(12,10))
ax = [plt.subplot(2,2,1), plt.subplot(2,2,2), plt.subplot(2,2,3), plt.subplot(2,2,4)]
for a in ax:
    a.set_xlim(0, nx); a.set_ylim(0, ny); a.set_aspect('equal', adjustable='box')
ax[0].set_title("Free-slip — divu_1 + native (uE, vN)")
ax[1].set_title("No-slip — divu_1 + native (uE, vN)")
ax[2].set_title("Free-slip — shear_1 (or sigP_1) + native (uE, vN)")
ax[3].set_title("No-slip — shear_1 (or sigP_1) + native (uE, vN)")
use_sigP = False
norm_bot = sig_norm if use_sigP else she_norm
im_top_L = ax[0].imshow(np.full((ny,nx), np.nan), origin="lower", extent=[0,nx,0,ny], norm=div_norm)
im_top_R = ax[1].imshow(np.full((ny,nx), np.nan), origin="lower", extent=[0,nx,0,ny], norm=div_norm)
im_bot_L = ax[2].imshow(np.full((ny,nx), np.nan), origin="lower", extent=[0,nx,0,ny], norm=norm_bot)
im_bot_R = ax[3].imshow(np.full((ny,nx), np.nan), origin="lower", extent=[0,nx,0,ny], norm=norm_bot)
# Shared colorbars (one per row)
cax_top = fig.add_axes([0.92, 0.56, 0.015, 0.34])
cb_top = fig.colorbar(im_top_L, cax=cax_top); cb_top.set_label("divu (s$^{-1}$)")
cax_bot = fig.add_axes([0.92, 0.10, 0.015, 0.34])
cb_bot = fig.colorbar(im_bot_L, cax=cax_bot); cb_bot.set_label("sigP (Pa)" if use_sigP else "shear (s$^{-1}$)")
# --- Initialize quivers ONCE (k=0), then update with set_UVC ---
k0 = 0
# Free-slip top/bottom use same quivers per run; duplicate per row so both rows show arrows
UeF = Emask(uE_F, emaskF, k0);   VnF = Nmask(vN_F, nmaskF, k0)
UeN = Emask(uE_N, emaskN, k0);   VnN = Nmask(vN_N, nmaskN, k0)
# Quivers for each panel
q_tL_uE = ax[0].quiver(xE, yE, *horiz(UeF), scale=1/quiv_scale, angles='xy', units='xy', width=0.003)
q_tL_vN = ax[0].quiver(xN, yN, *vert(VnF), scale=1/quiv_scale, angles='xy', units='xy', width=0.003)
q_tR_uE = ax[1].quiver(xE, yE, *horiz(UeN), scale=1/quiv_scale, angles='xy', units='xy', width=0.003)
q_tR_vN = ax[1].quiver(xN, yN, *vert(VnN), scale=1/quiv_scale, angles='xy', units='xy', width=0.003)

q_bL_uE = ax[2].quiver(xE, yE, *horiz(UeF), scale=1/quiv_scale, angles='xy', units='xy', width=0.003)
q_bL_vN = ax[2].quiver(xN, yN, *vert(VnF), scale=1/quiv_scale, angles='xy', units='xy', width=0.003)
q_bR_uE = ax[3].quiver(xE, yE, *horiz(UeN), scale=1/quiv_scale, angles='xy', units='xy', width=0.003)
q_bR_vN = ax[3].quiver(xN, yN, *vert(VnN), scale=1/quiv_scale, angles='xy', units='xy', width=0.003)
# Stable quiver-key: reference a dummy quiver you NEVER remove or update
qkey_ref = ax[0].quiver([], [], [], [], scale=1/quiv_scale, angles='xy', units='xy', width=0.003)
ax[0].quiverkey(qkey_ref, X=0.84, Y=1.03, U=0.05, label="0.05 m s$^{-1}$", labelpos="E", coordinates="axes")

### animate

In [ ]:
writer = FFMpegWriter(fps=6, metadata=dict(artist="CICE box test"))
out = Path("/home/581/da1339/graphical/free-slip/bc_panel_native_inst.mp4")
times = dsF["time"].values
with writer.saving(fig, str(out), dpi=140):
    for k in range(len(times)):
        im_top_L.set_data(Tmask(divF,  tmaskF, k));  im_top_R.set_data(Tmask(divN,  tmaskN, k))
        botF = Tmask(sigPF if use_sigP else sheF, tmaskF, k)
        botN = Tmask(sigPN if use_sigP else sheN, tmaskN, k)
        im_bot_L.set_data(botF)
        im_bot_R.set_data(botN)
        UeF = Emask(uE_F, emaskF, k)
        VnF = Nmask(vN_F, nmaskF, k)
        UeN = Emask(uE_N, emaskN, k)
        VnN = Nmask(vN_N, nmaskN, k)
        q_tL_uE.set_UVC(*horiz(UeF))
        q_tL_vN.set_UVC(*vert(VnF))
        q_tR_uE.set_UVC(*horiz(UeN))
        q_tR_vN.set_UVC(*vert(VnN))
        q_bL_uE.set_UVC(*horiz(UeF))
        q_bL_vN.set_UVC(*vert(VnF))
        q_bR_uE.set_UVC(*horiz(UeN))
        q_bR_vN.set_UVC(*vert(VnN))
        fig.suptitle(fmt_time(dsF["time"], k), y=0.98)
        writer.grab_frame()
print(f"Wrote {out.resolve()}")

## plot results from uniform experiments

In [ ]:
def plot_two_vars(da1, da2, tit_fig=None, tit1_str="", tit2_str="", cbar_lab="", P_save=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True, constrained_layout=True)
    vmin = np.nanmin([da1.min().item(), da2.min().item()])
    vmax = np.nanmax([da1.max().item(), da2.max().item()])
    pcm1 = ax1.pcolormesh(da1, vmin=vmin, vmax=vmax, shading='nearest')
    pcm2 = ax2.pcolormesh(da2, vmin=vmin, vmax=vmax, shading='nearest')
    ax1.set_title(tit1_str or "")
    ax2.set_title(tit2_str or "")
    for ax in (ax1, ax2):
        ax.set_aspect('equal', adjustable='box')
    add_cbar_same_height(ax2, pcm2, cbar_lab)
    if tit_fig:
        fig.suptitle(tit_fig, fontsize=14)
    if P_save:
        fig.savefig(P_save, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def plot_diff(da, tit_str="", cbar_lab="", P_save=None):
    fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
    vals = np.asarray(da.values, dtype=float)
    finite = np.isfinite(vals)
    vmax = np.nanmax(np.abs(vals[finite])) if finite.any() else 0.0
    if vmax <= 0 or not np.isfinite(vmax):
        # all-zero or all-NaN: use a tiny symmetric range, no TwoSlopeNorm needed
        eps = 1e-12
        pcm = ax.pcolormesh(da, shading='nearest', cmap='coolwarm',
                            vmin=-eps, vmax=eps)
    else:
        norm = TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)
        pcm = ax.pcolormesh(da, shading='nearest', cmap='coolwarm', norm=norm)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(tit_str)
    add_cbar_same_height(ax, pcm, cbar_lab)
    if P_save:
        fig.savefig(P_save, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def animate_two_vars(dir1, dir2, pattern, var, N_1, N_2, out_path, unit="m/s", fps=2):
    files1 = sorted(glob.glob(os.path.join(dir1, pattern)))
    files2 = sorted(glob.glob(os.path.join(dir2, pattern)))
    assert files1 and files2 and len(files1) == len(files2), "Mismatched file lists."
    # Load time stacks
    ds1 = xr.open_mfdataset(files1, combine="nested", concat_dim="time")
    ds2 = xr.open_mfdataset(files2, combine="nested", concat_dim="time")
    da1 = ds1[var]
    da2 = ds2[var]
    T = da1.sizes["time"]
    assert T == da2.sizes["time"], "Mismatched time lengths."
    # Fixed scale across both runs & all times
    vmin = float(np.nanmin([da1.min(), da2.min()]))
    vmax = float(np.nanmax([da1.max(), da2.max()]))
    # Build figure for frame 0
    fig, ax1, ax2, pcm1, pcm2 = plot_two_vars(da1.isel(time=0), da2.isel(time=0),
                                              tit_fig=var, tit1_str=N_1, tit2_str=N_2, cbar_lab=unit)
    # time label in the suptitle (append after var name)
    def timestr(i):
        return np.datetime_as_string(da1["time"].values[i], unit="s") if "time" in da1.coords else f"t={i}"
    fig.suptitle(f"{var} | {timestr(0)}", fontsize=14)
    # For pcolormesh+shading='nearest', set_array wants a flat M*N array
    def update(i):
        pcm1.set_array(np.ravel(da1.isel(time=i).values))
        pcm2.set_array(np.ravel(da2.isel(time=i).values))
        fig.suptitle(f"{var} | {timestr(i)}", fontsize=14)
        return pcm1, pcm2
    ani = FuncAnimation(fig, update, frames=T, interval=int(1000/fps), blit=False)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    # Save: prefer mp4, fallback to gif if ffmpeg unavailable
    try:
        from matplotlib.animation import FFMpegWriter
        ani.save(out_path.with_suffix(".mp4"), writer=FFMpegWriter(fps=fps), dpi=200)
        saved = out_path.with_suffix(".mp4")
    except Exception:
        from matplotlib.animation import PillowWriter
        ani.save(out_path.with_suffix(".gif"), writer=PillowWriter(fps=fps))
        saved = out_path.with_suffix(".gif")
    plt.close(fig)
    ds1.close(); ds2.close()
    return saved

def animate_all_vars(dir1, dir2, pattern, var_list, N_1, N_2, out_dir, unit="m/s", fps=2):
    out_dir = Path(out_dir)
    results = {}
    for var in var_list:
        try:
            out_path = out_dir / f"{var}_{N_1}_vs_{N_2}"
            saved = animate_two_vars(dir1, dir2, pattern, var, N_1, N_2, out_path, unit=unit, fps=fps)
            print(f"Saved: {saved}")
            results[var] = str(saved)
        except Exception as e:
            print(f"[WARN] {var}: {e}")
    return results
    
def add_cbar_same_height(ax, mappable, label=""):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3.5%", pad=0.05)  # height matches ax
    cb = ax.figure.colorbar(mappable, cax=cax)
    if label:
        cb.set_label(label)
    # Optional: remove the “1e-12” style offset on the bar
    cb.formatter = ScalarFormatter(useMathText=True)
    cb.formatter.set_powerlimits((-3, 3))
    cb.update_ticks()
    return cb

def append_diff_summary(summary_path, var, N_1, N_2, diff):
    summary_path = Path(summary_path)
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    # Stats line
    stats = (f"{var} diff:\n({N_1}) - ({N_2}) | "
             f"min={diff.min().item():.3e}, "
             f"max={diff.max().item():.3e}, "
             f"mean={diff.mean().item():.3e}")
    # Full array as text (scientific notation; don’t truncate; wider lines)
    arr_txt = aligned_array_str(diff.values, prec=3)
    block = f"{stats}\n{arr_txt}\n\n"
    # Print to console
    # print(block, end="")
    # Append to file
    with summary_path.open("a", encoding="utf-8") as f:
        f.write(block)

def aligned_array_str(a, prec=3):
    a = np.asarray(a, dtype=float)
    w = len(f"{0.0: .{prec}e}")          # width of e-format like " 0.000e+00"
    def fmt(x):
        return f"{'nan':>{w}}" if np.isnan(x) else f"{x:>{w}.{prec}e}"
    return np.array2string(a,
                           formatter={"float_kind": fmt},
                           threshold=a.size,                # don't truncate
                           max_line_width=10_000,
                           separator=" ")

In [ ]:
D_results = "/g/data/gv90/da1339/cice-dirs/runs/RESULTS"
D_graph   = "/g/data/gv90/da1339/GRAPHICAL/free-slip"
N_1       = 'free-slip_uniform-east_noCDP'
N_2       = 'free-slip_uniform-east_CDP'
P_1       = f"{D_results}/{N_1}/history"
P_2       = f"{D_results}/{N_2}/history"
F_inst    = "iceh.2005-01-04.nc"
F_inst_pat= "iceh.2005-01-*.nc"
ds1       = xr.open_dataset(f"{P_1}/{F_inst}")
ds2       = xr.open_dataset(f"{P_2}/{F_inst}")
F_base    = Path(F_inst).stem
D_comp    = Path(f"{D_graph}/{N_1}_{N_2}_{F_base}")
D_comp.mkdir(parents=True, exist_ok=True)
var_list  = ['aice', 'hi', 'uvelE', 'vvelN', 'KuxN', 'KuyE']#'uvelE_1','vvelN_1','uvelN_1','vvelE_1',]
cbar_list = ['%'   , 'm' , 'm/s'  , 'm/s'  , ''    , '']
for i,var in enumerate(var_list):
    P_vars = f"{D_comp}/{var}.png"
    P_diff = f"{D_comp}/{var}_diff.png"
    if var not in ds1 or var not in ds2:
        print(f"Skipping {var} (missing in one dataset)")
        continue
    da1 = ds1[var].isel(time=0)
    da2 = ds2[var].isel(time=0)
    plot_two_vars(da1, da2, tit_fig=var, tit1_str=N_1, tit2_str=N_2, cbar_lab=cbar_list[i], P_save=P_vars)
    diff = da1 - da2
    diff_str = f"{var} diff:\n({N_1}) - ({N_2})"
    P_sum = Path(D_comp) / f"{var}_diff_summary.txt"
    append_diff_summary(P_sum, var, N_1, N_2, diff)
    plot_diff(diff, tit_str=diff_str, cbar_lab=cbar_list[i], P_save=P_diff)

# 2. Form-factor creation from coastline and grounded icebergs

The static form factor is the geometric part of the parameterisation. It identifies where unresolved lateral contact or anchoring is possible. In this implementation, the form factor differs from the simpler coastline-only approach because it combines:

1. high-resolution coastline geometry;
2. grounded-iceberg locations and/or geometric properties;
3. CICE C-grid E/N face placement;
4. optional combination of coastline-derived and grounded-iceberg-derived factors.

The current `shuga.grid.lateral_drag.FormFactors` class builds coastline form factors using nearest coastline segment geometry and a distance taper, builds grounded-iceberg form factors from grounded iceberg inputs, and combines the two using `max`, `mean`, or `sum`-and-clip logic. The published figure for this section should show the coastline, grounded-iceberg, and combined form-factor fields in a Liu-style Antarctic projection.

## optional build form factors through shuga: only perform if absolutely necessary

In [ ]:
run   = RunSpec(sim_name   = "LD-static-Cs1e-3",
                start_date = "1993-01-01",
                end_date   = "1999-12-31",
                hemisphere = HEMISPHERE,
                project    = PROJECT,
                user       = USER)
cls   = ClassificationSpec(ice_type     = ICE_TYPE,
                           grid_type    = GRID_TYPE,
                           ispd_thresh  = ISPD_THRESH,
                           methods      = tuple('binary-days'),
                           bin_window   = BIN_WINDOW,
                           bin_min_days = BIN_MIN_DAYS,
                           roll_window  = ROLL_WINDOW)
paths = ShugaPaths(run=run, classify=cls)
ff    = FormFactors(paths=paths)
# Uncomment to rebuild:
# ds_coast    = ff.build_F2_from_high_res_coastline(overwrite=False)
# ds_gi       = ff.build_F2_from_grounded_iceberg_dataframe(overwrite=False)
# ds_combined = ff.build_F2_combined(overwrite=False)
print(ff)


## otherwise just plot form factors as they are presently written on disk 

In [ ]:
def load_form_factor_dataset(path: Path = FORM_FACTOR_FILE) -> xr.Dataset:
    if not Path(path).exists():
        raise FileNotFoundError(path)
    return xr.open_dataset(path)

def form_factor_magnitude(ds: xr.Dataset) -> xr.DataArray:
    candidates_x = ["F2x", "F2E", "F2N_x", "F2_x"]
    candidates_y = ["F2y", "F2N", "F2E_y", "F2_y"]
    x = first_present(ds, candidates_x)
    y = first_present(ds, candidates_y)
    if x and y:
        return xr.apply_ufunc(np.hypot, ds[x], ds[y], dask="allowed").rename("F2mag")
    if "F2x" in ds:
        return abs(ds["F2x"]).rename("F2mag")
    raise KeyError(f"Could not infer F2 fields from {list(ds.data_vars)}")

def plot_form_factor_pygmt(ds: xr.Dataset, out: Path | None = None, title: str = "Combined lateral-drag form factor"):
    import pygmt
    f2 = form_factor_magnitude(ds)
    lon_name = first_present(ds, ["lon", "TLON", "ULON", "NLON"])
    lat_name = first_present(ds, ["lat", "TLAT", "ULAT", "NLAT"])
    if lon_name is None or lat_name is None:
        raise KeyError("Need lon/lat variables in form-factor dataset.")
    lon = ds[lon_name].values.ravel()
    lat = ds[lat_name].values.ravel()
    val = f2.values.ravel()
    keep = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(val) & (lat < -45) & (val > 0)
    fig = pygmt.Figure()
    fig.basemap(region=[-180, 180, -90, -45], projection="S0/-90/14c", frame=["af", f"+t{title}"])
    pygmt.makecpt(cmap="viridis", series=[0, float(np.nanpercentile(val[keep], 99.5)) if keep.any() else 1, 0.05])
    fig.coast(shorelines="0.4p,black", land="lightgray", water="white")
    fig.plot(x=lon[keep], y=lat[keep], style="s0.035c", fill=val[keep], cmap=True, pen=None)
    fig.colorbar(frame='af+l"|F2|"')
    if out is not None:
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(out))
        print(f"Saved: {out}")
    fig.show()
    return fig

In [ ]:
ds_f2 = load_form_factor_dataset()
plot_form_factor_pygmt(ds_f2, NOTEBOOK_FIG_ROOT / "publication_form_factor_combined_pygmt.png" if SAVE_FIGS else None)

# 3. Form-function theory: static, quadratic, linear, and corrected `blend_strain`

The form factor tells CICE **where** lateral drag can act. The form function tells CICE **how strongly** that drag acts as a function of ice state.

The corrected `blend_strain` implementation is intended to be dynamically selective:

$$
\phi_\mathrm{static} = \frac{C_s}{|\mathbf{u}|+u_0}, \quad
\phi_\mathrm{quad} = C_q |\mathbf{u}|
$$

$$
w_\epsilon = \frac{1}{1+(\epsilon_\mathrm{eff}/\epsilon_\mathrm{blend})^p}, \quad
w_u = \frac{1}{1+(|\mathbf{u}|/u_\mathrm{blend})^p}, \quad
w_\mathrm{lock}=w_\epsilon w_u
$$

$$
\phi_\mathrm{blend}=w_\mathrm{lock}\phi_\mathrm{static}+(1-w_\mathrm{lock})\phi_\mathrm{quad}.
$$

The `static` branch is accessed only when the ice is both slow and *low-strain*. The analytical 1-D plots below project the two-dimensional *speed-strain* rule onto speed using $\epsilon_\mathrm{eff}=u/L_\mathrm{eff}$, where $L_\mathrm{eff}$ represents a typical fast-ice cell scale near the Antarctic coastline in $1/4^{\circ}$-degree global model that we are using in CICE.

## form functions

In [ ]:
def cice_phi_static(u, Cs, u0):
    return Cs / (u + u0)

def cice_phi_quad(u, Cq):
    return Cq * u

def cice_phi_linear(u, CL):
    return CL + np.zeros_like(u)

def lock_weight_eps(eps_eff, eps_blend, p):
    r = np.maximum(eps_eff, 0.0) / max(eps_blend, 1.0e-20)
    return 1.0 / (1.0 + r**p)

def lock_weight_speed(u, u_blend, p):
    r = np.maximum(u, 0.0) / max(u_blend, 1.0e-20)
    return 1.0 / (1.0 + r**p)

def tau_blend_strain_1d(u, *, M_F2, Cs, Cq, u0, u_blend, eps_blend, blend_exp, L_eff):
    eps_eff    = u / L_eff
    w_eps      = lock_weight_eps(eps_eff, eps_blend, blend_exp)
    w_spd      = lock_weight_speed(u, u_blend, blend_exp)
    w_lock     = w_eps * w_spd
    phi_static = cice_phi_static(u, Cs, u0)
    phi_quad   = cice_phi_quad(u, Cq)
    phi        = w_lock * phi_static + (1.0 - w_lock) * phi_quad
    tau        = M_F2 * u * phi
    return xr.Dataset(data_vars = {"tau"    : ("u", tau),
                                   "w_lock" : ("u", w_lock),
                                   "w_eps"  : ("u", w_eps),
                                   "w_spd"  : ("u", w_spd),
                                   "eps_eff": ("u", eps_eff),
                                   "phi"    : ("u", phi)},
                      coords = {"u": u})

## create two figures
1. showing the form-functions as a function of ice speed
2. `blend-strain` weight-gating

In [ ]:
def plot_form_function_single(blend_cfg: Mapping[str, float], out: Path | None = None):
    u          = np.linspace(0.0, 1.1e-3, 900)
    M_F2       = blend_cfg.get("M_F2", 910.0)
    L_eff      = blend_cfg.get("L_eff", 12.0e3)
    static_cfg = dict(Cs=1.0e-3, Cq=700.0, CL=0.25, u0=5.0e-5)
    tau_static = M_F2 * u * cice_phi_static(u, static_cfg["Cs"], static_cfg["u0"])
    tau_quad   = M_F2 * u * cice_phi_quad(u, static_cfg["Cq"])
    tau_linear = M_F2 * u * cice_phi_linear(u, static_cfg["CL"])
    dsb        = tau_blend_strain_1d(u,
                                     M_F2      = M_F2,
                                     Cs        = blend_cfg["Cs"],
                                     Cq        = blend_cfg["Cq"],
                                     u0        = blend_cfg["u0"],
                                     u_blend   = blend_cfg["u_blend"],
                                     eps_blend = blend_cfg["eps_blend"],
                                     blend_exp = blend_cfg["blend_exp"],
                                     L_eff     = L_eff)
    fig, ax    = plt.subplots(figsize=(11, 6))
    ax.plot(u, tau_static, label="static", lw=2)
    ax.plot(u, tau_quad, label="quadratic", lw=2)
    ax.plot(u, tau_linear, label="linear", lw=2)
    ax.plot(u, dsb["tau"], label=("blend_strain"))
                                  #+ rf"($u_b={blend_cfg['u_blend']:.1e}$, "
                                  #+ rf"$\epsilon_b={blend_cfg['eps_blend']:.1e}$, "
                                  #+ rf"$p={blend_cfg['blend_exp']:g}$)"), lw=3)
    ax.axvspan(static_cfg["u0"], 1.0e-3, alpha=0.12, label="fast-ice speed band")
    ax.axvline(static_cfg["u0"], ls="--", lw=1, label="$u_0$")
    ax.axvline(blend_cfg["u_blend"], ls=":", lw=1.5, label="$u_{blend}$")
    u_eps = blend_cfg["eps_blend"] * L_eff
    if u.min() <= u_eps <= u.max():
        ax.axvline(u_eps, ls="-.", lw=1.5, label=r"$\epsilon_{eff}(u)=\epsilon_{blend}$")
    ax.set_xlim(0, 1.1e-3)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Sea-ice speed $|u|$ (m s$^{-1}$)")
    ax.set_ylabel(r"Stress scale $K_u |u|\phi$ (Pa for $K_u=910$ kg m$^{-2}$)")
    ax.set_title(rf"Analytical lateral-drag stress scale; $\epsilon_{{eff}}=u/L$, $L={L_eff/1000:g}$ km")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, ncol=2, fontsize=9)
    fig.tight_layout()
    if out is not None:
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print(f"Saved: {out}")
    plt.show()
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.plot(u, dsb["w_lock"], lw=3, label=r"$w_{lock}=w_\epsilon w_u$")
    ax2.plot(u, dsb["w_eps"], lw=2, ls="--", label=r"$w_\epsilon$")
    ax2.plot(u, dsb["w_spd"], lw=2, ls=":", label=r"$w_u$")
    ax2.axvline(blend_cfg["u_blend"], ls=":", lw=1.5, label="$u_{blend}$")
    if u.min() <= u_eps <= u.max():
        ax2.axvline(u_eps, ls="-.", lw=1.5, label=r"$\epsilon_{eff}(u)=\epsilon_{blend}$")
    ax2.set_xlim(0, 1.1e-3)
    ax2.set_ylim(-0.02, 1.02)
    ax2.set_xlabel("Sea-ice speed $|u|$ (m s$^{-1}$)")
    ax2.set_ylabel("Static/locking branch weight")
    ax2.grid(alpha=0.25)
    ax2.legend(frameon=False, ncol=2, fontsize=12)
    fig2.tight_layout()
    if out is not None:
        out2 = out.with_name(out.stem + "_weights" + out.suffix)
        fig2.savefig(out2, dpi=300, bbox_inches="tight")
        print(f"Saved: {out2}")
    plt.show()
    return dsb

In [ ]:
blend_base_cfg = dict(Cs=5.0e-4, Cq=350.0, u0=5.0e-5, u_blend=5.0e-4, eps_blend=5.0e-8, blend_exp=10.0, L_eff=12.0e3)
dsb = plot_form_function_single(blend_base_cfg, NOTEBOOK_FIG_ROOT / "publication_form_function_blend_base.png" if SAVE_FIGS else None)

In [ ]:
OUTDIR = NOTEBOOK_FIG_ROOT
OUTDIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------
Cs = 2.5e-4          # static coefficient
Cq = 300.0           # quadratic coefficient
C_L = 0.25           # linear / Rayleigh-like coefficient

u0 = 5.0e-5          # m s-1; static regularisation speed
u_blend = 5.0e-4     # m s-1
eps_blend = 5.0e-8   # s-1
blend_exp = 3.0

u = np.logspace(-6, -2, 800)

# ---------------------------------------------------------------------
# Base form functions
# ---------------------------------------------------------------------
phi_static = Cs / (u + u0)
phi_quad = Cq * u
phi_linear = np.full_like(u, C_L)

def w_speed(u, u_blend=u_blend, p=blend_exp):
    return 1.0 / (1.0 + (u / u_blend)**p)

def w_strain(eps, eps_blend=eps_blend, p=blend_exp):
    return 1.0 / (1.0 + (eps / eps_blend)**p)

def phi_blend_for_fixed_eps(eps):
    wu = w_speed(u)
    weps = w_strain(eps)
    wlock = wu * weps
    return wlock * phi_static + (1.0 - wlock) * phi_quad

eps_values = [
    1.0e-9,
    5.0e-8,
    5.0e-7,
    5.0e-6,
]

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)

ax.plot(u, phi_static, lw=2.0, label=r"$\phi_{static}=C_s/(|u|+u_0)$")
ax.plot(u, phi_quad, lw=2.0, label=r"$\phi_{quad}=C_q|u|$")
ax.plot(u, phi_linear, lw=2.0, label=r"$\phi_{linear}=C_L$")

for eps in eps_values:
    ax.plot(
        u,
        phi_blend_for_fixed_eps(eps),
        lw=2.0,
        ls="--",
        label=rf"blend, $\epsilon={eps:.0e}$ s$^{{-1}}$",
    )

ax.axvline(u_blend, color="0.35", lw=1.0)
ax.text(
    u_blend * 1.05,
    0.08,
    r"$u_{blend}$",
    rotation=90,
    va="bottom",
    ha="left",
    fontsize=9,
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_title("Analytical form-function behaviour (LD-blend-base parameters)")
ax.set_xlabel(r"ice speed $|u|$ (m s$^{-1}$)")
ax.set_ylabel(r"form function $\phi$ (s$^{-1}$)")

ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="lower right", fontsize=9, ncol=2)

fig.savefig(
    OUTDIR / "blend_form_function_curves.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 3.1 Phase-space view of the corrected `blend_strain` gate

The 1-D plot is useful for the paper, but the implementation is genuinely two-dimensional in speed--strain space. This plot shows the static/locking branch weight over the \((|u|,\epsilon_\mathrm{eff})\) plane.

In [ ]:
def plot_blend_weight_phase_space(blend_cfg: Mapping[str, float], out: Path | None = None):
    u = np.linspace(0.0, 1.2e-3, 300)
    eps = np.logspace(-10, -5, 300)
    U, E = np.meshgrid(u, eps)
    p = blend_cfg["blend_exp"]
    W = lock_weight_speed(U, blend_cfg["u_blend"], p) * lock_weight_eps(E, blend_cfg["eps_blend"], p)
    fig, ax = plt.subplots(figsize=(9, 6))
    pcm = ax.pcolormesh(U, E, W, shading="auto", vmin=0, vmax=1)
    ax.set_yscale("log")
    ax.axvline(blend_cfg["u_blend"], ls=":", lw=1.5)
    ax.axhline(blend_cfg["eps_blend"], ls="--", lw=1.5)
    ax.set_xlabel("Sea-ice speed $|u|$ (m s$^{-1}$)")
    ax.set_ylabel(r"Effective strain rate $\epsilon_{eff}$ (s$^{-1}$)")
    ax.set_title("Corrected blend_strain static/locking branch weight")
    cb = fig.colorbar(pcm, ax=ax)
    cb.set_label(r"$w_{lock}$")
    fig.tight_layout()
    if out is not None:
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print(f"Saved: {out}")
    plt.show()
    return fig

plot_blend_weight_phase_space(
    blend_base_cfg,
    NOTEBOOK_FIG_ROOT / "publication_blend_weight_phase_space.png" if SAVE_FIGS else None,
)


In [ ]:
OUTDIR = NOTEBOOK_FIG_ROOT
OUTDIR.mkdir(exist_ok=True)
# ---------------------------------------------------------------------
# LD-blend-base-style parameters used in the analytical schematic
# ---------------------------------------------------------------------
u_blend = 5.0e-4          # m s-1
eps_blend = 5.0e-8        # s-1
blend_exp = 3.0           # transition sharpness

# Analytical speed/strain space
u = np.logspace(-6, -2, 500)          # m s-1
eps_eff = np.logspace(-10, -4, 500)   # s-1

U, EPS = np.meshgrid(u, eps_eff)

# Blend gates
w_u = 1.0 / (1.0 + (U / u_blend)**blend_exp)
w_eps = 1.0 / (1.0 + (EPS / eps_blend)**blend_exp)

# Product gate
w_lock = w_u * w_eps

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 5.8), constrained_layout=True)

pcm = ax.pcolormesh(
    U,
    EPS,
    w_lock,
    shading="auto",
    vmin=0.0,
    vmax=1.0,
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.axvline(u_blend, linestyle="--", linewidth=1.0, color="k")
ax.axhline(eps_blend, linestyle="--", linewidth=1.0, color="k")

ax.text(
    1.4e-6,
    eps_blend * 1.2,
    r"$\epsilon_{blend}$",
    fontsize=9,
    ha="left",
    va="bottom",
)

ax.text(
    u_blend * 1.08,
    1.4e-10,
    r"$u_{blend}$",
    fontsize=9,
    ha="left",
    va="bottom",
    rotation=90,
)

ax.set_title("Static/locking branch weight in speed-strain space")
ax.set_xlabel(r"ice speed $|u|$ (m s$^{-1}$)")
ax.set_ylabel(r"effective strain rate $\epsilon_{eff}$ (s$^{-1}$)")

cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(r"$w_{lock} = w_\epsilon w_u$")

fig.savefig(
    OUTDIR / "publication_blend_weight_phase_space_v2.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# 4. Corrected `blend_strain` intercomparison, 1994-10-01 to 1994-12-15

This is the main experiment section for the corrected `blend_strain` formulation. It compares the base, smooth, sharp, speed-strict, strain-strict, and strain-permissive settings using:

- binary-days FIA and FIP;
- spatial skill against AF2020 where available;
- realised speed, strain, locking weight, and lateral-drag diagnostics;
- phase-space diagnostics to test whether the scheme is doing what it claims physically.

The publication-level question is not simply which run has the lowest FIA error. The stronger question is whether the selected run places static-like locking in the physically expected part of phase space: low speed, low strain, and form-factor-active coastal/GI cells.

In [ ]:
BLEND_CORE_VARS = ["aice", "hi", "tarea", "TLON", "TLAT", "uvel", "vvel", "uocn", "vocn",
                   "KuxE", "KuyE", "KuxN", "KuyN", "F2E", "F2N",
                   "ldphiE", "ldphiN", "ldwgtE", "ldwgtN",
                   "ldepsE", "ldepsN", "ldspdE", "ldspdN",
                   "ldpstatE", "ldpstatN", "ldpquadE", "ldpquadN", "ldplinE", "ldplinN"]

## 3-month circum-Antarctic FIA comparison

In [ ]:
def blend_daily_fia_table(sims: Sequence[str] = BLEND_SIMS) -> pd.DataFrame:
    rows = []
    for sim in sims:
        print(f"Computing FIA for {sim}")
        fia = daily_fia_from_mask(sim, ANALYSIS_START, ANALYSIS_END)
        df  = safe_to_series(fia, "FIA_10^3_km2")
        df["sim_name"] = sim
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

In [ ]:
# run = RunSpec(sim_name   = "LD-blend-CsCq-small",
#                    start_date = "1994-09-01",
#                    end_date   = "1994-12-31",
#                    hemisphere = HEMISPHERE,
#                    project    = PROJECT,
#                    user       = USER)
# cls  = ClassificationSpec(ice_type=ICE_TYPE,
#                             grid_type=GRID_TYPE,
#                             ispd_thresh=ISPD_THRESH,
#                             methods='binary-days',
#                             bin_window=BIN_WINDOW,
#                             bin_min_days=BIN_MIN_DAYS,
#                             roll_window=ROLL_WINDOW,)
# ds   =  load_cice(run=run, classify=cls, variables=BLEND_CORE_VARS, hemisphere=HEMISPHERE)
# print(ds)
BLEND_FIA = blend_daily_fia_table(sims=CORE_SIMS)#sims=["LD-blend-base","LD-blend-CsCq-small","LD-blend-ktens-0p1"])


In [ ]:
if len(BLEND_FIA):
    plot_timeseries_df(BLEND_FIA, "time", "FIA_10^3_km2", "sim_name",
                       title=f"Corrected blend_strain binary-days FIA: {ANALYSIS_START} to {ANALYSIS_END}",
                       ylabel="FIA (10$^3$ km$^2$)",
                       out=NOTEBOOK_FIG_ROOT / f"blend_fia_{ANALYSIS_START}_{ANALYSIS_END}.png")

## diagnostic summary table

In [ ]:
def blend_diagnostic_summary(sim_name: str) -> dict:
    ds      = load_pub_cice(sim_name, ANALYSIS_START, ANALYSIS_END, variables=BLEND_CORE_VARS)
    cls     = load_pub_classified(sim_name, ANALYSIS_START, ANALYSIS_END, variables=["FI_mask"])
    ds, cls = robust_align_time(ds, cls)
    diag    = xr.merge([compute_diagnostic_terms(ds), add_blend_diagnostics(ds)], compat="override")
    area    = area_da(ds)
    fi      = cls["FI_mask"].astype(bool)
    pack    = (~fi) & np.isfinite(fi)
    row     = {"sim_name": sim_name, "start": ANALYSIS_START, "end": ANALYSIS_END}
    for zone_name, mask in {"FI_binary_days": fi, "pack_or_mobile": pack}.items():
        for var in ["ice_speed", "ldspd", "strain_invariant", "ldeps", "ldwgt", "ldphi", "tau_ld_mag", "P_ld"]:
            if var in diag:
                row[f"{zone_name}__{var}__mean"] = float(area_weighted_mean(diag[var], area, mask).mean("time").compute())
                try:
                    row[f"{zone_name}__{var}__p95"] = float(diag[var].where(mask).quantile(0.95).compute())
                except Exception:
                    pass
    fia                      = area_sum(fi, area) / 1e9
    row["FIA_mean_10^3_km2"] = float(fia.mean("time").compute())
    row["FIA_max_10^3_km2"]  = float(fia.max("time").compute())
    row["FIA_min_10^3_km2"]  = float(fia.min("time").compute())
    return row

In [ ]:
BLEND_SUMMARY_ROWS = []
for sim in BLEND_SIMS:
    try:
        BLEND_SUMMARY_ROWS.append(blend_diagnostic_summary(sim))
    except Exception as exc:
        print(f"Skipping blend summary for {sim}: {exc}")
BLEND_SUMMARY = pd.DataFrame(BLEND_SUMMARY_ROWS)
display_df(BLEND_SUMMARY, n=20)
BLEND_SUMMARY.to_csv(NOTEBOOK_FIG_ROOT / f"blend_diagnostic_summary_{ANALYSIS_START}_{ANALYSIS_END}.csv", index=False)

## blend-strain diagnostic performance check

### Table interpretation

This table reports summaries for the two sea-ice state variables used for context (`aice`, `hi`) and the lateral-drag diagnostics written on C-grid faces (`E` and `N`). The `E` diagnostics correspond to the C-grid face used by the `stepu_C()` momentum update, while the `N` diagnostics correspond to the face used by `stepv_C()`.

The table columns are:

- `q50`           : **median**.
- `q75`           : 75th percentile.
- `q95`           : 95th percentile.
- `q100`          : maximum value.
- `nonzero_count` : number of array elements satisfying `da != 0`.
- `finite_count`  : number of finite array elements.

**Important Note**, these diagnostics are reported over the full sea ice domain (i.e. they have **not** been classified).

---

### Sea-ice state variables

#### `aice`

`aice` is the sea-ice concentration. It is dimensionless and should lie between 0 and 1.

Interpretation:

- `q50 = 0` indicates that at least half of all sampled grid-cell/time records are ice-free or inactive in this full-domain diagnostic sample. 
- `q75` $\le 0.4$ would indicate that there are some cells in the sea-ice coverage that are greater than 15% (the classification threshold) but less than 1 which is a physical reality at $1/4^{\circ}$ grid, and still points to the upper quarter of the sampled domain/time field contains partial-to-substantial ice cover.
- `q95` $\ge 0.95$ and `q100` $= 1$ would indicate most cells are fully ice-covered.

We should expect to see near-identical `aice` quantiles across experiments which would indicate that these `blend-strain` parameter changes are not radically changing the bulk concentration distribution over this analysis window.

#### `hi`

`hi` is the reported ice thickness field in metres. In this full-domain summary, zeros include ice-free cells, so the distribution should not be interpreted as a conditional thickness distribution over ice-covered cells unless the data are masked by `aice > threshold`.

Interpretation:

- `q50 = 0` is expected for full ocean/sea ice domain.
- `q75` $\le 0.2$ m would indicate that the upper quartile includes relatively thin grid-cell (mean) ice.
- `q95` $\ge 1.5$ m would indicate that the sea-ice thickness is *behaving* reasonably well.
- `q100` $\ge 20$ m would indicate that there is potential issue either thermodynamically or mechanically within the sea-ice domain. Given that these results are initialised from global sea that is already at 10 m in a few isolated grid cells we would *not* expect this to grow another 10 m on top of that. The thermodynamics and mechanical forces acting on the sea ice should overcome that vertical growth.

If `hi` has very similar values across experiments then this would suggest that `blend_strain` variants are modifying lateral-drag dynamics **without** strongly reorganising the bulk ice-thickness distribution over this diagnostic window (time period).

---

### Lateral-drag diagnostics

The lateral-drag diagnostics are face-centred:

- `*E` values are reported on C-grid `E` faces.
- `*N` values are reported on C-grid `N` faces.

Close agreement between the `E` and `N` distributions suggests that the diagnostic behaviour is broadly *isotropic* with respect to C-grid face orientation, rather than being dominated by a numerical asymmetry between the `u` and `v` momentum updates.

---

#### `ldspdE`, `ldspdN`

`ldspd` is the local ice-speed magnitude used by the lateral-drag form function, in `m s^-1`.

In the implementation, the relevant speed scale is effectively

$$
|\mathbf{u}| = \sqrt{u^2 + v^2}.
$$

Interpretation:

- `q50 = 0` expected with ocean/sea ice domain: most C-grid faces have zero diagnosed speed.
- `q75` $\le 5.0 \times 10^{-2} \mathrm{m} \mathrm{s}^{-1}$ corresponds to typical nonzero speeds on the order centimetres per second and that the mean sea ice speed is *behaving*.
- `q95` $\ge 10^{-1} \mathrm{m} \mathrm{s}^{-1}$ corresponds to tens of centimeters per second ice movement which indicates there are cells that are *fast-moving*.
- `q100` $> 5 \mathrm{m} \mathrm{s}^{-1}$ would indicate potentially something wrong with the either the surface current forcing or the CICE mechanical dyamics as this a 10 knot current which (without tides and even more so averaged over a day) should not be occuring on this horizontal scale.

The main interpretation between experiments should be watching what happens to `q75` and ensuring that no experiments accelerate sea ice to absurd values. 

#### `ldeps`

In `ice_dyn_shared.F90` `stepu_C()` subroutine:

```fortran
eps_eff = (deltaU(i,j) + deltaU(i,j-1)) / max(uarea(i,j) + uarea(i,j-1), 1.0e-20_dbl_kind)
```
and `stepv_C()` subroutine:
```fortran
eps_eff = (deltaU(i,j) + deltaU(i-1,j)) / max(uarea(i,j) + uarea(i-1,j), 1.0e-20_dbl_kind)
```

If `ldeps` genuinely has quantiles near $1$ to $10 s^{-1}$, then something is wrong in the CICE diagnostic itself or in how the variable is being loaded. For this problem, I would expect most meaningful coastal values to be more like $[1.0 \times 10^{-10}, \, 1.0 \times 10^{-5}] \text{ s}^{-1}$, depending on deformation state.

1. `ldeps` is nonzero but very small, and the table formatting is hiding it; to prevent this values are reported in scientific notation
2. `ldeps` is genuinely zero everywhere, meaning the diagnosed *strain-rate gate* is not currently active and `blend_strain` has effectively become a *speed-gated* blend.

If `ldeps` $== 0$, then: $\epsilon_{\mathrm{eff}} = 0$ and $w_\epsilon = 1$, therefore $w_\mathrm{lock} = w_u$

In effect `blend_strain` becomes *speed-gated* (static/quadratic) blend and not *speed-and-strain-gated* blend.

---

#### `ldwgtE`, `ldwgtN`

`ldwgt` is the realised *static/locking-branch* weight used by `blend_strain`. It is dimensionless and should lie between 0 and 1.

For `blend_strain`, the conceptual structure is:

$$
w_\epsilon = \frac{1}{1 + (\max(\epsilon_\mathrm{eff},0)/\epsilon_\mathrm{blend})^{p}},
$$

$$
w_u = \frac{1}{1 + (|\mathbf{u}|/u_\mathrm{blend})^{p}},
$$

$$
w_\mathrm{lock} = w_\epsilon w_u.
$$

The diagnostic `ldwgt` stores this realised *locking/static-branch* weight.

Interpretation:

- `ldwgt = 1` means the form function is fully on the *static/locking* branch.
- `ldwgt = 0` means it is fully on the *quadratic/mobile* branch.
- Intermediate values indicate a smooth transition between the two branches.

In the table:

- The median is zero for all three runs.
- `q75` and `q95` are very small.
- `q100 = 1` indicates that some cells/times do fully enter the locking/static branch.

The runs differ strongly in the upper-tail weight:

- `LD-blend-sharp`: `q95 ≈ 1.1–1.2 \times 10^{-6}`
- `LD-blend-base`: `q95 ≈ 1.8 \times 10^{-5}`
- `LD-blend-smooth`: `q95 ≈ 6.4–6.6 \times 10^{-4}`

Thus, among these three, `LD-blend-smooth` keeps the static/locking branch active over a larger part of the distribution, while `LD-blend-sharp` switches most aggressively toward the quadratic/mobile branch. The fact that all `q95` values remain small means that, over the full domain/time sample, the diagnosed lateral drag is overwhelmingly in or near the quadratic/mobile branch.

Given the suspicious `ldeps` behaviour, this `ldwgt` distribution is likely being controlled mainly by speed rather than by the intended strain-rate gate.

---

#### `ldphiE`, `ldphiN`

`ldphi` is the realised lateral-drag **damping rate**, in $\mathrm{s}^{-1}$.

For `blend_strain`, the realised damping rate is approximately

$$
\phi = w_\mathrm{lock}\phi_\mathrm{static} + (1 - w_\mathrm{lock})\phi_\mathrm{quad}
$$

where

$$
\phi_\mathrm{static} = \frac{C_s}{|\mathbf{u}| + u_0}
$$

and

$$
\phi_\mathrm{quad} = C_q |\mathbf{u}|
$$

The actual lateral-drag stress coefficient is then formed using the geometric form-factor field:

$$
C_l = K_u \phi
$$

Therefore, `ldphi` is **not itself the stress**, it is the realised damping rate that becomes dynamically important only after multiplication by the local lateral-drag factor `Ku`.

Interpretation:

- `q50 = 0` again reflects many inactive or zero-drag faces.
- `q75` gives the moderate-drag regime.
- `q95` gives the robust upper-tail damping regime.
- `q100` is an extreme maximum and should not be over-interpreted without mapping where it occurs.

In the table:

- `LD-blend-base` has `ldphi q95 ≈ 70 s^-1` and `q100 ≈ 700–720 s^-1`.
- `LD-blend-sharp` and `LD-blend-smooth` have `ldphi q95 ≈ 140 s^-1` and `q100 ≈ 1400–1440 s^-1`.

This is consistent with the upper tail being dominated by the quadratic branch:

$$
\phi_\mathrm{quad} = C_q |\mathbf{u}|
$$

For example, if `Cq = 700 m^-1` and `ldspd q95 ≈ 0.20 m s^-1`, then

$$
C_q |\mathbf{u}| \approx 700 \times 0.20 \approx 140 \ \mathrm{s}^{-1}.
$$

So the large `ldphi` values in the sharp/smooth cases are not necessarily pathological; they are consistent with a strong quadratic damping branch at high local speeds. What matters dynamically is where these large values occur and whether `Ku` is nonzero there.

---

### Overall interpretation of these three experiments

The table suggests the following:

1. The sea-ice state variables `aice` and `hi` are broadly similar across the three experiments. The tuning differences are not causing a dramatic full-domain redistribution of concentration or thickness over this diagnostic sample.

2. `ldspdE` and `ldspdN` are also very similar across experiments, especially in the high-speed tail. The sharp and smooth variants slightly reduce the 75th percentile speed relative to the base case, but the 95th percentile and maximum are almost unchanged.

3. `ldepsE` and `ldepsN` are the most concerning diagnostics. The upper quantiles and maximum are all zero, despite large `nonzero_count` values. This should be checked before making a strong physical claim about strain-rate-dependent behaviour.

4. Because the `blend_strain` gate uses `max(eps_eff, 0)`, zero or negative `ldeps` means the strain-rate component of the gate is effectively saturated in the low-strain/locking direction. In practice, the blend then behaves mainly as a speed-gated transition.

5. `ldwgt` confirms that the static/locking branch is active only over a small fraction of the full sampled distribution. `LD-blend-smooth` has the largest realised locking weight, `LD-blend-sharp` the smallest, and `LD-blend-base` sits between them.

6. `ldphi` indicates that the upper-tail damping is much stronger in `LD-blend-sharp` and `LD-blend-smooth` than in `LD-blend-base`. The magnitude is consistent with the quadratic branch, especially where `ldwgt` is close to zero and speeds are high.

7. The close agreement between `E` and `N` diagnostics is reassuring: the scheme does not appear to be producing a gross directional asymmetry between the two C-grid momentum components.

### Working conclusion

These diagnostics do **not yet demonstrate** that the intended strain-rate gate is operating as designed. They show that the lateral-drag diagnostics are being populated, that the speed-dependent and realised damping-rate terms are behaving coherently, and that the sharp/smooth tuning changes the realised locking weight. However, the `ldeps` distribution needs further checking.

At present, the safest interpretation is:

> The current `blend_strain` experiments appear to be behaving primarily as speed-gated blends between the static and quadratic branches, with little clear evidence from this table that the strain-rate gate is actively controlling the transition.

Before presenting this as a confirmed `blend_strain` result, inspect the signed and absolute distribution of `ldepsE/N`, especially `min`, `q01`, `q05`, `positive_count`, `negative_count`, and `abs_q95`.

#### table function

In [ ]:
def blend_diag_range_report(sim_name : str,
                            start    : str = ANALYSIS_START,
                            end      : str = ANALYSIS_END,
                            variables: list[str] | None = None):
    if variables is None:
        variables = ["aice", "hi", #"TLAT", "F2N", "F2E",
                     "ldspdN", "ldspdE", "ldepsN", "ldepsE",
                     "ldwgtN", "ldwgtE", "ldphiN", "ldphiE"]
    ds   = load_pub_cice(sim_name, start, end, variables=variables)
    rows = []
    for v in variables:
        if v not in ds:
            continue
        da   = ds[v]
        q    = da.where(np.isfinite(da)).quantile([0.5, 0.75, 0.95, 1.0], skipna=True).compute()
        vals = q.values
        rows.append({"variable"     : v,
                     "q50"          : f"{float(q.sel(quantile=0.5)):.4e}",
                     "q75"          : f"{float(q.sel(quantile=0.75)):.4e}",
                     "q95"          : f"{float(q.sel(quantile=0.95)):.4e}",
                     "q100"         : f"{float(q.sel(quantile=1.0)):.4e}",
                     "nonzero_count": int((da != 0).sum().compute()),
                     "finite_count" : int(np.isfinite(da).sum().compute())})
    out = pd.DataFrame(rows)
    P_CSV = NOTEBOOK_FIG_ROOT / f"blend_diagnostic_quantile_report_{sim_name}_{ANALYSIS_START}_{ANALYSIS_END}.csv"
    out.to_csv(P_CSV, index=False)
    print(f"CSV written to {P_CSV}")
    print(f"experiment: {sim_name}")
    display_df(out, n=100)

#### generate table for each experiment

In [ ]:
for sim in BLEND_SIMS:
    blend_diag_range_report(sim)

### `ldwgt` diagnosed blend-strain phase space: static/locking branch weight

This figure shows the diagnosed static/locking branch weight, $w_{\mathrm{lock}}$, from the corrected `blend_strain` lateral-drag implementation. The left and right panels show the N-face and E-face C-grid diagnostics separately.

The x-axis is the diagnosed speed used by the lateral-drag form function, $u = |\mathbf{u}|$, from `ldspdN` and `ldspdE`. The vertical dashed line marks the binary-days fast-ice speed threshold used in the classification workflow.

The y-axis is the diagnosed effective strain-rate scale, $\epsilon_{\mathrm{eff}}$, from `ldepsN` and `ldepsE`. This diagnostic is derived from the U-grid EVP deformation invariant after area normalisation, giving units of $s^{-1}$. The logarithmic y-axis is used because the relevant strain-rate values span several orders of magnitude.

The colour shows the mean static/locking branch weight, $\mathrm{w}_{\mathrm{lock}}$, within each *speed-strain* phase-space bin. In the corrected `blend_strain` formulation,

$$
\mathrm{w}_\epsilon = \frac{1}{1 + \left(\epsilon_{\mathrm{eff}} / \epsilon_{\mathrm{blend}}\right)^p},
$$

$$
\mathrm{w}_\mathrm{u} = \frac{1}{1 + \left(\mathrm{u} / \mathrm{u}_{\mathrm{blend}}\right)^p},
$$

$$
\mathrm{w}_{\mathrm{lock}} = \mathrm{w}_\epsilon \mathrm{w}_\mathrm{u}.
$$

The realised lateral-drag form function is then

$$
\phi = \mathrm{w}_{\mathrm{lock}}\phi_{\mathrm{static}} + \left(1-\mathrm{w}_{\mathrm{lock}}\right)\phi_{\mathrm{quad}}.
$$

Thus, $w_{\mathrm{lock}}=1$ indicates that the local lateral-drag response is fully static-like, while $w_{\mathrm{lock}}=0$ indicates that the local response has transitioned fully to the quadratic branch.

This figure is the most direct diagnostic of whether `blend_strain` is behaving as intended. The physically desired structure is that appreciable static/locking weight is confined to the low-speed, low-strain part of phase space. Cells with either high speed or high effective strain rate should have low $w_{\mathrm{lock}}$, meaning they are not treated as dynamically eligible for static landfast locking even if geometric form factors are present.

The figure should be interpreted together with the corresponding $\phi$ phase-space plot (below/next sub-section). The `ldwgt` figure shows **which branch is active**, while the `ldphi` figure shows the **final damping rate applied in the momentum equation**. A high value of $\phi$ does not necessarily imply static locking, because the quadratic branch can also produce large damping at higher speeds. By contrast, \(w_{\mathrm{lock}}\) directly diagnoses the static branch contribution.

For the corrected implementation, the key expected behaviour is:

1. $w_{\mathrm{lock}}$ is largest where both $\mathrm{u} < \mathrm{u}_{\mathrm{blend}}$ and $\epsilon_{\mathrm{eff}} < \epsilon_{\mathrm{blend}}$;
2. $w_{\mathrm{lock}}$ decreases as either speed or strain rate increases;
3. high-strain cells are released from the static branch even if their speed is low;
4. high-speed cells are released from the static branch even if their strain rate is low;
5. N-face and E-face diagnostics show broadly similar phase-space structure.

A populated strain-rate axis confirms that the corrected free-slip strain-rate path is active. Earlier pre-correction experiments had `ldepsN/E` values equal to zero, causing `blend_strain` to collapse to a speed-gated blend. This figure therefore provides a direct diagnostic check that the full speed-and-strain gating is now operating.

#### plot function

In [ ]:
def _face_phase_dataframe(ds: xr.Dataset, face: str,
                          sample_frac: float = SAMPLE_FRAC,
                          max_points : int   = 250_000,
                          min_aice   : float = 0.15,
                          min_ldspd  : float = 0.0,
                          max_ldspd  : float = 2.0e-3,
                          min_ldeps  : float = 1.0e-11,
                          max_ldeps  : float = 1.0e-4,
                          sh_only    : bool  = True) -> pd.DataFrame:
    """
    Build a memory-conscious phase-space dataframe for one face:
      face='N' -> ldspdN, ldepsN, ldwgtN, ldphiN
      face='E' -> ldspdE, ldepsE, ldwgtE, ldphiE
    """
    face = face.upper()
    required = [f"ldspd{face}", f"ldeps{face}", f"ldwgt{face}", f"ldphi{face}"]
    missing = [v for v in required if v not in ds]
    if missing:
        raise KeyError(f"Missing required variables for face {face}: {missing}")
    keep = xr.Dataset({"ldspd": ds[f"ldspd{face}"],
                       "ldeps": ds[f"ldeps{face}"],
                       "ldwgt": ds[f"ldwgt{face}"],
                       "ldphi": ds[f"ldphi{face}"]})
    if "aice" in ds:
        keep["aice"] = ds["aice"]
    if "TLAT" in ds:
        keep["TLAT"] = ds["TLAT"]
    if f"F2{face}" in ds:
        keep["F2"] = ds[f"F2{face}"]
    mask = np.isfinite(keep["ldspd"])
    mask = mask & np.isfinite(keep["ldeps"])
    mask = mask & np.isfinite(keep["ldwgt"])
    mask = mask & np.isfinite(keep["ldphi"])
    mask = mask & (keep["ldspd"] > min_ldspd)
    mask = mask & (keep["ldspd"] <= max_ldspd)
    mask = mask & (keep["ldeps"] > min_ldeps)
    mask = mask & (keep["ldeps"] <= max_ldeps)
    mask = mask & (keep["ldphi"] >= 0.0)
    if "aice" in keep:
        mask = mask & (keep["aice"] >= min_aice)
    if sh_only and "TLAT" in keep:
        mask = mask & (keep["TLAT"] <= -45.0)
    # If F2 exists, require lateral-drag-active geometry.
    # Use >0 rather than a larger threshold for now to avoid removing valid weak-F2 cells.
    if "F2" in keep:
        mask = mask & (keep["F2"] > 0.0)
    df = (keep[["ldspd", "ldeps", "ldwgt", "ldphi"]].where(mask).to_dataframe().dropna().reset_index())
    if df.empty:
        return df
    if sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42)
    if len(df) > max_points:
        df = df.sample(n=max_points, random_state=42)
    df["face"] = face
    df["log10_ldeps"] = np.log10(df["ldeps"])
    return df
    
def plot_blend_ldwgt_phase_space_from_model(sim_name: str,
                                            out: Path | None = None,
                                            sample_frac: float = SAMPLE_FRAC,
                                            max_points_per_face: int = 250_000,
                                            min_aice: float = 0.15,
                                            max_ldspd: float = 2.0e-3,
                                            min_ldeps: float = 1.0e-11,
                                            max_ldeps: float = 1.0e-4,                                        
                                            cmap: str = "viridis"):
    """
    Plot corrected blend_strain phase space:
      x = ldspdN/E
      y = ldepsN/E
      colour = static/locking branch weight ldwgtN/E

    This is the direct diagnostic figure for where blend_strain is applying
    the static/locking branch versus the mobile/quadratic branch.
    """
    variables = ["aice", "TLAT", "F2N", "F2E",
                 "ldspdN", "ldspdE",
                 "ldepsN", "ldepsE",
                 "ldwgtN", "ldwgtE",
                 "ldphiN", "ldphiE"]
    ds  = load_pub_cice(sim_name, ANALYSIS_START, ANALYSIS_END, variables=variables)
    dfN = _face_phase_dataframe(ds, "N",
                                sample_frac=sample_frac,
                                max_points=max_points_per_face,
                                min_aice=min_aice,
                                max_ldspd=max_ldspd,
                                min_ldeps=min_ldeps,
                                max_ldeps=max_ldeps)
    dfE = _face_phase_dataframe(ds, "E",
                                sample_frac=sample_frac,
                                max_points=max_points_per_face,
                                min_aice=min_aice,
                                max_ldspd=max_ldspd,
                                min_ldeps=min_ldeps,
                                max_ldeps=max_ldeps)
    if dfN.empty and dfE.empty:
        raise RuntimeError(f"No valid ldphi phase-space points for {sim_name}. Try relaxing min_aice, min_ldeps/max_ldeps, or F2 filtering.")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), sharex=True, sharey=True, constrained_layout=False)
    last_hb = None
    for ax, face, df in zip(axes, ["N", "E"], [dfN, dfE]):
        if df.empty:
            ax.set_title(f"{sim_name}: {face} face has no valid points")
            ax.axis("off")
            continue
        hb = ax.hexbin(df["ldspd"], df["log10_ldeps"],
                       C=df["ldwgt"],
                       reduce_C_function=np.nanmean,
                       gridsize=(80, 60),
                       mincnt=1,
                       cmap=cmap,
                       vmin=0.0,
                       vmax=1.0)
        last_hb = hb
        ax.axvline(ISPD_THRESH, ls="--", lw=1.1, label="FI speed threshold")
        # Optional experiment-specific threshold lines if EXPERIMENTS exists.
        try:
            if sim_name in EXPERIMENTS.index:
                u_blend = EXPERIMENTS.loc[sim_name].get("u_blend", np.nan)
                eps_blend = EXPERIMENTS.loc[sim_name].get("eps_blend", np.nan)
                if np.isfinite(u_blend):
                    ax.axvline(float(u_blend), ls=":", lw=1.2, label=r"$u_{\mathrm{blend}}$")
                if np.isfinite(eps_blend):
                    ax.axhline(np.log10(float(eps_blend)), ls="-.", lw=1.2, label=r"$\epsilon_{\mathrm{blend}}$")
        except NameError:
            pass
        ax.set_title(f"{sim_name}: {face}-face $w_{{lock}}$")
        ax.set_xlabel(r"Diagnosed speed, $u$ (m s$^{-1}$)")
        ax.grid(alpha=0.25)
        print(f"{sim_name} {face}: n={len(df):,}; "
              f"ldspd={df['ldspd'].min():.2e}–{df['ldspd'].max():.2e}; "
              f"ldeps={df['ldeps'].min():.2e}–{df['ldeps'].max():.2e}; "
              f"ldwgt={df['ldwgt'].min():.2e}–{df['ldwgt'].max():.2e}; "
              f"ldphi={df['ldphi'].min():.2e}–{df['ldphi'].max():.2e}")
    yticks = np.array([1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4])
    yticks = yticks[(yticks >= min_ldeps) & (yticks <= max_ldeps)]
    for ax in axes:
        if ax.has_data():
            ax.set_xlim(0.0, max_ldspd)
            ax.set_ylim(np.log10(min_ldeps), np.log10(max_ldeps))
            ax.set_yticks(np.log10(yticks))
            ax.set_yticklabels([rf"$10^{{{int(np.log10(y))}}}$" for y in yticks])
            ax.legend(frameon=False, loc="upper right")
    axes[0].set_ylabel(r"Diagnosed effective strain rate, $\epsilon_{\mathrm{eff}}$ (s$^{-1}$)")
    fig.subplots_adjust(left=0.08, right=0.88, bottom=0.14, top=0.86, wspace=0.15)
    if last_hb is not None:
        cax = fig.add_axes([0.90, 0.18, 0.018, 0.62])
        cb = fig.colorbar(last_hb, cax=cax)
        cb.set_label(r"Mean static/locking branch weight, $w_{\mathrm{lock}}$")
    fig.suptitle(f"{sim_name}: diagnosed static/locking branch phase space, {ANALYSIS_START} to {ANALYSIS_END}", y=0.96)
    if out is not None:
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print(f"Saved: {out}")
    plt.show()
    df_all = pd.concat([df for df in [dfN, dfE] if not df.empty], ignore_index=True)
    return df_all

#### generate figure

In [ ]:
plt_exp = "LD-blend-base"
P_png   = NOTEBOOK_FIG_ROOT / f"phase_space_ldwgt_{plt_exp}_{ANALYSIS_START}_{ANALYSIS_END}.png"
PHASE_WGT_DF_BASE = plot_blend_ldwgt_phase_space_from_model(plt_exp, P_png if SAVE_FIGS else None,
                                                            sample_frac=0.20,
                                                            min_aice=0.15,
                                                            max_ldspd=2.0e-3,
                                                            min_ldeps=1.0e-11,
                                                            max_ldeps=1.0e-4)

### `ldphi`: diagnosed blend-strain phase space: realised lateral-drag form function

This figure shows the realised lateral-drag diagnostic, $\phi$, diagnosed from the corrected `blend_strain` implementation in CICE. The left and right panels show the N-face and E-face C-grid diagnostics separately.

The x-axis is the diagnosed speed used by the lateral-drag form function, $u = |\mathbf{u}|$, from `ldspdN` and `ldspdE`. The vertical dashed line marks the fast-ice speed threshold used in the classification workflow.

The y-axis is the diagnosed effective strain-rate scale, $\epsilon_{\mathrm{eff}}$, from `ldepsN` and `ldepsE`. This is computed from the U-grid EVP deformation invariant after area normalisation, so it has units of $\mathrm{s}^{-1}$. The logarithmic scale is used because dynamically relevant deformation spans several orders of magnitude.

The colour shows the mean realised lateral-drag form function, $\phi$, within each *speed-strain* *phase-space* bin. In the CICE momentum update, the lateral-drag stress is applied as

$$
\tau_{\mathrm{LD}} = -K_u \, \phi \, \mathbf{u},
$$

where $K_u$ contains the local ice/snow mass and the static (unchanging) **form factor** derived from coastline and grounded-iceberg geometry (described above in the Form Factor creation section). Thus, larger $\phi$ corresponds to a stronger local damping rate for a given $K_u$ and velocity.

For the corrected `blend_strain` form function,

$$
\mathrm{w}_\epsilon = \frac{1}{1 + \left(\epsilon_{\mathrm{eff}} / \epsilon_{\mathrm{blend}}\right)^p},
$$

$$
\mathrm{w}_{\mathrm{u}} = \frac{1}{1 + \left(\mathrm{u} / \mathrm{u}_{\mathrm{blend}}\right)^p},
$$

$$
w_{\mathrm{lock}} = w_\epsilon w_u,
$$

and

$$
\phi = \mathrm{w}_{\mathrm{lock}} \phi_{\mathrm{static}} + \left(1-\mathrm{w}_{\mathrm{lock}}\right)\phi_{\mathrm{quad}}.
$$

The static branch is therefore favoured only when the ice is both slow and weakly deforming. High speed or high strain rate pushes the local form function toward the mobile quadratic branch.

This $\phi$ *phase-space* figure should **not** be interpreted by colour alone as a map of _static locking_. The realised $\phi$ can be large either because the static branch is 'active' at low speed or because the quadratic branch grows with speed. Therefore, this figure is best interpreted together with the corresponding `ldwgt` phase-space plot. The `ldwgt` diagnostic directly shows the static/locking branch weight, while this $\phi$ figure shows the final damping rate actually used by the momentum equation.

The expected behaviour for a physically useful `blend_strain` configuration is:

1. low-speed, low-strain cells retain appreciable static-branch influence;
2. high-strain cells transition away from static locking even if their speed is low;
3. high-speed cells transition toward the mobile branch;
4. the N-face and E-face diagnostics show broadly similar structure, indicating that the result is not an artefact of one velocity staggering direction.

In this corrected run, the populated $\epsilon_{\mathrm{eff}}$ range confirms that the *strain-rate* gate is now active. This is an important distinction from earlier pre-correction runs, in which `ldepsN/E` remained zero and the `blend_strain` form function effectively reduced to a speed-gated blend.

#### plot functions

In [ ]:
def plot_blend_ldphi_phase_space_from_model(sim_name: str,
                                            out                : Path | None = None,
                                            sample_frac        : float = SAMPLE_FRAC,
                                            max_points_per_face: int = 250_000,
                                            min_aice           : float = 0.15,
                                            max_ldspd          : float = 2.0e-3,
                                            min_ldeps          : float = 1.0e-11,
                                            max_ldeps          : float = 1.0e-4,
                                            cmap               : str = "viridis"):
    """
    Plot corrected blend_strain phase space:
      x = ldspdN/E
      y = ldepsN/E
      colour = realised form function ldphiN/E

    This is the direct diagnostic figure for whether the realised damping
    rate is controlled by both speed and strain rate.
    """
    variables = ["aice", "TLAT", "F2N", "F2E",
                 "ldspdN", "ldspdE",
                 "ldepsN", "ldepsE",
                 "ldwgtN", "ldwgtE",
                 "ldphiN", "ldphiE"]
    ds  = load_pub_cice(sim_name, ANALYSIS_START, ANALYSIS_END, variables=variables)
    dfN = _face_phase_dataframe(ds, "N",
                                sample_frac=sample_frac,
                                max_points=max_points_per_face,
                                min_aice=min_aice,
                                max_ldspd=max_ldspd,
                                min_ldeps=min_ldeps,
                                max_ldeps=max_ldeps)
    dfE = _face_phase_dataframe(ds, "E",
                                sample_frac=sample_frac,
                                max_points=max_points_per_face,
                                min_aice=min_aice,
                                max_ldspd=max_ldspd,
                                min_ldeps=min_ldeps,
                                max_ldeps=max_ldeps)
    if dfN.empty and dfE.empty:
        raise RuntimeError(f"No valid ldphi phase-space points for {sim_name}. Try relaxing min_aice, min_ldeps/max_ldeps, or F2 filtering.")
    df_all = pd.concat([df for df in [dfN, dfE] if not df.empty], ignore_index=True)
    # Robust colour range: avoid a few extreme cells dominating the colourbar.
    vmin = float(df_all["ldphi"].quantile(0.01))
    vmax = float(df_all["ldphi"].quantile(0.99))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin = float(df_all["ldphi"].min())
        vmax = float(df_all["ldphi"].max())
    # Create figure with explicit colourbar axis
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), sharex=True, sharey=True, constrained_layout=False)
    last_hb = None
    for ax, face, df in zip(axes, ["N", "E"], [dfN, dfE]):
        if df.empty:
            ax.set_title(f"{sim_name}: {face} face has no valid points")
            ax.axis("off")
            continue
        hb = ax.hexbin(df["ldspd"], df["log10_ldeps"],
                       C                 = df["ldphi"],
                       reduce_C_function = np.nanmean,
                       gridsize          = (80, 60),
                       mincnt            = 1,
                       cmap              = cmap,
                       vmin              = vmin,
                       vmax              = vmax)
        last_hb = hb
        ax.axvline(ISPD_THRESH, ls="--", lw=1.1, label="FI speed threshold")
        # Optional experiment-specific threshold lines if EXPERIMENTS exists.
        try:
            if sim_name in EXPERIMENTS.index:
                u_blend = EXPERIMENTS.loc[sim_name].get("u_blend", np.nan)
                eps_blend = EXPERIMENTS.loc[sim_name].get("eps_blend", np.nan)
                if np.isfinite(u_blend):
                    ax.axvline(float(u_blend), ls=":", lw=1.2, label=r"$u_{blend}$")
                if np.isfinite(eps_blend):
                    ax.axhline(np.log10(float(eps_blend)), ls="-.", lw=1.2, label=r"$\epsilon_{blend}$")
        except NameError:
            pass
        ax.set_title(f"{sim_name}: {face}-face $\\phi$")
        ax.set_xlabel(r"Diagnosed speed, $u$ (m s$^{-1}$)")
        ax.grid(alpha=0.25)
        print(f"{sim_name} {face}: n={len(df):,}; "
              f"ldspd={df['ldspd'].min():.2e}–{df['ldspd'].max():.2e}; "
              f"ldeps={df['ldeps'].min():.2e}–{df['ldeps'].max():.2e}; "
              f"ldphi={df['ldphi'].min():.2e}–{df['ldphi'].max():.2e}; "
              f"ldwgt={df['ldwgt'].min():.2e}–{df['ldwgt'].max():.2e}")
    yticks = np.array([1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4])
    yticks = yticks[(yticks >= min_ldeps) & (yticks <= max_ldeps)]
    for ax in axes:
        if ax.has_data():
            ax.set_xlim(0.0, max_ldspd)
            ax.set_ylim(np.log10(min_ldeps), np.log10(max_ldeps))
            ax.set_yticks(np.log10(yticks))
            ax.set_yticklabels([rf"$10^{{{int(np.log10(y))}}}$" for y in yticks])
            ax.legend(frameon=False, loc="upper right")
    axes[0].set_ylabel(r"Diagnosed effective strain rate, $\epsilon_\mathrm{eff}$ (s$^{-1}$)")
    # After plotting both panels:
    fig.subplots_adjust(left=0.08, right=0.88, bottom=0.14, top=0.86, wspace=0.15)
    if last_hb is not None:
        cax = fig.add_axes([0.90, 0.18, 0.018, 0.62])  # [left, bottom, width, height]
        cb = fig.colorbar(last_hb, cax=cax)
        cb.set_label(r"Mean realised lateral-drag blend-strain, $\phi$ (s$^{-1}$)")
    fig.suptitle(f"{sim_name}: diagnosed $\\phi$ phase space, {ANALYSIS_START} to {ANALYSIS_END}", y=0.96)
    if out is not None:
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print(f"Saved: {out}")
    plt.show()
    return df_all

#### generate figure for an experiment

In [ ]:
plt_exp = "LD-blend-base"
P_png   = NOTEBOOK_FIG_ROOT / f"phase_space_ldphi_{plt_exp}_{ANALYSIS_START}_{ANALYSIS_END}.png"
PHASE_PHI_DF_BASE = plot_blend_ldphi_phase_space_from_model(plt_exp, P_png if SAVE_FIGS else None,
                                                            sample_frac = 0.20,
                                                            min_aice    = 0.15,
                                                            max_ldspd   = 2.0e-3,
                                                            min_ldeps   = 1.0e-11,
                                                            max_ldeps   = 1.0e-4)

## combined diagnostic builder

In [ ]:
def blend_spatial_skill_table(sims: Sequence[str] = BLEND_SIMS) -> pd.DataFrame:
    rows = []
    try:
        obs_mask = load_af2020_mask(ANALYSIS_START, ANALYSIS_END, reference_sim=sims[0])
    except Exception as exc:
        print(f"AF2020 load failed: {exc}")
        return pd.DataFrame()
    for sim in sims:
        try:
            cice      = load_pub_cice(sim, ANALYSIS_START, ANALYSIS_END, variables=["tarea", "TLON", "TLAT"])
            cls       = load_pub_classified(sim, ANALYSIS_START, ANALYSIS_END, variables=["FI_mask"])
            cice, cls = robust_align_time(cice, cls)
            skill     = binary_spatial_skill(cls["FI_mask"], obs_mask, area_da(cice))
            skill["sim_name"] = sim
            rows.append(skill)
        except Exception as exc:
            print(f"Skipping spatial skill for {sim}: {exc}")
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

In [ ]:
BLEND_SKILL = blend_spatial_skill_table()
if len(BLEND_SKILL):
    BLEND_SKILL_MEAN = BLEND_SKILL.groupby("sim_name")[["precision", "recall", "F1", "Jaccard", "false_alarm_ratio", "hit_area", "miss_area", "false_alarm_area"]].mean().reset_index()
    display_df(BLEND_SKILL_MEAN, n=20)
    BLEND_SKILL_MEAN.to_csv(NOTEBOOK_FIG_ROOT / f"blend_spatial_skill_mean_{ANALYSIS_START}_{ANALYSIS_END}.csv", index=False)
    for metric in ["F1", "Jaccard", "false_alarm_ratio"]:
        plot_timeseries_df(BLEND_SKILL, "time", metric, "sim_name",
                           title=f"Corrected blend_strain daily spatial skill: {metric}",
                           ylabel=metric,
                           out=NOTEBOOK_FIG_ROOT / f"blend_spatial_skill_{metric}_{ANALYSIS_START}_{ANALYSIS_END}.png")

# 5. Corrected `blend_strain` versus static, quadratic, and linear form functions

This section is the bridge from the blend-only sensitivity to the full form-function comparison. The static, quadratic, and linear runs may lack the new `ld*` diagnostic fields, so the comparison focuses on outcome and common dynamical fields:

- binary-days FIA and FIP;
- spatial skill against AF2020;
- pack/coastal speed;
- available stress-budget fields;
- lateral-drag stress components where present.

The paper framing should keep `static` as the Liu-style reference, `blend_strain` as the physical candidate, and `quadratic`/`linear` as diagnostic comparators rather than recommended public options.

In [2]:
import shuga
from shuga import configs 
from shuga import loaders 
from shuga import regions 
sim_name     = "LD-blend-base"
START        = "2000-01-01"
END          = "2003-12-31"
ICE_TYPE     = "FI"
METHOD       = "binary-days"
FLD_NAME     = "FIHI"
REG_NAME     = "BS"
FIG_SIZE     = 15
GRD_STY      = "c0.15c"
MOS          = [9,10,11]
REG_PLOT     = regions.ANTARCTIC_8_REGIONS[REG_NAME]["plot_region"]
pth_cfg      = configs.ShugaPaths(lateral_drag = configs.LateralDragSpec())#combined_form_factors_file="/g/data/gv90/da1339/coastal_drag/form_factors/ADD_high-res_cstln_v7p9_GI_CICE_free-slip.nc"))
P_CPT_FIP    = pth_cfg.fip_cmap
P_F2         = pth_cfg.combined_form_factors_path
D_pub        = pth_cfg.graphics_root_path / "LD-pub-workspace"
D_s_f        = D_pub / sim_name / FLD_NAME
D_s_f_r      = D_s_f / REG_NAME
D_ts         = D_pub / "timeseries"
EXPS_SPD_GTE = ["LD-static-Cs1e-3",
                "LD-static-Cs5e-4",
                "LD-quad-Cq350",
                "LD-quad-Cq75",
                "LD-linear-CL0p25"]
EXPS_BLN_GTE = ["LD-blend-base",
                "LD-blend-exp10",
                "LD-blend-eDef-ktens0p1",
                "LD-blend-ef_lt_eg",
                "LD-blend-Cs7p5e-4"]
EXPS_ALL     = EXPS_SPD_GTE + EXPS_BLN_GTE
DYN_VARS     = ["aice", "hi", "tarea", "TLON", "TLAT", "uvel", "vvel", "uocn", "vocn",
                "divu", "shear", "sig1", "sig2", "sigP", "strength",
                "strairx", "strairy", "strocnx", "strocny", "strintx", "strinty", "strcorx", "strcory", "strtltx", "strtlty",
                "KuxE", "KuyE", "KuxN", "KuyN", "F2E", "F2N"]
PLT_FLD_DICT = {"FIP"       : {"series": [0.0, 1.0],
                               "cmap"  : P_CPT_FIP,
                               "frame" : ["a0.2f0.1", 'x+lpersistence']},
                "FIHI"      : {"series": [0.0, 5],    
                               "cmap"  : "cmocean/ice",
                               "frame" : ["xaf", 'x+lthickness', 'y+lm']},
                "FIST"      : {"series": [0.0, 10],
                               "cmap"  : "cmocean/dense",
                               "frame" : ["xaf", 'x+lstrength', "y+lMPa"]},
                "FI_Ku_p90" : {"series": [0.0, 3],
                               "cmap"  : "batlow",
                               "frame" : ["xaf", 'x+lstress', "y+l(N m@+-2@+)"]},
                "strain_p90": {"series": [0.0, 200],
                               "cmap"  : "cmocean/amp",
                               "frame" : ["xaf", 'x+lstrain', "y+l(10@+-6@+ s@+-1@+)"]},
                "FIMAR_YR"  : {"series": [-1, 1],
                               "cmap"  : "polar",
                               "frame" : ["xaf", 'x+lrate' , "y+lkm@+2@+ dy@+-1@+"]}}
print(P_F2)

<frozen importlib._bootstrap>:241: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject


/g/data/gv90/da1339/coastal_drag/form_factors/ADD_high-res_cstln_v7p9_GI_CICE_free-slip.nc


## table comparisons

In [ ]:
COMP_ROWS = []
for sim in EXPERIMENTS:
    for method in ['FI','PI']:
        run_cfg  = RunSpec(sim_name = sim_name, start_date = DT0_STR, end_date = DTN_STR)
        cls_cfg  = ClassificationSpec(ice_type = ICE_TYPE, methods = METHOD)
        mets     = load_metrics(run = run_cfg, classify = cls_cfg, classification = METHOD)
        mets_ext = met_cls.report_metric_extrema(METHOD, variable="FIA")
    for zone_name, mask in {"FI_binary_days": fi, "pack_or_mobile": pack}.items():
        for var in ["ice_speed", "strain_invariant", "tau_air", "tau_ocean", "tau_internal", "tau_ld_est", "R_ld_budget", "P_ld_est"]:
            if var in diag:
                row[f"{zone_name}__{var}__mean"] = float(area_weighted_mean(diag[var], area, mask).mean("time").compute())
    COMP_ROWS.append(form_function_comparison_summary(sim))
COMP_SUMMARY = pd.DataFrame(COMP_ROWS)
display_df(COMP_SUMMARY, n=20)
COMP_SUMMARY.to_csv(NOTEBOOK_FIG_ROOT / f"form_function_comparison_summary_{ANALYSIS_START}_{ANALYSIS_END}.csv", index=False)

In [ ]:
FIA_ROWS = []
for sim in COMPARISON_SIMS:
    fia = daily_fia_from_mask(sim, ANALYSIS_START, ANALYSIS_END)
    df = safe_to_series(fia, "FIA_10^3_km2")
    df["sim_name"] = sim
    FIA_ROWS.append(df)
FIA_COMP = pd.concat(FIA_ROWS, ignore_index=True) if FIA_ROWS else pd.DataFrame()
if len(FIA_COMP):
    plot_timeseries_df(FIA_COMP, "time", "FIA_10^3_km2", "sim_name", 
                       title=f"Binary-days FIA: lateral-drag form-function comparison",
                       ylabel="FIA (10$^3$ km$^2$)",
                       out=NOTEBOOK_FIG_ROOT / f"form_function_fia_comparison_{ANALYSIS_START}_{ANALYSIS_END}.png")


In [ ]:
obs_mask   = load_af2020_mask(ANALYSIS_START, ANALYSIS_END, reference_sim="LD-static-Cs1e-3")
skill_rows = []
for sim in COMPARISON_SIMS:
    cice              = load_pub_cice(sim, ANALYSIS_START, ANALYSIS_END, variables=["tarea", "TLON", "TLAT"])
    cls               = load_pub_classified(sim, ANALYSIS_START, ANALYSIS_END, variables=["FI_mask"])
    cice, cls         = robust_align_time(cice, cls)
    skill             = binary_spatial_skill(cls["FI_mask"], obs_mask, area_da(cice))
    skill["sim_name"] = sim
    skill_rows.append(skill)
COMP_SKILL = pd.concat(skill_rows, ignore_index=True) if skill_rows else pd.DataFrame()
if len(COMP_SKILL):
    COMP_SKILL_MEAN = COMP_SKILL.groupby("sim_name")[["precision", "recall", "F1", "Jaccard", "false_alarm_ratio", "hit_area", "miss_area", "false_alarm_area"]].mean().reset_index()
    display_df(COMP_SKILL_MEAN, n=20)
    COMP_SKILL_MEAN.to_csv(NOTEBOOK_FIG_ROOT / f"form_function_spatial_skill_mean_{ANALYSIS_START}_{ANALYSIS_END}.csv", index=False)


## FIA


In [ ]:
pltr = shuga.CICEPlotter(sim_name   = "LD-blend-base",
                         start_date = START,
                         end_date   = END,
                         ice_type   = ICE_TYPE,
                         method     = METHOD)
plt_var = "FIA"
plt_suf = "speed-gates"
P_fig   = pltr.plot_timeseries_multi(variable        = plt_var,
                                     method          = METHOD,
                                     simulations     = EXPS_SPD_GTE,
                                     region          = "total",
                                     legend_position = "JMR+jCM+o2c",
                                     output_path     = str(Path(D_ts,f"{plt_var}_SH_{plt_suf}.png")),
                                     add_f2020       = True,
                                     f2020_mode      = "overlap")
display(Image(filename=P_fig))

## FIP

In [ ]:
FLD_NAME = "FIP"
for REG_NAME in regions.ANTARCTIC_8_REGIONS.keys():
    for sim_name in EXPERIMENTS:
        D_s_f_r = D_pub / sim_name / FLD_NAME / REG_NAME
        P_png   = D_s_f_r / f"{START}_{END}.png"
        pltr    = shuga.CICEPlotter(sim_name   = sim_name,
                                    start_date = START,
                                    end_date   = END,
                                    ice_type   = ICE_TYPE,
                                    method     = METHOD)
        pltr.plot_fip(method            = METHOD,
                      sim_name          = sim_name,
                      dt0_str           = START,
                      dtN_str           = END,
                      region_name       = REG_NAME,
                      output_path       = str(P_png),
                      title             = sim_name,
                      fig_size          = 20.0,
                      grid_style        = "c0.15c",
                      colorbar_position = 'JMB+w15c',
                      show              = False)

## $Q_{0.90,t}(|\boldsymbol{\tau}_{\mathrm{LD},u}|)$ and $\overline{h}_{\mathrm{FI}}$

### Dynamic lateral-drag stress magnitude

The lateral-drag stress diagnostic is computed from the CICE lateral-drag stress components saved on the east and north staggered velocity locations. For each simulation, the four component fields are loaded from the CICE history output:

```python
KuxE, KuyE, KuxN, KuyN
```

where `KuxE` and `KuyE` are the two horizontal components of the lateral-drag stress vector on the east velocity location, and `KuxN` and `KuyN` are the corresponding components on the north velocity location. The fast-ice mask is also loaded from the selected classification method:

```python
FI_mask
```

The analysis is restricted to the requested time interval and, subsequently, to the selected months. The lateral-drag stress magnitude is first computed separately at the east and north velocity locations as $\boldsymbol{\tau}_{\mathrm{LD},E}$

$$
\left[K_{ux,E}^{2} + K_{uy,E}^{2}\right]^{1/2},
$$

and $\boldsymbol{\tau}_{\mathrm{LD},N}$

$$
\left[K_{ux,N}^{2} + K_{uy,N}^{2}\right]^{1/2}.
$$

These two staggered-grid magnitudes are then averaged to give a single representative lateral-drag stress magnitude, $\boldsymbol{\tau}_{\mathrm{LD},u}$

$$
\frac{1}{2}\left(|\boldsymbol{\tau}*{\mathrm{LD},E}| + |\boldsymbol{\tau}*{\mathrm{LD},N}|\right).
$$

The plotted field is the 90th percentile in time of this stress magnitude:

$$
Q_{0.90,t}\left(|\boldsymbol{\tau}_{\mathrm{LD},u}|\right),
$$

where ($Q_{0.90,t}$) denotes the temporal 90th percentile. This diagnostic highlights regions where lateral-drag stresses are episodically strong during the selected seasonal window, rather than showing the seasonal mean.

In [7]:
import pygmt
FLD_NAME = "FIHI_and_LD-tau" # only for PNG writing
MET_NAME = "FIHI"
tau_mnmx = [0,3]
met_mnmx = [0,5]
cmap_tau = "cmocean/turbid"
cmap_met = "cmocean/matter"
cbar_tau = ["xaf", 'x+lstress', "y+l(N m@+-2@+)"]
cbar_met = ["xaf", 'x+lthickness', 'y+lm']
show_fig = False
overwrite = True
FF       = xr.open_dataset(P_F2) # geometric form-factors
# use GICB-only form-factor for visual overlay; coastal part is redundant with fig.coast().
F2_mag   = xr.apply_ufunc(np.hypot, FF["F2x_gi"], FF["F2y_gi"], dask="allowed")
# explicit binary mask; keeps non-F2 cells as 0.0, not NaN, so GMT can contour the 0/1 transition.
F2_bin   = xr.where(np.isfinite(F2_mag) & (F2_mag > 0.0), 1.0, 0.0)
for sim_name in EXPS_ALL:
    run_cfg  = configs.RunSpec(sim_name = sim_name, start_date = START, end_date = END)
    cls_cfg  = configs.ClassificationSpec(ice_type = ICE_TYPE, methods = METHOD)
    met_cfg  = configs.MetricsSpec(methods = METHOD)
    plt_cfg  = configs.PlottingSpec()
    obs_cfg  = configs.ObservationSpec()
    pltr     = shuga.CICEPlotter(run = run_cfg, classify = cls_cfg, metrics = met_cfg, plotting = plt_cfg, observations = obs_cfg)
    coords   = pltr._load_static_lonlat()
    lon      = coords["TLON"]
    lat      = coords["TLAT"]
    met_da   = shuga.load_metrics(sim_name = sim_name, classification = METHOD)[MET_NAME]
    ld_taus  = shuga.load_cice(sim_name = sim_name, variables = ["KuxE", "KuyE", "KuxN", "KuyN"]).sel(time=slice(START, END))
    cls_ds   = shuga.load_classified(sim_name = sim_name, classification = METHOD, variables=["FI_mask"])
    FI_mask  = cls_ds["FI_mask"].astype(bool).sel(time=slice(START, END))
    ds       = ld_taus.sel(time=ld_taus["time"].dt.month.isin(MOS))
    FI_mask  = FI_mask.sel(time=FI_mask["time"].dt.month.isin(MOS))
    KuxE, KuyE, KuxN, KuyN, FI_mask = xr.align(ld_taus["KuxE"], ld_taus["KuyE"], ld_taus["KuxN"], ld_taus["KuyN"], FI_mask, join="inner")
    KuE      = np.hypot(KuxE, KuyE)
    KuN      = np.hypot(KuxN, KuyN)
    Ku       = 0.5 * (KuE + KuN)
    Ku       = Ku.where(FI_mask)
    Ku       = Ku.quantile(0.90, dim="time", skipna=True).drop_vars("quantile", errors="ignore")
    for REG_NAME in regions.ANTARCTIC_8_REGIONS.keys():
        D_s_f_r = D_pub / sim_name / FLD_NAME / REG_NAME
        D_s_f_r.mkdir(parents=True, exist_ok=True)
        P_png   = D_s_f_r / f"{START}_{END}.png"
        if P_png.exists() and not overwrite:
            print(f"{P_png} exists and not overwriting")
            continue
        REG_PLOT = regions.ANTARCTIC_8_REGIONS[REG_NAME]["plot_region"]
        plt_met  = pltr.pygmt_da_prep(met_da, lon=lon, lat=lat, mask_zero=False, region=REG_PLOT)
        plt_tau  = pltr.pygmt_da_prep(Ku, lon=lon, lat=lat, mask_zero=False, region=REG_PLOT)
        plt_F2   = pltr.pygmt_da_prep(F2_bin, lon=lon, lat=lat, mask_zero=False, region=REG_PLOT)
        fig      = pygmt.Figure()
        proj     = pltr.projection_from_region(REG_PLOT, fig_size=FIG_SIZE)
        fig.basemap(region = REG_PLOT, projection = proj, frame = ["af", r"+t@[Q_{0.90,t}(|\boldsymbol{\tau}_{\mathrm{LD},u}|)@["])
        fig.coast(shorelines = "0.25p,black", land = "lightgray", water = "white")
        pygmt.makecpt(cmap = cmap_tau, series = tau_mnmx)
        fig.plot(x = plt_tau["lon"], y = plt_tau["lat"], style = GRD_STY, fill = plt_tau["z"], cmap = True, pen = None)
        fig.contour(x = plt_F2['lon'], y = plt_F2['lat'], z = plt_F2['z'], levels = [0.5], pen = "0.45p,blue")
        fig.colorbar(position = "JBC+w6.5c+o0c/-0.1c+h", frame = cbar_tau)
        fig.shift_origin(xshift="w+10c", yshift="0c")
        fig.basemap(region = REG_PLOT, projection = proj, frame = ["af", r"+t@[\overline{h}_{\mathrm{FI}}@["])
        fig.coast(shorelines = "0.25p,black", land = "lightgray", water = "white")
        pygmt.makecpt(cmap = cmap_met, series = tau_mnmx)
        fig.plot(x = plt_met["lon"], y = plt_met["lat"], style = GRD_STY, fill = plt_met["z"], cmap = True, pen = None)
        fig.colorbar(position = "JBC+w6.5c+o0c/-0.1c+h", frame = cbar_met)
        if overwrite:
            print(f"writing to {P_png}")
            fig.savefig(P_png)
        if show_fig:
            display(Image(filename=P_png))

INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Resolving classification context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Resolved metrics store for LD-static-Cs1e-3 [Tc/binary-days] by exact path: /home/581/da1339/AFIM_archive/LD-static-Cs1e-3/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Opening metrics store for LD-static-Cs1e-3 [Tc/binary-days]: /home/581/da1339/AFIM_archive/LD-static-Cs1e-3/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Opening grouped monthly CICE Zarr between 1999-01-01 00:00:00 and 2003-12-31 00:00:00 (60 groups)
INFO:shuga.io.zarr_loading:Opening group 1999-01 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1999-02 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1999-03 with c

writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs1e-3/FIHI_and_LD-tau/DML/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs1e-3/FIHI_and_LD-tau/WIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs1e-3/FIHI_and_LD-tau/EIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs1e-3/FIHI_and_LD-tau/Aus/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs1e-3/FIHI_and_LD-tau/VOL/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs1e-3/FIHI_and_LD-tau/AS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs1e-3/FIHI_and_LD-tau/BS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs1e-3/FIHI_and_LD-tau/WS/2000-01-01_2003-12-31.png


INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Resolving classification context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Resolved metrics store for LD-static-Cs5e-4 [Tc/binary-days] by exact path: /home/581/da1339/AFIM_archive/LD-static-Cs5e-4/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Opening metrics store for LD-static-Cs5e-4 [Tc/binary-days]: /home/581/da1339/AFIM_archive/LD-static-Cs5e-4/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Opening grouped monthly CICE Zarr between 1994-09-01 00:00:00 and 2003-12-31 00:00:00 (64 groups)
INFO:shuga.io.zarr_loading:Opening group 1994-09 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1994-10 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1994-11 with c

writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs5e-4/FIHI_and_LD-tau/DML/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs5e-4/FIHI_and_LD-tau/WIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs5e-4/FIHI_and_LD-tau/EIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs5e-4/FIHI_and_LD-tau/Aus/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs5e-4/FIHI_and_LD-tau/VOL/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs5e-4/FIHI_and_LD-tau/AS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs5e-4/FIHI_and_LD-tau/BS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-static-Cs5e-4/FIHI_and_LD-tau/WS/2000-01-01_2003-12-31.png


INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Resolving classification context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Resolved metrics store for LD-quad-Cq350 [Tc/binary-days] by exact path: /home/581/da1339/AFIM_archive/LD-quad-Cq350/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Opening metrics store for LD-quad-Cq350 [Tc/binary-days]: /home/581/da1339/AFIM_archive/LD-quad-Cq350/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Opening grouped monthly CICE Zarr between 1999-01-01 00:00:00 and 2003-12-31 00:00:00 (60 groups)
INFO:shuga.io.zarr_loading:Opening group 1999-01 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1999-02 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1999-03 with chunks={'time

...

writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-eDef-ktens0p1/FIHI_and_LD-tau/DML/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-eDef-ktens0p1/FIHI_and_LD-tau/WIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-eDef-ktens0p1/FIHI_and_LD-tau/EIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-eDef-ktens0p1/FIHI_and_LD-tau/Aus/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-eDef-ktens0p1/FIHI_and_LD-tau/VOL/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-eDef-ktens0p1/FIHI_and_LD-tau/AS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-eDef-ktens0p1/FIHI_and_LD-tau/BS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-eDef-ktens0p1/FIHI_and_LD-tau/WS/2000-01-01_20

INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Resolving classification context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Resolved metrics store for LD-blend-ef_lt_eg [Tc/binary-days] by exact path: /home/581/da1339/AFIM_archive/LD-blend-ef_lt_eg/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Opening metrics store for LD-blend-ef_lt_eg [Tc/binary-days]: /home/581/da1339/AFIM_archive/LD-blend-ef_lt_eg/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Opening grouped monthly CICE Zarr between 1999-01-01 00:00:00 and 2003-12-31 00:00:00 (60 groups)
INFO:shuga.io.zarr_loading:Opening group 1999-01 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1999-02 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1999-03 wi

writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-ef_lt_eg/FIHI_and_LD-tau/DML/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-ef_lt_eg/FIHI_and_LD-tau/WIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-ef_lt_eg/FIHI_and_LD-tau/EIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-ef_lt_eg/FIHI_and_LD-tau/Aus/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-ef_lt_eg/FIHI_and_LD-tau/VOL/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-ef_lt_eg/FIHI_and_LD-tau/AS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-ef_lt_eg/FIHI_and_LD-tau/BS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-ef_lt_eg/FIHI_and_LD-tau/WS/2000-01-01_2003-12-31.png


INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Resolving classification context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Resolved metrics store for LD-blend-Cs7p5e-4 [Tc/binary-days] by exact path: /home/581/da1339/AFIM_archive/LD-blend-Cs7p5e-4/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Opening metrics store for LD-blend-Cs7p5e-4 [Tc/binary-days]: /home/581/da1339/AFIM_archive/LD-blend-Cs7p5e-4/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Opening grouped monthly CICE Zarr between 1999-01-01 00:00:00 and 2003-12-31 00:00:00 (60 groups)
INFO:shuga.io.zarr_loading:Opening group 1999-01 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1999-02 with chunks={'time': 31}
INFO:shuga.io.zarr_loading:Opening group 1999-03 wi

writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-Cs7p5e-4/FIHI_and_LD-tau/DML/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-Cs7p5e-4/FIHI_and_LD-tau/WIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-Cs7p5e-4/FIHI_and_LD-tau/EIO/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-Cs7p5e-4/FIHI_and_LD-tau/Aus/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-Cs7p5e-4/FIHI_and_LD-tau/VOL/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-Cs7p5e-4/FIHI_and_LD-tau/AS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-Cs7p5e-4/FIHI_and_LD-tau/BS/2000-01-01_2003-12-31.png
writing to /g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/LD-blend-Cs7p5e-4/FIHI_and_LD-tau/WS/2000-01-01_2003-12-31.png


## $Q_{0.90,t}\!\left(\dot{\varepsilon}_{\mathrm{inv}}\right)$ and $S_{c,\mathrm{FI}}$

### Strain-rate invariant

The strain-rate invariant diagnostic is computed from the CICE divergence and shear strain-rate diagnostics: `divu` and `shear` . After optional seasonal subsetting, the dataset is cropped to the regional bounding box and the model-grid longitude and latitude fields are retained for plotting. The scalar strain-rate magnitude is then computed as $\dot{\varepsilon}_{\mathrm{inv}}$

$$
\left[(\nabla \cdot \mathbf{u})^{2} + \dot{\gamma}^{2}\right]^{1/2},
$$

where ($\nabla \cdot \mathbf{u}$) is the horizontal sea-ice velocity divergence and ($\dot{\gamma}$) is the shear strain rate. In the code, this is evaluated as

```python
eps = np.hypot(ds["divu"], ds["shear"])
```

which is numerically equivalent to

$$
\sqrt{\mathrm{divu}^{2} + \mathrm{shear}^{2}}.
$$

If an ice-concentration threshold is specified, the strain-rate invariant is masked to retain only sea-ice-covered cells:

$$
a_{\mathrm{ice}} \geq a_{\mathrm{ice,thresh}}.
$$

The field is then reduced in time using the selected dynamic time statistic, typically the 90th percentile:

$$
Q_{0.90,t}\left(\dot{\varepsilon}_{\mathrm{inv}}\right).
$$

Finally, the reduced field is masked to the selected Antarctic sector using the regional polygon mask.


#### Implementation note

The code currently applies

```python
eps2 = eps2.where(SP._region_mask(lon, lat, reg)) / 1e6
```

while preserving units inferred from `divu`, usually `s-1`. If the intention is to plot the field in ($10^{-6}\ \mathrm{s}^{-1}$), then the usual scaling would be multiplication by ($10^6$), with the colorbar labelled as ($10^{-6}\ \mathrm{s}^{-1}$). If the division by ($10^6$) is intentional, the units/label should be adjusted to avoid ambiguity.

### Area-weighted mean fast-ice compressive strength ( $\overline{\sigma}_{c,\mathrm{FI}}$ )

The diagnostic previously referred to as `FIST` is computed as an area-weighted mean of thickness-normalised sea-ice compressive strength over the selected sea-ice or fast-ice mask. In CICE, the strength variable represents an ice-strength quantity integrated over ice thickness. To express this as a stress-like compressive strength, the field is normalised by ice thickness and converted from Pa to MPa ($\sigma_c$):

$$
\frac{P}{h_i},
$$

where ($P$) is the modelled compressive ice strength and ($h_i$) is sea-ice thickness. Grid cells with non-positive ice thickness are excluded. When a fast-ice mask is supplied, the area-weighted mean fast-ice compressive strength is defined as $\overline{\sigma}_{c,\mathrm{FI}}$

$$
\frac{\sum_{\Omega} M_{\mathrm{FI}},a_i,A\left(P / h_i\right)}{\sum_{\Omega}M_{\mathrm{FI}},a_i,A},
$$

where ($M_{\mathrm{FI}}$) is the fast-ice mask, ($a_i$) is sea-ice concentration, ($A$) is grid-cell area, and ($\Omega$) is the spatial domain of the selected region. The sea-ice concentration weighting accounts for the fractional ice-covered area within each grid cell. The resulting diagnostic has units of MPa. This diagnostic therefore represents the mean compressive strength of the ice within the selected fast-ice-covered area, after normalising the modelled ice strength by thickness. It is not a thickness field; rather, it is a stress-like measure of the local resistance of the ice cover to compression, averaged over the fast-ice mask.

In [ ]:
FLD_NAME = "FIST_and_strain-invariant" # only for PNG writing
MET_NAME = "FIST"
tau_mnmx = [0,200]
met_mnmx = [0,10]
cmap_tau = "cmocean/turbid"
cmap_met = "cmocean/matter"
cbar_tau = ["xaf", 'x+lstrain', r"y+l@[10^6\ s^{-1}@[)"]
cbar_met = ["xaf", 'x+lstrength', "y+lMPa"]
show_fig = False
overwrite = True
AICE_THRESH = 0.15
FF       = xr.open_dataset(P_F2) # geometric form-factors
# use GICB-only form-factor for visual overlay; coastal part is redundant with fig.coast().
F2_mag   = xr.apply_ufunc(np.hypot, FF["F2x_gi"], FF["F2y_gi"], dask="allowed")
# explicit binary mask; keeps non-F2 cells as 0.0, not NaN, so GMT can contour the 0/1 transition.
F2_bin   = xr.where(np.isfinite(F2_mag) & (F2_mag > 0.0), 1.0, 0.0)
for sim_name in EXPS_ALL:
    run_cfg  = configs.RunSpec(sim_name = sim_name, start_date = START, end_date = END)
    cls_cfg  = configs.ClassificationSpec(ice_type = ICE_TYPE, methods = METHOD)
    met_cfg  = configs.MetricsSpec(methods = METHOD)
    plt_cfg  = configs.PlottingSpec()
    obs_cfg  = configs.ObservationSpec()
    pltr     = shuga.CICEPlotter(run = run_cfg, classify = cls_cfg, metrics = met_cfg, plotting = plt_cfg, observations = obs_cfg)
    coords   = pltr._load_static_lonlat()
    lon      = coords["TLON"]
    lat      = coords["TLAT"]
    met_da   = shuga.load_metrics(sim_name = sim_name, classification = METHOD)[MET_NAME]
    si_strn  = shuga.load_cice(sim_name = sim_name, variables = ["divu", "shear"]).sel(time=slice(START, END))
    si_strn  = si_strn.sel(time=si_strn["time"].dt.month.isin(months))
    eps      = np.hypot(si_strn["divu"], si_strn["shear"])
    eps      = eps.where(ds["aice"] >= AICE_THRESH)
    eps      = eps.quantile(0.90, dim="time", skipna=True).drop_vars("quantile", errors="ignore") / 1e6 #otherwise values are on the order of 10^7
    for REG_NAME in regions.ANTARCTIC_8_REGIONS.keys():
        D_s_f_r = D_pub / sim_name / FLD_NAME / REG_NAME
        D_s_f_r.mkdir(parents=True, exist_ok=True)
        P_png   = D_s_f_r / f"{START}_{END}.png"
        if P_png.exists() and not overwrite:
            print(f"{P_png} exists and not overwriting")
            continue
        REG_PLOT = regions.ANTARCTIC_8_REGIONS[REG_NAME]["plot_region"]
        plt_met  = pltr.pygmt_da_prep(met_da, lon=lon, lat=lat, mask_zero=False, region=REG_PLOT)
        plt_tau  = pltr.pygmt_da_prep(Ku, lon=lon, lat=lat, mask_zero=False, region=REG_PLOT)
        plt_F2   = pltr.pygmt_da_prep(F2_bin, lon=lon, lat=lat, mask_zero=False, region=REG_PLOT)
        fig      = pygmt.Figure()
        proj     = pltr.projection_from_region(REG_PLOT, fig_size=FIG_SIZE)
        fig.basemap(region = REG_PLOT, projection = proj, frame = ["af", r"+t@Q_{0.90,t}\!\left(\dot{\varepsilon}_{\mathrm{inv}}\right)@["])
        fig.coast(shorelines = "0.25p,black", land = "lightgray", water = "white")
        pygmt.makecpt(cmap = cmap_tau, series = tau_mnmx)
        fig.plot(x = plt_tau["lon"], y = plt_tau["lat"], style = GRD_STY, fill = plt_tau["z"], cmap = True, pen = None)
        fig.contour(x = plt_F2['lon'], y = plt_F2['lat'], z = plt_F2['z'], levels = [0.5], pen = "0.45p,blue")
        fig.colorbar(position = "JBC+w6.5c+o0c/-0.1c+h", frame = cbar_tau)
        fig.shift_origin(xshift="w+10c", yshift="0c")
        fig.basemap(region = REG_PLOT, projection = proj, frame = ["af", r"+t@[\overline{h}_{\mathrm{FI}}@["])
        fig.coast(shorelines = "0.25p,black", land = "lightgray", water = "white")
        pygmt.makecpt(cmap = cmap_met, series = tau_mnmx)
        fig.plot(x = plt_met["lon"], y = plt_met["lat"], style = GRD_STY, fill = plt_met["z"], cmap = True, pen = None)
        fig.colorbar(position = "JBC+w6.5c+o0c/-0.1c+h", frame = cbar_met)
        if overwrite:
            print(f"writing to {P_png}")
            fig.savefig(P_png)
        if show_fig:
            display(Image(filename=P_png))

## multi-map: FIP, FIHI, FIST, FI |Ku| p90, strain invariant p90, FIMAR_YR

In [ ]:
import calendar
import pygmt
from shuga.core.naming import normalize_method
# -------------------------------------------------------------------
START            = "2000-01-01"
END              = "2003-12-31"
HEMISPHERE       = "SH"
GRID_TYPE        = "Tc"
ICE_TYPE         = "FI"
METHOD           = normalize_method("binary-days")
PROJECT          = "gv90"
USER             = "da1339"
ISP_THRESH       = 5e-4
BIN_WINDOW       = 11
BIN_MIN_DAYS     = 9
ROLL_WINDOW      = 15
CHUNKS           = {"time": 31}
REGION_NAME      = "Aus"
MONTHS_DYN       = [9,10,11]      # Dynamic/mechanistic panels use a seasonal subset.
TIME_STAT_DYN    = "p90"       # Time statistic for Ku and strain panels
AICE_MASK_THRESH = 0.15
FIG_SIZE         = 20.0              
GRID_STYLE       = "s0.15c"      # smaller point size; less blocky
SHOW_FIGS        = False
OVERWRITE        = True
ADD_F2_OUTLINE   = True
F2_FILE          = Path("/g/data/gv90/da1339/coastal_drag/form_factors/ADD_high-res_cstln_v7p9_GI_CICE_free-slip.nc")
GRAPH_ROOT       = Path(f"/g/data/gv90/da1339/GRAPHICAL")
D_out            = GRAPH_ROOT / "LD-pub-workspace"
D_SUMMARY        = D_out / "per_experiment_summary"
D_SUMMARY.mkdir(parents=True, exist_ok=True)
print("Output directory:", D_SUMMARY)
EXPERIMENTS  = [#"LD-NIL",
                "LD-static-Cs1e-3",
                "LD-static-Cs5e-4",
                "LD-quad-Cq350",
                "LD-quad-Cq75",
                "LD-linear-CL0p25",
                "LD-blend-base",
                "LD-blend-exp10",
                "LD-blend-Cs7p5e-4",
                "LD-blend-ktens0p2",
                "LD-blend-eDef-ktens0p1",
                "LD-blend-ef_lt_eg"]

### helpers

In [ ]:
def make_specs(sim_name):
    run = RunSpec(sim_name=sim_name,
                    start_date=START,
                    end_date=END,
                    hemisphere=HEMISPHERE,
                    project=PROJECT,
                    user=USER,
                    iceh_frequency="daily")
    classify = ClassificationSpec(ice_type=ICE_TYPE,
                                    grid_type=GRID_TYPE,
                                    ispd_thresh=ISP_THRESH,
                                    methods=(METHOD,),
                                    bin_window=BIN_WINDOW,
                                    bin_min_days=BIN_MIN_DAYS,
                                    roll_window=ROLL_WINDOW)
    metrics = MetricsSpec(methods=(METHOD))
    plotting = PlottingSpec()
    observations = ObservationSpec()
    return run, classify, metrics, plotting, observations

def make_plotter(sim_name):
    run, classify, metrics, plotting, observations = make_specs(sim_name)
    return CICEPlotter(run=run,
                        classify=classify,
                        metrics=metrics,
                        plotting=plotting,
                        observations=observations,
                        chunks=CHUNKS)

def months_token(months):
    if months is None:
        return "all-months"
    months = sorted(int(m) for m in months)
    names = {(12, 1, 2): "DJF",
            (1, 2, 12): "DJF",
            (3, 4, 5): "MAM",
            (4, 5, 6): "AMJ",
            (5, 6, 7): "MJJ",
            (6, 7, 8): "JJA",
            (7, 8, 9): "JAS",
            (8, 9, 10): "ASO",
            (9, 10, 11): "SON",
            (10, 11, 12): "OND"}
    key = tuple(months)
    if key in names:
        return names[key]
    return "m" + "-".join(f"{m:02d}" for m in months)

def time_reduce(da, stat="p90"):
    stat = stat.lower()
    if "time" not in da.dims:
        return da
    if stat == "mean":
        return da.mean("time", skipna=True)
    if stat == "median":
        return da.median("time", skipna=True)
    if stat == "p90":
        out = da.quantile(0.90, dim="time", skipna=True)
        return out.drop_vars("quantile", errors="ignore")
    if stat == "p95":
        out = da.quantile(0.95, dim="time", skipna=True)
        return out.drop_vars("quantile", errors="ignore")
    if stat == "max":
        return da.max("time", skipna=True)
    raise ValueError(f"Unknown time statistic: {stat}")

def crop_to_region_bbox(SP, ds, lon, lat, reg):
    mask = SP._region_mask(lon, lat, reg)
    mask_np = mask.compute().values
    if not np.any(mask_np):
        raise ValueError(f"No cells found inside region {reg}")
    yy, xx = np.where(mask_np)
    ydim, xdim = lon.dims
    yslice = slice(int(yy.min()), int(yy.max()) + 1)
    xslice = slice(int(xx.min()), int(xx.max()) + 1)
    indexer = {ydim: yslice, xdim: xslice}
    ds_c = ds.isel(indexer)
    lon_c = lon.isel(indexer)
    lat_c = lat.isel(indexer)
    return ds_c, lon_c, lat_c, indexer

def load_f2_outline(SP, coords, reg):
    if not ADD_F2_OUTLINE:
        return None
    if not F2_FILE.exists():
        print(f"[WARN] F2 file not found: {F2_FILE}")
        return None
    FF = xr.open_dataset(F2_FILE)
    F2_mag = np.hypot(FF["F2x"], FF["F2y"])
    F2_mask = xr.where(F2_mag > 0.0, 1.0, 0.0)
    f2_reg_mask = SP._region_mask(coords["TLON"], coords["TLAT"], reg)
    F2_plot = xr.where(f2_reg_mask, F2_mask, np.nan)
    lon2d = SP._lon_to_180(coords["TLON"]).values
    lat2d = coords["TLAT"].values
    z2d = F2_plot.values
    m = np.isfinite(lon2d) & np.isfinite(lat2d) & np.isfinite(z2d)
    return lon2d, lat2d, z2d, m

def infer_units(da, fallback=""):
    units = da.attrs.get("units", fallback)
    if units is None:
        units = fallback
    return str(units)

def region_mask_from_plotter(SP, lon, lat, reg):
    return SP._region_mask(lon, lat, reg)

def collect_z(field_key, finite_only=True):
    vals = []
    for summary in SUMMARIES.values():
        df = summary["fields"][field_key]["df"]
        if len(df) == 0:
            continue
        z = df["z"].to_numpy()
        if finite_only:
            z = z[np.isfinite(z)]
        if z.size:
            vals.append(z)
    if not vals:
        return np.array([])
    return np.concatenate(vals)

def load_metric_2d(SP, sim, field_name):
    run = replace(SP.run, sim_name=sim)
    ds_met = load_metrics(run=run,
                            classify=SP.classify,
                            metrics=SP.metrics,
                            plotting=SP.plotting,
                            observations=SP.observations,
                            paths=SP.paths,
                            classification=METHOD,
                            dt0_str=START,
                            dtN_str=END,
                            hemisphere=run.hemisphere,
                            chunks=SP.chunks)
    if field_name not in ds_met:
        raise KeyError(f"{field_name!r} not found in metrics store for {sim}. Available variables: {list(ds_met.data_vars)}")
    da = ds_met[field_name]
    if "time" in da.dims:
        da = da.sel(time=slice(START, END)).mean("time", skipna=True)
    return da

def compute_fi_ku_p90(SP, sim, reg, months=MONTHS_DYN):
    """
    Compute FI-masked p90 lateral-drag magnitude:
        Ku = 0.5 * (sqrt(KuxE^2 + KuyE^2) + sqrt(KuxN^2 + KuyN^2))
    """
    run = replace(SP.run, sim_name=sim)
    ds = load_cice(run=run,
                    classify=SP.classify,
                    metrics=SP.metrics,
                    plotting=SP.plotting,
                    observations=SP.observations,
                    paths=SP.paths,
                    variables=["KuxE", "KuyE", "KuxN", "KuyN", "TLON", "TLAT"],
                    hemisphere=run.hemisphere,
                    chunks={"time": 31}).sel(time=slice(START, END))
    cls = load_classified(run=run,
                            classify=SP.classify,
                            metrics=SP.metrics,
                            plotting=SP.plotting,
                            observations=SP.observations,
                            paths=SP.paths,
                            classification=METHOD,
                            dt0_str=START,
                            dtN_str=END,
                            variables=["FI_mask"],
                            hemisphere=run.hemisphere,
                            chunks={"time": 31})
    FI_mask = cls["FI_mask"].astype(bool).sel(time=slice(START, END))
    if months is not None:
        ds = ds.sel(time=ds["time"].dt.month.isin(months))
        FI_mask = FI_mask.sel(time=FI_mask["time"].dt.month.isin(months))
    lon = ds["TLON"]
    lat = ds["TLAT"]
    ds, lon, lat, bbox_indexer = crop_to_region_bbox(SP, ds, lon, lat, reg)
    # This is the critical correction:
    FI_mask = FI_mask.isel(bbox_indexer)
    KuxE, KuyE, KuxN, KuyN, FI_mask = xr.align(ds["KuxE"], ds["KuyE"], ds["KuxN"], ds["KuyN"], FI_mask, join="inner")
    KuE = np.hypot(KuxE, KuyE)
    KuN = np.hypot(KuxN, KuyN)
    Ku = 0.5 * (KuE + KuN)
    Ku = Ku.where(FI_mask)
    Ku2 = time_reduce(Ku, TIME_STAT_DYN)
    Ku2 = Ku2.where(SP._region_mask(lon, lat, reg))
    Ku2.name = "FI_Ku_p90"
    Ku2.attrs["units"] = infer_units(ds["KuxE"], "N m-2")
    return Ku2, lon, lat

def compute_strain_invariant_p90(SP, sim, reg, months=MONTHS_DYN):
    """
    Compute p90 strain invariant:
        eps = sqrt(divu^2 + shear^2)

    This is sea-ice masked, not FI-masked.
    """
    run = replace(SP.run, sim_name=sim)
    ds = load_cice(run=run,
                    classify=SP.classify,
                    metrics=SP.metrics,
                    plotting=SP.plotting,
                    observations=SP.observations,
                    paths=SP.paths,
                    variables=["divu", "shear", "aice", "TLON", "TLAT"],
                    hemisphere=run.hemisphere,
                    chunks={"time": 31}).sel(time=slice(START, END))
    if months is not None:
        ds = ds.sel(time=ds["time"].dt.month.isin(months))
    lon = ds["TLON"]
    lat = ds["TLAT"]
    ds, lon, lat, bbox_indexer = crop_to_region_bbox(SP, ds, lon, lat, reg)
    eps = np.hypot(ds["divu"], ds["shear"])
    if AICE_MASK_THRESH is not None:
        eps = eps.where(ds["aice"] >= AICE_MASK_THRESH)
    eps2 = time_reduce(eps, TIME_STAT_DYN)
    eps2 = eps2.where(SP._region_mask(lon, lat, reg)) / 1e6
    eps2.name = "strain_invariant_p90"
    eps2.attrs["units"] = infer_units(ds["divu"], "s-1")
    return eps2, lon, lat
    
def load_experiment_summary(sim):
    SP = make_plotter(sim)
    reg = SP._resolve_regions(region_name=REGION_NAME)[REGION_NAME]
    coords = SP._load_static_lonlat()
    lon = coords["TLON"]
    lat = coords["TLAT"]
    fields = {}
    # Metrics-store fields
    for name in ["FIP", "FIHI", "FIST", "FIMAR_YR"]:
        print(f"  loading metric {name}")
        da = load_metric_2d(SP, sim, name)
        fields[name] = {"da": da, "lon": lon, "lat": lat}
    # Dynamic fields
    print("  computing FI |Ku| p90")
    Ku2, Ku_lon, Ku_lat = compute_fi_ku_p90(SP, sim, reg, months=MONTHS_DYN)
    fields["FI_Ku_p90"] = {"da": Ku2, "lon": Ku_lon,"lat": Ku_lat}
    print("  computing strain invariant p90")
    eps2, eps_lon, eps_lat = compute_strain_invariant_p90(SP, sim, reg, months=MONTHS_DYN)
    fields["strain_p90"] = {"da": eps2, "lon": eps_lon, "lat": eps_lat}
    # Convert to PyGMT dataframes
    for key, item in fields.items():
        da_plot = item["da"]
        lon_plot = item["lon"]
        lat_plot = item["lat"]
        # Plot strain in microstrain/s for readable colourbar.
        if key == "strain_p90":
            da_plot = da_plot * 1e6
            da_plot.attrs["plot_units"] = "10^-6 s^-1"
        # Do NOT divide FIST unless you confirm units.
        # Your previous figure had tiny hPa values; keep native units first.
        if key == "FIST":
            da_plot = da_plot
            da_plot.attrs["plot_units"] = infer_units(item["da"], "Pa")
        # FIMAR_YR can be sparse/signed; keep native values.
        if key == "FIMAR_YR":
            da_plot = da_plot / 1e6
            da_plot.attrs["plot_units"] = infer_units(item["da"], "")
        df = SP.pygmt_da_prep(da_plot, lon=lon_plot, lat=lat_plot, mask_zero=False, region=reg)
        fields[key]["plot_da"] = da_plot
        fields[key]["df"] = df
        if len(df) > 0:
            print(f"    {key:12s}: n={len(df):7d}, "
                  f"min={df['z'].min():.3e}, "
                  f"p50={df['z'].median():.3e}, "
                  f"p98={df['z'].quantile(0.98):.3e}, "
                  f"max={df['z'].max():.3e}")
        else:
            print(f"    {key:12s}: EMPTY")
    f2_outline = load_f2_outline(SP, coords, reg)
    return {"sim": sim,
            "SP": SP,
            "region": reg,
            "coords": coords,
            "fields": fields,
            "f2_outline": f2_outline}

### load all the data

In [ ]:
SUMMARIES = {}
for sim in EXPERIMENTS:
    print(f"\n=== {sim} ===")
    SUMMARIES[sim] = load_experiment_summary(sim)
print("\nLoaded experiments:", list(SUMMARIES))

### plot

In [ ]:
SP0          = SUMMARIES[EXPERIMENTS[0]]["SP"]

PANEL_ORDER = [("FIP", "LFI Persistence"),
               ("FIHI", "LFI-mean Thickness"),
               ("FIST", "LFI-mean Strength"),
               ("FI_Ku_p90", f"LFI-LD-ttl-strees-P90 ({months_token(MONTHS_DYN)})"),
               ("strain_p90", f"Strain Invariant P90 ({months_token(MONTHS_DYN)})"),
               ("FIMAR_YR", "LFI Mechanical Area Rate")]

def plot_summary_panel(fig, SP, reg, proj, df, scale, title, f2_outline=None):
    fig.basemap(region = reg, projection = proj, frame = ["af", f"+t{title}"])
    fig.coast(shorelines = "0.25p,black", land = "lightgray", water = "white")
    pygmt.makecpt(cmap = scale["cmap"], series = scale["series"])#, continuous = True)
    if len(df) > 0:
        fig.plot(x = df["lon"], y = df["lat"], style = GRID_STYLE, fill = df["z"], cmap = True, pen = None)
    if f2_outline is not None:
        lon2d, lat2d, z2d, m = f2_outline
        fig.contour(x = lon2d[m], y = lat2d[m], z = z2d[m], levels = [0.5], pen = "0.45p,blue")
    fig.colorbar(position = "JBC+w6.5c/0.22c+o0c/-0.45c+h", frame = scale["frame"])

def plot_experiment_summary(summary):
    sim        = summary["sim"]
    SP         = summary["SP"]
    reg        = summary["region"]
    fields     = summary["fields"]
    f2_outline = summary["f2_outline"]
    proj       = SP.projection_from_region(reg, fig_size=FIG_SIZE)
    dyn_token  = months_token(MONTHS_DYN)
    P_out      = (D_SUMMARY / f"{sim}_{REGION_NAME}_FIP_FIHI_FIST_Ku_strain_FIMAR_{TIME_STAT_DYN}_{dyn_token}_{START}_{END}.png")
    if P_out.exists() and not OVERWRITE:
        print(f"exists; skipping {P_out}")
        return P_out
    fig = pygmt.Figure()
    # Much tighter than the previous version.
    # xshift uses panel width plus small gap.
    x_step = "w+14c"
    # yshift uses panel height plus room for colourbar/title.
    y_step = "-h-8c"
    for i, (field_key, title) in enumerate(PANEL_ORDER):
        row = i // 3
        col = i % 3
        if i == 0:
            pass
        elif col > 0:
            fig.shift_origin(xshift=x_step, yshift="0c")
        else:
            # Return to left column and move down.
            fig.shift_origin(xshift="-6w-2c", yshift=y_step)
        df    = fields[field_key]["df"]
        scale = SCALES[field_key]
        if i == 4:
            plot_summary_panel(fig = fig, SP = SP, reg = reg, proj = proj, df = df, scale = scale, title = title, f2_outline = f2_outline)
        else:
            plot_summary_panel(fig = fig, SP = SP, reg = reg, proj = proj, df = df, scale = scale, title = title, f2_outline = None)
    # crop=True helps remove page margins.
    fig.savefig(P_out, crop=True)
    if SHOW_FIGS:
        fig.show()
    print(f"saved {P_out}")
    return P_out
    
P_SUMMARY = []
for sim in EXPERIMENTS:
    P_SUMMARY.append(plot_experiment_summary(SUMMARIES[sim]))
P_SUMMARY

In [ ]:
display(Image(filename="/g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/per_experiment_summary/LD-static-Cs1e-3_Aus_FIP_FIHI_FIST_Ku_strain_FIMAR_p90_AMJ_2000-01-01_2003-12-31.png"))

In [ ]:
display(Image(filename="/g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/per_experiment_summary/LD-blend-base_Aus_FIP_FIHI_FIST_Ku_strain_FIMAR_p90_AMJ_2000-01-01_2003-12-31.png"))

In [ ]:
display(Image(filename="/g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/per_experiment_summary/LD-static-Cs1e-3_Aus_FIP_FIHI_FIST_Ku_strain_FIMAR_p90_SON_2000-01-01_2003-12-31.png"))

In [ ]:
display(Image(filename="/g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/per_experiment_summary/LD-blend-base_Aus_FIP_FIHI_FIST_Ku_strain_FIMAR_p90_SON_2000-01-01_2003-12-31.png"))

In [ ]:
from PIL import Image, ImageDraw
from math import ceil

def stitch_images(image_paths, out_path, ncols=2, pad=30, bg="white", labels=None):
    imgs = [Image.open(p).convert("RGB") for p in image_paths]

    if labels is None:
        labels = [Path(p).stem for p in image_paths]

    max_w = max(im.width for im in imgs)
    max_h = max(im.height for im in imgs)

    norm_imgs = []
    for im, label in zip(imgs, labels):
        canvas = Image.new("RGB", (max_w, max_h), bg)
        canvas.paste(im, ((max_w - im.width) // 2, (max_h - im.height) // 2))

        draw = ImageDraw.Draw(canvas)
        draw.rectangle((10, 10, 10 + 620, 58), fill="white")
        draw.text((22, 22), label, fill="black")

        norm_imgs.append(canvas)

    nrows = ceil(len(norm_imgs) / ncols)
    out_w = ncols * max_w + (ncols + 1) * pad
    out_h = nrows * max_h + (nrows + 1) * pad

    mosaic = Image.new("RGB", (out_w, out_h), bg)

    for i, im in enumerate(norm_imgs):
        r = i // ncols
        c = i % ncols
        x0 = pad + c * (max_w + pad)
        y0 = pad + r * (max_h + pad)
        mosaic.paste(im, (x0, y0))

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    mosaic.save(out_path, dpi=(200, 200))
    return out_path


P_CONTACT = D_SUMMARY / f"ALL_{REGION_NAME}_per_experiment_summary_{TIME_STAT_DYN}_{months_token(MONTHS_DYN)}_{START}_{END}.png"

stitch_images(
    P_SUMMARY,
    P_CONTACT,
    ncols=2,
    labels=EXPERIMENTS,
)

P_CONTACT

## ku

In [ ]:
# Cell: 3x2 PyGMT map of lateral-drag stress magnitude over Aus
#
# Computes:
#   KuE_mag = sqrt(KuxE^2 + KuyE^2)
#   KuN_mag = sqrt(KuxN^2 + KuyN^2)
#   Ku_mag  = 0.5 * (KuE_mag + KuN_mag)
#
# Then maps either mean, median, p90, p95, max over the requested time window.
#
# Assumes you already have:
#   EXPERIMENTS, START, END, D_out, make_plotter
#
# Example:
#   START = "2000-01-01"
#   END   = "2003-12-31"
#   EXPERIMENTS = [
#       "LD-blend-base",
#       "LD-blend-exp10",
#       "LD-static-Cs5e-4",
#       "LD-quad-Cq75",
#       "LD-linear-CL0p25",
#       "LD-NIL",
#   ]

In [ ]:
def _lon_to_180(lon):
    return ((lon + 180.0) % 360.0) - 180.0

def _region_mask(lon, lat, region):
    lon180 = _lon_to_180(lon)
    lon_min, lon_max, lat_min, lat_max = [float(v) for v in region]
    if lon_min <= lon_max:
        lon_ok = (lon180 >= lon_min) & (lon180 <= lon_max)
    else:
        lon_ok = (lon180 >= lon_min) | (lon180 <= lon_max)
    lat_ok = (lat >= lat_min) & (lat <= lat_max)
    return lon_ok & lat_ok

def _crop_to_region_bbox(ds, lon, lat, region):
    """
    Crop the dataset to the smallest i/j bounding box around the region.
    This reduces the amount of data touched before the time aggregation.
    """
    mask = _region_mask(lon, lat, region)
    # compute only the 2D mask, not the 3D stress fields
    mask_np = mask.compute().values
    if not np.any(mask_np):
        raise ValueError(f"No grid cells found in region {region}")
    yy, xx = np.where(mask_np)
    ydim, xdim = lon.dims
    yslice = slice(int(yy.min()), int(yy.max()) + 1)
    xslice = slice(int(xx.min()), int(xx.max()) + 1)
    indexer = {}
    if ydim in ds.dims:
        indexer[ydim] = yslice
    if xdim in ds.dims:
        indexer[xdim] = xslice
    ds_c = ds.isel(indexer)
    lon_c = lon.isel({ydim: yslice, xdim: xslice})
    lat_c = lat.isel({ydim: yslice, xdim: xslice})
    return ds_c, lon_c, lat_c

def _time_reduce(da, stat):
    stat = stat.lower()
    if "time" not in da.dims:
        return da
    if stat == "mean":
        return da.mean("time", skipna=True)
    if stat == "median":
        return da.median("time", skipna=True)
    if stat == "p90":
        return da.quantile(0.90, dim="time", skipna=True).drop_vars("quantile", errors="ignore")
    if stat == "p95":
        return da.quantile(0.95, dim="time", skipna=True).drop_vars("quantile", errors="ignore")
    if stat == "max":
        return da.max("time", skipna=True)
    raise ValueError(f"Unknown TIME_STAT={stat!r}")

def load_ku_mag_map(sim, start=DT0_STR, end=DTN_STR, months=[9,10,11], time_stat='p95'):
    """
    Load Kux/Kuy lateral-drag fields, compute Ku magnitude, crop to Aus,
    reduce over time, and return a PyGMT-ready dataframe plus the 2D field.
    """
    plotter   = make_plotter(sim)
    run       = replace(plotter.run, sim_name=sim)
    variables = ["KuxE", "KuyE", "KuxN", "KuyN", "aice", "TLON", "TLAT", "ULON", "ULAT"]
    ds        = load_cice(run=run,
                          classify=plotter.classify,
                          metrics=plotter.metrics,
                          plotting=plotter.plotting,
                          observations=plotter.observations,
                          paths=plotter.paths,
                          variables=variables,
                          hemisphere=run.hemisphere,
                          chunks={"time": 31})
    # Prefer TLON/TLAT if the Ku fields are on the T-like diagnostic grid.
    # Fall back to ULON/ULAT if needed.
    if "TLON" in ds and "TLAT" in ds:
        lon, lat = ds["TLON"], ds["TLAT"]
    elif "ULON" in ds and "ULAT" in ds:
        lon, lat = ds["ULON"], ds["ULAT"]
    else:
        lon, lat = plotter._detect_lonlat(ds)
    # Time subset
    print("slicing time")
    ds = ds.sel(time=slice(start, end))
    if months is not None:
        print(f"sub-setting for months: {MONTHS}")
        ds = ds.sel(time=ds["time"].dt.month.isin(months))
    if ds.sizes.get("time", 0) == 0:
        raise ValueError(f"No data for {sim} in {start} to {end}, months={months}")
    # Crop before computing Ku/time stats.
    print("cropping data to region for stats")
    ds, lon, lat = _crop_to_region_bbox(ds, lon, lat, REGION)
    # Magnitudes for E and N faces. This assumes these diagnostic fields
    # have the same horizontal dimensions. If they do not, handle E/N separately.
    print("computing Ku magnitudes")
    KuE = np.hypot(ds["KuxE"], ds["KuyE"])
    KuN = np.hypot(ds["KuxN"], ds["KuyN"])
    Ku  = 0.5 * (KuE + KuN)
    Ku.name = "Ku_mag"
    # Optional sea-ice mask 
    if AICE_MASK_THRESH is not None and "aice" in ds:
        print(f"masking for sea ice concentration > {AICE_MASK_THRESH}")
        si = ds["aice"] >= AICE_MASK_THRESH
        Ku = Ku.where(si)
    # Time reduction: p90 recommended first.
    print("statistical method for sub-setted months and region")
    Ku2 = _time_reduce(Ku, time_stat)
    # Mask to the exact region after reduction.
    mask = _region_mask(lon, lat, REGION)
    Ku2 = Ku2.where(mask)
    # Convert to PyGMT-friendly dataframe using shuga's existing helper.
    data = plotter.pygmt_da_prep(Ku2, lon=lon,lat=lat, mask_zero=True, region=REGION)
    return data, Ku2, lon, lat

import calendar

def months_token(months):
    """
    Return a clean filename token for MONTHS.
    
    Examples
    --------
    None           -> "all-months"
    [4, 5, 6]      -> "AMJ"
    [6, 7, 8]      -> "JJA"
    [9, 10, 11]    -> "SON"
    [10, 11, 12]   -> "OND"
    [1, 3, 12]     -> "m01-03-12"
    """
    if months is None:
        return "all-months"
    months       = sorted(int(m) for m in months)
    season_names = {(12, 1, 2): "DJF",
                    (3, 4, 5): "MAM",
                    (6, 7, 8): "JJA",
                    (9, 10, 11): "SON",
                    (4, 5, 6): "AMJ",
                    (5, 6, 7): "MJJ",
                    (7, 8, 9): "JAS",
                    (8, 9, 10): "ASO",
                    (10, 11, 12): "OND"}
    key = tuple(months)
    if key in season_names:
        return season_names[key]
    # Handle DJF supplied as [1, 2, 12]
    if set(months) == {12, 1, 2}:
        return "DJF"
    # For any contiguous 3-month window not listed above
    if len(months) == 3 and months[1] == months[0] + 1 and months[2] == months[1] + 1:
        return "".join(calendar.month_abbr[m][0] for m in months)
    return "m" + "-".join(f"{m:02d}" for m in months)

In [ ]:
REGION_NAME = "Aus"
REGION = ANTARCTIC_8_REGIONS[REGION_NAME]["plot_region"]
# Options:
#   None                 -> full START/END window
#   [4, 5, 6]             -> autumn/early formation
#   [6, 7, 8]             -> JJA
#   [9, 10, 11]           -> SON
#   [10, 11, 12]          -> peak / breakout-sensitive season
MONTHS = [4,5,6]
# Options: "mean", "median", "p90", "p95", "max"
TIME_STAT = "p90"
# Avoid plotting open ocean / very weak numerical residue.
# Set to None to disable.
AICE_MASK_THRESH = 0.15
# Set this higher/lower depending on density of points.
GRID_STYLE = "s0.055c"
# Common colour scale: percentile across all six regional maps.
# 98 is usually better than max because max can be dominated by tiny spikes.
COMMON_VMAX_PERCENTILE = 98.0
# -------------------------------------------------------------------
# Load all panels first so we can use a common colour scale
# -------------------------------------------------------------------
panel_data = {}
for sim in EXPERIMENTS:
    print(f"loading {sim}")
    data, Ku2, lon, lat = load_ku_mag_map(sim, months=MONTHS)
    panel_data[sim] = data
    print(f"  n points: {len(data):,}, z range: {data['z'].min():.3e} to {data['z'].max():.3e}")
all_z = np.concatenate([d["z"].to_numpy() for d in panel_data.values() if len(d) > 0])
if all_z.size == 0:
    raise ValueError("No finite Ku magnitude values found for any experiment.")
vmin = 0.0
vmax = float(np.nanpercentile(all_z, COMMON_VMAX_PERCENTILE))
if not np.isfinite(vmax) or vmax <= 0:
    vmax = float(np.nanmax(all_z))
print(f"common colour scale: {vmin:.3e} to {vmax:.3e}")

In [ ]:
M_TOKEN = months_token(MONTHS)
P_out = D_out / f"Ku_mag_{REGION_NAME}_{TIME_STAT}_{M_TOKEN}_{START}_{END}.png"
P_out.parent.mkdir(parents=True, exist_ok=True)
fig = pygmt.Figure()
projection = make_plotter(EXPERIMENTS[0]).projection_from_region(REGION, fig_size=13.5)
pygmt.makecpt(cmap="batlow", series=[0, 3], continuous=True)
panel_width_shift = "15.0c"
panel_height_shift = "-8.3c"
months_str = "all months" if MONTHS is None else "months " + ",".join(f"{m:02d}" for m in MONTHS)
main_title = f"{REGION_NAME}: {TIME_STAT.upper()} lateral-drag magnitude Ku, {START} to {END}, {months_str}"
for i, sim in enumerate(EXPERIMENTS):
    row = i // 2
    col = i % 2
    if i == 0:
        pass
    elif col == 1:
        fig.shift_origin(xshift=panel_width_shift, yshift="0c")
    else:
        fig.shift_origin(xshift=f"-{panel_width_shift}", yshift=panel_height_shift)
    title = sim
    if i == 0:
        title = main_title + f"+t{sim}"
    # Use shuga base-layer logic, but directly here for simplicity.
    fig.basemap(region=REGION, projection=projection, frame=["af", f"+t{sim}"])
    fig.coast(shorelines="0.25p,black", land="lightgray", water="white")
    data = panel_data[sim]
    if len(data) > 0:
        fig.plot(x=data["lon"], y=data["lat"], style=GRID_STYLE, fill=data["z"], cmap=True, pen=None)
# Shared colorbar at bottom of whole layout.
fig.shift_origin(xshift="-15.0c", yshift="-1.3c")
fig.colorbar(position="JBC+w14c/0.35c+o7.5c/-0.7c+h", frame=['xaf+llateral-drag stress |Ku|','y+l(N m@+-2@+)'])
fig.savefig(P_out)
fig.show()
P_out

# 6. Multi-year comparison: static versus best `blend_strain`, 2000--2003

This section is a scaffold for the later multi-year result. The goal is not to repeat every process diagnostic, but to show that the best corrected `blend_strain` configuration improves or maintains skill against AF2020 across a longer period.

Recommended minimal multi-year comparison:

- `LD-static-Cs5e-4`;
- best corrected `blend_strain` experiment, rerun or extended over 2000--2018;
- AF2020 daily/periodic fast-ice masks;
- binary-days classification and metrics through `shuga`.

The key outputs should be a compact table and one figure: annual/seasonal FIA skill and spatial overlap skill.

In [ ]:
MULTIYEAR_START = "2000-01-01"
MULTIYEAR_END  = "2003-12-31"
MULTIYEAR_SIMS = ["LD-blend-base", "LD-blend-exp10", "LD-static-Cs5e-4", "LD-quad-Cq75" , "LD-linear-CL0p25", "LD-NIL"]  # replace LD-blend-best with the actual extended run name.

def multiyear_fia_skill_scaffold():
    rows = []
    for sim in MULTIYEAR_SIMS:
        fia = daily_fia_from_mask(sim, MULTIYEAR_START, MULTIYEAR_END)
        df = safe_to_series(fia, "FIA_10^3_km2")
        df["sim_name"] = sim
        df["year"] = pd.to_datetime(df["time"]).dt.year
        rows.append(df)
    if not rows:
        return pd.DataFrame(), pd.DataFrame()
    daily = pd.concat(rows, ignore_index=True)
    annual = daily.groupby(["sim_name", "year"])["FIA_10^3_km2"].agg(["mean", "max", "min", "std"]).reset_index()
    return daily, annual

# Uncomment once the best blend run has multi-year output.
MULTI_DAILY, MULTI_ANNUAL = multiyear_fia_skill_scaffold()
display_df(MULTI_ANNUAL, n=40)


# 7. pack-ice checks

A lateral-drag parameterisation is not publishable if it improves fast ice by damaging the mobile pack. This section is intended for main-text support or supplementary/appendix figures showing that the selected `blend_strain` or `static` configuration does not degrade broader pack-ice behaviour.

Candidate checks:

- total sea-ice area/extent/volume relative to existing benchmarks;
- pack-ice speed outside binary-days fast ice;
- stress-budget ratios outside the coastal/fast-ice zone;
- optional drift comparison against observed drift products if available;
- optional comparison against the grounded-iceberg-as-landmask experiment from the first fast-ice sensitivity paper.

In [ ]:
def collateral_pack_summary(sim_name: str) -> dict:
    ds = load_pub_cice(sim_name, ANALYSIS_START, ANALYSIS_END, variables=[
        "aice", "hi", "tarea", "uvel", "vvel", "TLON", "TLAT",
        "KuxE", "KuyE", "KuxN", "KuyN",
        "strairx", "strairy", "strocnx", "strocny", "strintx", "strinty",
    ])
    cls = load_pub_classified(sim_name, ANALYSIS_START, ANALYSIS_END, variables=["FI_mask"])
    ds, cls = robust_align_time(ds, cls)
    area = area_da(ds)
    fi = cls["FI_mask"].astype(bool)
    pack = (~fi) & (ds["aice"] > 0.15)

    row = {"sim_name": sim_name}
    if "aice" in ds:
        sia = area_sum(ds["aice"] > 0.15, area) / 1e12  # million km2
        row["SIA_mean_million_km2"] = float(sia.mean("time").compute())
    if all(v in ds for v in ["hi", "aice"]):
        siv = (ds["hi"] * ds["aice"] * area).sum(dim=[d for d in area.dims if d in ds["hi"].dims], skipna=True) / 1e12
        row["SIV_mean_10^12_m3"] = float(siv.mean("time").compute())
    diag = compute_diagnostic_terms(ds)
    for var in ["ice_speed", "tau_ld_est", "R_ld_budget"]:
        if var in diag:
            row[f"pack__{var}__mean"] = float(area_weighted_mean(diag[var], area, pack).mean("time").compute())
    return row

COLLATERAL_ROWS = []
for sim in COMPARISON_SIMS:
    try:
        COLLATERAL_ROWS.append(collateral_pack_summary(sim))
    except Exception as exc:
        print(f"Skipping collateral summary for {sim}: {exc}")
COLLATERAL = pd.DataFrame(COLLATERAL_ROWS)
display_df(COLLATERAL, n=20)
COLLATERAL.to_csv(NOTEBOOK_FIG_ROOT / f"collateral_pack_summary_{ANALYSIS_START}_{ANALYSIS_END}.csv", index=False)


# 8. Publication figure checklist

Candidate main-text figures:

1. **Conceptual figure:** no-slip versus free-slip + explicit lateral drag.
2. **Form-factor figure:** high-resolution coastline, grounded-iceberg contribution, combined form factor.
3. **Analytical form-function figure:** static/quadratic/linear/corrected `blend_strain`, with speed and strain gate annotations.
4. **Corrected `blend_strain` process figure:** FIA, spatial skill, phase-space `ldspd`--`ldeps`--`ldwgt`, and one map panel.
5. **Form-function comparison figure/table:** static/quadratic/linear/best-blend over 1994-10-01 to 1994-12-15.
6. **Collateral/supplement:** pack-ice speed/SIA/SIV checks.

Key narrative test:

> Does corrected `blend_strain` move the model closer to observed Antarctic fast-ice persistence and retreat while applying static-like drag only in slow, coherent, low-strain, form-factor-active cells?